# ICLR-II Flat Notebook
This notebook is a flat, self-contained version of `iclrii.py`. All repo-level dependencies are defined below (no local imports). Adjust hyperparameters at the top, then run top-to-bottom to reproduce the three stages: (A,B) ensemble, x0 scores, and estimator errors.

In [ ]:
# Imports (Python libraries only)
import argparse
import math
import pathlib
import warnings
from dataclasses import dataclass
from typing import Any, Dict, Iterable, Mapping, Optional, Sequence, Tuple

import numpy as np
import numpy.linalg as npl
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib import colormaps

import scipy.linalg
from scipy.linalg import expm, eigvals, solve_continuous_lyapunov, solve_discrete_lyapunov, null_space, qr

import pysindy
import torch

from IPython.display import display

## Hyperparameters
Set these to match the CLI flags in `iclrii.py`.

In [ ]:
# Output directories
base_outdir = "pyident_results"
ensemble_dir = "fresh_ensemble"
x0_dir = "fresh_ABx0"
pbh_dir = "fresh_nonidentifiable_ABx0"

# Ensemble sweep
sparsity_grid = "0.0:0.1:1.0"
ndim_grid = "2:1:10"
samples = 10000

# x0 sampling and plotting
x0_samples = 10
outlier_trim = 0.05
sphere = True
sparse_p = None  # set a float in (0,1] to use masked x0

# Estimation / simulation
seed = 12345
T = 100
dt = 1.0
u_scale = 3.0
dwell = 1
algos = "DMDc"  # e.g., "SINDy,DMDc,MOESP,NODE"

input_family = "prbs"  # "prbs" or "multisine"
pe_method = "block"    # "block" or "moment"
pe_tol = 1e-8
pe_max_tries = 128

dmdc_z_cond_max = 1e8

# x0 constraints
x0_min_norm = 1e-6
x0_min_support = 1
x0_max_attempts = 128
min_visible_dim = 0

# Selection/output tweaks
selected_suffix = None  # override selected dataset suffix (default: all_x0)

# DMDc ridge option
ridge = False
ridge_lam = 1e-6

## Dependencies
Below are the repo-level dependencies, inlined and ordered by use.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
import numpy as np

@dataclass
class TolerancePolicy:
    """Centralized numerical tolerances (logged to the ledger)."""
    svd_rtol: float = 1e-9
    svd_atol: float = 1e-12
    ev_atol: float = 1e-10
    pbh_cluster_tol: float = 1e-8

    def rank_from_singulars(self, s: np.ndarray) -> int:
        if s.size == 0:
            return 0
        s = np.asarray(s, dtype=float)
        smax = float(s[0])
        thresh = max(self.svd_atol, self.svd_rtol * smax)
        return int((s > thresh).sum())

    def cluster_eigs(self, lam: np.ndarray) -> np.ndarray:
        lam = np.asarray(lam)
        if lam.size == 0:
            return lam
        used = np.zeros(lam.shape, dtype=bool)
        reps = []
        for i in range(len(lam)):
            if used[i]:
                continue
            group = [i]
            for j in range(i + 1, len(lam)):
                if abs(lam[j] - lam[i]) <= self.pbh_cluster_tol:
                    group.append(j)
            used[group] = True
            reps.append(lam[group].mean())
        return np.array(reps, dtype=lam.dtype)

In [ ]:
from __future__ import annotations
import numpy as np
import math
from typing import Tuple, Optional

# ---------------------------------------------------------------------
# Input generators + PE tests
# ---------------------------------------------------------------------


def prbs(
    T: int,
    m: int,
    rng: np.random.Generator,
    levels: Tuple[float, float] = (-1.0, 1.0),
    period: int = 31,
) -> np.ndarray:
    """PRBS: repeat a random +/- pattern of length `period` to fill horizon T."""
    if period <= 0:
        raise ValueError("period must be positive.")
    base = rng.choice(levels, size=(period, m))
    reps = int(np.ceil(T / period))
    u = np.tile(base, (reps, 1))[:T]
    return u


def multisine(
    T: int,
    m: int,
    rng: np.random.Generator,
    k_lines: int = 8,
) -> np.ndarray:
    """Sum of sines with random phases on k_lines distinct bins; per-channel normalized."""
    t = np.arange(T)
    u = np.zeros((T, m))
    # guard small T
    max_bin = max(2, T // 8)
    for j in range(m):
        k = min(k_lines, max_bin - 1)
        freqs = rng.choice(np.arange(1, max_bin), size=k, replace=False)
        phases = rng.uniform(0, 2 * np.pi, size=k)
        w = np.zeros(T)
        for f, ph in zip(freqs, phases):
            w += np.sin(2 * np.pi * f * t / T + ph)
        denom = np.max(np.abs(w)) if np.max(np.abs(w)) > 0 else 1.0
        u[:, j] = w / denom
    return u


# ------------------ PE (block-Hankel) --------------------------------
def hankel_blocks(u: np.ndarray, s: int) -> np.ndarray:
    """Build block Hankel of depth s for multi-input u (T×m) -> (s*m)×(T-s+1)."""
    T, m = u.shape
    if s <= 0:
        raise ValueError("s must be positive.")
    cols = T - s + 1
    if cols <= 0:
        raise ValueError("Not enough samples for requested Hankel depth.")
    blocks = [u[i:i + cols].T for i in range(s)]  # each (m × cols)
    return np.vstack(blocks)                       # (s*m × cols)


def hankel_blocks_old(u: np.ndarray, s: int) -> np.ndarray:
    """Build block Hankel of depth s for multi-input u (T×m) -> (s*m)×cols."""
    T, m = u.shape
    cols = T - 2 * s + 1
    if s <= 0:
        raise ValueError("s must be positive.")
    if cols <= 0:
        raise ValueError("Not enough samples for requested Hankel depth.")
    H = [u[i:i + cols].T for i in range(s)]  # each (m×cols)
    return np.vstack(H)  # (s*m × cols)


def estimate_pe_order_block(u: np.ndarray, s_max: int, tol: float = 1e-8) -> int:
    # Preserve API; route through TolerancePolicy for consistency
    pol = TolerancePolicy(svd_atol=tol)  # respect old param while centralizing
    m = u.shape[1]
    s_max_eff = min(s_max, u.shape[0] // 2)
    for s in range(s_max_eff, 0, -1):
        H = hankel_blocks(u, s)
        # SVD-based rank
        svals = np.linalg.svd(H, compute_uv=False)
        r = pol.rank_from_singulars(svals)
        if r == H.shape[0]:  # full row rank
            return s
    return 0


def estimate_pe_order_block_old(u: np.ndarray, s_max: int, tol: float = 1e-8) -> int:
    """Largest s such that block-Hankel H_s has full row rank (up to tol)."""
    T, m = u.shape
    s_max_eff = min(s_max, max(1, T // 2))
    for s in range(s_max_eff, 0, -1):
        H = hankel_blocks(u, s)
        r = np.linalg.matrix_rank(H, tol=tol)
        if r == H.shape[0]:
            return s
    return 0

def estimate_pe_order(u: np.ndarray, s_max: int, tol: float = 1e-8) -> int:
    # kept for backward compatibility
    return estimate_pe_order_block(u, s_max=s_max, tol=tol)

# --- add these helpers near the top (after imports) ---
def _estimate_block_pe(U, s_max, tol=None):
    # Prefer tolerant signature if available; otherwise fall back.
    try:
        if tol is not None:
            return int(estimate_pe_order(U, s_max=s_max, tol=tol))
    except TypeError:
        pass
    return int(estimate_pe_order(U, s_max=s_max))

def _estimate_moment_pe(U, r_max, dt, tol=None):
    try:
        if tol is not None:
            return int(estimate_moment_pe_order(U, r_max=r_max, dt=dt, tol=tol))
    except TypeError:
        pass
    return int(estimate_moment_pe_order(U, r_max=r_max, dt=dt))



# ------------------ Moment-PE ------------------------

def _moment_map_old(u: np.ndarray, r: int, t0: Optional[int], dt: float = 1.0) -> np.ndarray:
    """Discrete approximation to psi_k(u) = int_0^{t0} ((t0-s)^k/k!) u(s) ds.

    Disclaimer: this is a *Riemann-sum* discretization with step `dt`.
    It approximates the continuous-time functional used in the manuscript.
    """
    T, m = u.shape
    if r <= 0:
        raise ValueError("r must be positive.")
    if t0 is None:
        t0 = T - 1
    if not (0 <= t0 < T):
        raise ValueError(f"t0 must be in [0, T-1], got {t0} for T={T}.")

    # times 0..t0 (inclusive)
    s = np.arange(t0 + 1)
    out = np.zeros((m * r,), dtype=float)
    idx = 0
    for k in range(r):
        # weights: ((t0 - s)^k / k!)
        w = ((t0 - s) ** k) / (np.math.factorial(k))
        # channel-wise integration
        psi_k = (u[: t0 + 1, :] * w[:, None]).sum(axis=0) * dt
        out[idx: idx + m] = psi_k
        idx += m
    return out


def _moment_map(u: np.ndarray, r: int, t0: Optional[int] = None, dt: float = 1.0) -> np.ndarray:
    T, m = u.shape                    # we use time-major: (T, m)
    if r <= 0:
        raise ValueError("r must be positive.")
    if t0 is None:
        t0 = T - 1
    if not (0 <= t0 < T):
        raise ValueError(f"t0 must be in [0, T-1], got {t0} for T={T}.")
    s = np.arange(t0 + 1, dtype=float)
    w = np.empty((r, t0 + 1), dtype=float)
    for k in range(r):
        denom = math.factorial(k) if k <= 20 else math.exp(math.lgamma(k+1))
        w[k, :] = ((t0 - s) ** k) / denom
    M = (w * dt) @ u[:t0+1, :]        # (r × m)
    return M.reshape(-1, order="C")   # stack ψ_k blocks of size m

def estimate_moment_pe_order(u: np.ndarray, r_max: int, t0: Optional[int] = None,
                             dt: float = 1.0, tol: float = 1e-10) -> int:
    T, _ = u.shape
    r_max_eff = max(1, min(int(r_max), T))
    best, prev = 0, 0.0
    for r in range(1, r_max_eff + 1):
        nv = float(np.linalg.norm(_moment_map(u, r, t0=t0, dt=dt)))
        if nv > tol and nv >= prev * (1 - 1e-12):
            best, prev = r, nv
        else:
            break
    return best

# ------------------ Pointwise restriction ----------------------------

def restrict_pointwise(u: np.ndarray, W: np.ndarray) -> np.ndarray:
    """Project inputs onto span(W) (columns of W form admissible directions).

    u: (T×m), W: (m×q). Returns u_proj ∈ span(W).

    Disclaimer: orthogonal projection uses P = W W^+.
    If W is ill-conditioned, results depend on pseudoinverse regularization.
    """
    if W.ndim != 2:
        raise ValueError("W must be a 2D array (m×q).")
    m = u.shape[1]
    if W.shape[0] != m:
        raise ValueError(f"W first dimension must equal m={m}, got {W.shape}.")
    Winv = np.linalg.pinv(W)
    P = W @ Winv  # (m×m) projector onto span(W)
    return u @ P  # row-wise projection of each time sample

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Callable, Dict, Optional, Sequence, Tuple, Literal

import numpy as np



ArrayLike = np.ndarray


@dataclass
class SignalBundle:
    """Container for a continuous-time input signal and its sampled sequence."""

    t: np.ndarray
    u: np.ndarray  # time-major samples, shape (T, m)
    u_of_t: Callable[[ArrayLike], np.ndarray]
    meta: Dict[str, object]
    pe_order_est: Optional[int] = None


@dataclass
class PRBSSpec:
    """Continuous-time PRBS specification.

    Notes:
        - PRBS is generated with a linear feedback shift register (LFSR) when
          `taps` are provided. Otherwise, a random ±1 pattern is used.
        - `clock` is the hold time (seconds) of each PRBS chip.
    """

    register: int = 7
    clock: float = 1.0
    levels: Tuple[float, float] = (-1.0, 1.0)
    taps: Optional[Sequence[int]] = None
    seed_state: Optional[Sequence[int]] = None


@dataclass
class MultisineSpec:
    """Continuous-time multisine specification."""

    k_lines: int = 8
    freqs: Optional[Sequence[float]] = None  # rad/s
    amps: Optional[Sequence[float]] = None


def _as_1d_time(t: ArrayLike) -> np.ndarray:
    t_arr = np.asarray(t, dtype=float)
    if t_arr.ndim == 0:
        return t_arr.reshape(1)
    return t_arr.reshape(-1)


def _zoh_callable(u: np.ndarray, dt: float, t0: float = 0.0) -> Callable[[ArrayLike], np.ndarray]:
    if dt <= 0:
        raise ValueError("dt must be positive for zero-order hold.")
    T, m = u.shape

    def u_of_t(t_query: ArrayLike) -> np.ndarray:
        tq = _as_1d_time(t_query)
        idx = np.floor((tq - t0) / dt).astype(int)
        idx = np.clip(idx, 0, T - 1)
        out = u[idx]
        if np.asarray(t_query).ndim == 0:
            return out.reshape(m)
        return out

    return u_of_t


def _lfsr_bits(length: int, register: int, taps: Sequence[int], seed_state: Optional[Sequence[int]] = None) -> np.ndarray:
    """Generate LFSR bits using a Fibonacci LFSR with 1-based tap positions.

    Taps are counted from the least significant bit (LSB). For example, taps
    (register, 1) corresponds to the polynomial x^register + x + 1.
    """
    if register <= 1:
        raise ValueError("register must be >= 2.")
    if length <= 0:
        raise ValueError("length must be positive.")
    if not taps:
        raise ValueError("taps must be a non-empty sequence of 1-based positions.")
    for t in taps:
        if t < 1 or t > register:
            raise ValueError(f"tap {t} is out of range for register length {register}.")

    if seed_state is None:
        state = np.ones(register, dtype=np.uint8)
    else:
        state = np.asarray(seed_state, dtype=np.uint8).reshape(-1)
        if state.size != register:
            raise ValueError("seed_state length must equal register length.")
        if not np.any(state):
            raise ValueError("seed_state must be non-zero.")
        state = state.copy()

    bits = np.zeros(length, dtype=np.uint8)
    for k in range(length):
        bits[k] = state[-1]
        fb = 0
        for t in taps:
            fb ^= state[-t]
        state[1:] = state[:-1]
        state[0] = fb
    return bits


def generate_prbs(
    T: int,
    m: int,
    dt: float,
    rng: np.random.Generator,
    spec: PRBSSpec,
    pe_order: Optional[int] = None,
) -> SignalBundle:
    """Generate a continuous-time PRBS signal and its samples.

    The output samples are time-major with shape (T, m).
    """
    if T <= 0 or m <= 0:
        raise ValueError("T and m must be positive.")
    if spec.clock <= 0:
        raise ValueError("PRBS clock must be positive.")

    if pe_order is not None:
        spec = PRBSSpec(
            register=max(int(pe_order), spec.register),
            clock=spec.clock,
            levels=spec.levels,
            taps=spec.taps,
            seed_state=spec.seed_state,
        )

    dwell = max(1, int(np.round(spec.clock / dt)))
    chips_needed = int(np.ceil(T / dwell))

    levels_arr = np.asarray(spec.levels, dtype=float)
    if levels_arr.shape != (2,):
        raise ValueError("levels must be a pair of floats (low, high).")

    if spec.taps is not None:
        period = (2 ** spec.register) - 1
        seq = _lfsr_bits(period, spec.register, spec.taps, seed_state=spec.seed_state)
        chips = np.zeros((chips_needed, m), dtype=float)
        for j in range(m):
            shift = int(rng.integers(0, period)) if m > 1 else 0
            shifted = np.roll(seq, shift)
            tiled = np.tile(shifted, int(np.ceil(chips_needed / period)))[:chips_needed]
            chips[:, j] = levels_arr[tiled]
    else:
        idx = rng.integers(0, 2, size=(chips_needed, m))
        chips = levels_arr[idx]

    U = np.repeat(chips, dwell, axis=0)[:T]

    t = np.arange(T, dtype=float) * dt
    u_of_t = _zoh_callable(U, dt=dt)
    meta = {
        "family": "prbs",
        "register": spec.register,
        "clock": spec.clock,
        "levels": spec.levels,
        "taps": tuple(spec.taps) if spec.taps is not None else None,
        "dwell": dwell,
    }
    return SignalBundle(t=t, u=U, u_of_t=u_of_t, meta=meta)


def _multisine_callable(
    freqs: np.ndarray,
    phases: np.ndarray,
    amps: np.ndarray,
    m: int,
) -> Callable[[ArrayLike], np.ndarray]:
    def u_of_t(t_query: ArrayLike) -> np.ndarray:
        tq = _as_1d_time(t_query)
        out = np.zeros((tq.size, m), dtype=float)
        for j in range(m):
            for w, ph, a in zip(freqs, phases[j], amps[j]):
                out[:, j] += a * np.sin(w * tq + ph)
        if np.asarray(t_query).ndim == 0:
            return out.reshape(m)
        return out

    return u_of_t


def generate_multisine(
    T: int,
    m: int,
    dt: float,
    rng: np.random.Generator,
    spec: MultisineSpec,
    pe_order: Optional[int] = None,
    normalize: bool = True,
) -> SignalBundle:
    if T <= 0 or m <= 0:
        raise ValueError("T and m must be positive.")
    horizon = T * dt

    k_lines = spec.k_lines
    if pe_order is not None:
        k_lines = max(int(pe_order), k_lines)
    if k_lines <= 0:
        raise ValueError("k_lines must be positive.")

    if spec.freqs is None:
        max_bin = max(2, T // 8)
        k_use = min(k_lines, max_bin - 1)
        bins = rng.choice(np.arange(1, max_bin), size=k_use, replace=False)
        freqs = 2.0 * np.pi * bins / max(horizon, dt)
    else:
        freqs = np.asarray(spec.freqs, dtype=float)
        if freqs.ndim != 1 or freqs.size == 0:
            raise ValueError("freqs must be a 1D array of length >= 1.")

    k_use = freqs.size
    phases = rng.uniform(0.0, 2.0 * np.pi, size=(m, k_use))
    if spec.amps is None:
        amps = np.ones((m, k_use), dtype=float)
    else:
        amps_vec = np.asarray(spec.amps, dtype=float)
        if amps_vec.ndim != 1 or amps_vec.size != k_use:
            raise ValueError("amps must be 1D and match number of freqs.")
        amps = np.repeat(amps_vec[None, :], m, axis=0)

    u_of_t = _multisine_callable(freqs, phases, amps, m)
    t = np.arange(T, dtype=float) * dt
    U = u_of_t(t)
    if normalize:
        scale = np.maximum(np.max(np.abs(U), axis=0), 1.0)
        U = U / scale

    meta = {
        "family": "multisine",
        "k_lines": int(k_use),
        "freqs": freqs,
        "phases": phases,
        "amps": amps,
    }
    return SignalBundle(t=t, u=U, u_of_t=u_of_t, meta=meta)


def generate_pe_signal(
    family: Literal["prbs", "multisine"],
    T: int,
    m: int,
    dt: float = 1.0,
    pe_order: Optional[int] = None,
    rng: Optional[np.random.Generator] = None,
    prbs: Optional[PRBSSpec] = None,
    multisine: Optional[MultisineSpec] = None,
    ensure_pe: bool = True,
    pe_method: Literal["block", "moment"] = "block",
    pe_tol: float = 1e-8,
    max_tries: int = 128,
) -> SignalBundle:
    """Generate a PE control input signal with a desired PE order.

    The returned bundle includes a continuous-time callable and sampled data
    compatible with the rest of the codebase (time-major (T, m)).
    """
    if rng is None:
        rng = np.random.default_rng()

    prbs = prbs if prbs is not None else PRBSSpec()
    multisine = multisine if multisine is not None else MultisineSpec()

    last_bundle: Optional[SignalBundle] = None
    last_pe: Optional[int] = None

    for _ in range(max_tries):
        if family == "prbs":
            bundle = generate_prbs(T, m, dt, rng, spec=prbs, pe_order=pe_order)
        elif family == "multisine":
            bundle = generate_multisine(T, m, dt, rng, spec=multisine, pe_order=pe_order)
        else:
            raise ValueError("family must be 'prbs' or 'multisine'.")

        if not ensure_pe or pe_order is None:
            return bundle

        if pe_method == "block":
            last_pe = estimate_pe_order(bundle.u, s_max=pe_order, tol=pe_tol)
        elif pe_method == "moment":
            last_pe = estimate_moment_pe_order(bundle.u, r_max=pe_order, dt=dt, tol=pe_tol)
        else:
            raise ValueError("pe_method must be 'block' or 'moment'.")

        bundle.pe_order_est = int(last_pe)
        last_bundle = bundle
        if last_pe >= pe_order:
            return bundle

    if last_bundle is None:
        raise RuntimeError("Failed to generate any signal bundle.")
    raise RuntimeError(
        f"Unable to reach PE order {pe_order} after {max_tries} tries; "
        f"best achieved order={last_pe}."
    )

In [ ]:
from __future__ import annotations
from typing import Optional, Tuple, Union
import warnings
import numpy as np
from numpy.random import Generator


def simulate_dt(
    *args: Union[int, np.ndarray],
    noise_std: float = 0.0,
    rng: Optional[Generator] = None,
) -> np.ndarray:
    """Simulate a discrete-time LTI system with optional Gaussian process noise.

    Args:
        x0: Initial state vector ``(n,)``.
        Ad: Discrete-time state matrix ``(n, n)``.
        Bd: Discrete-time input matrix ``(n, m)``.
        U: Time-major input sequence with shape ``(T, m)``.
        noise_std: Optional standard deviation for additive process noise.
        rng: Optional ``numpy.random.Generator`` used when ``noise_std > 0``.

    Returns:
        X: array of shape (n, T+1) including initial state
    Notes:
        The preferred call signature is ``simulate_dt(x0, Ad, Bd, U, ...)``.
        For backwards compatibility, the legacy
        ``simulate_dt(T, x0, Ad, Bd, U, ...)`` form is also accepted but will
        emit a ``DeprecationWarning``.
    """

    legacy_T: Optional[int] = None

    if len(args) == 4:
        x0, Ad, Bd, U = args  # type: ignore[misc]
    elif len(args) == 5:
        # Backwards-compatible call pattern simulate_dt(T, x0, Ad, Bd, U, ...)
        legacy_T, x0, Ad, Bd, U = args  # type: ignore[misc]
        warnings.warn(
            "simulate_dt(T, x0, Ad, Bd, U, ...) is deprecated; "
            "call simulate_dt(x0, Ad, Bd, U, ...) instead.",
            DeprecationWarning,
            stacklevel=2,
        )
    else:
        raise TypeError(
            "simulate_dt expected 4 positional arguments (x0, Ad, Bd, U); "
            "optionally accept legacy 5-argument form (T, x0, Ad, Bd, U)."
        )

    U = np.asarray(U, dtype=float)

    if legacy_T is not None:
        if U.ndim != 2:
            raise ValueError(
                "Legacy simulate_dt(T, ...) call received non 2-D U with shape {}.".format(
                    U.shape
                )
            )
        if U.shape[0] != legacy_T and U.shape[1] == legacy_T:
            warnings.warn(
                "simulate_dt detected channel-major inputs (m, T) in the deprecated "
                "signature and will transpose them to time-major (T, m).",
                DeprecationWarning,
                stacklevel=2,
            )
            U = np.asarray(U.T, dtype=float)
        elif U.shape[0] != legacy_T:
            raise ValueError(
                "Input horizon mismatch: provided T={} but U has shape {}.".format(
                    legacy_T, U.shape
                )
            )

    if U.ndim != 2:
        raise ValueError(f"U must be 2-D (T, m); got shape {U.shape}.")

    T, m = U.shape
    if Bd.shape[1] != m:
        raise ValueError(
            f"Bd has {Bd.shape[1]} input columns but U provides m={m} channels."
        )

    if rng is None:
        rng = np.random.default_rng()

    n = Ad.shape[0]

    if Ad.shape[1] != n:
        raise ValueError("Ad must be square (n, n).")

    X = np.zeros((n, T + 1), dtype=float)
    X[:, 0] = np.asarray(x0).reshape(n)

    for t in range(T):
        X[:, t + 1] = Ad @ X[:, t] + Bd @ U[t, :]
        if noise_std > 0.0:
            X[:, t + 1] += rng.normal(0.0, noise_std, size=n)

    return X

def prbs(
    T: int,
    m: int,
    scale: float = 1.0,
    dwell: int = 1,
    rng: Optional[Generator] = None,
) -> np.ndarray:
    """Generate a pseudo-random binary sequence (PRBS) input.

    Args:
        T: Horizon length (number of time steps).
        m: Number of input channels.
        scale: Amplitude applied to the ±1 sequence.
        dwell: Number of repeated samples per binary draw.
        rng: Optional ``numpy.random.Generator``.

    Returns:
        U: array of shape (T, m)
    """
    if rng is None:
        rng = np.random.default_rng()

    if dwell <= 0:
        raise ValueError("dwell must be positive.")

    U = np.zeros((T, m), dtype=float)
    blocks = int(np.ceil(T / dwell)) + 1
    draws = rng.choice([-1.0, 1.0], size=(blocks, m))
    tiled = np.repeat(draws, dwell, axis=0)[:T, :]
    U[:, :] = tiled

    return scale * U

In [ ]:
from __future__ import annotations

from typing import Optional, Tuple

import numpy as np



def spectral_radius(A: np.ndarray) -> float:
    if A.size == 0:
        return 0.0
    vals = np.linalg.eigvals(A)
    return float(np.max(np.abs(vals))) if vals.size else 0.0


def b_norms(B: np.ndarray) -> Tuple[float, float, float]:
    if B.size == 0:
        return 0.0, 0.0, 0.0
    fro = float(np.linalg.norm(B, ord="fro"))
    row_norms = np.linalg.norm(B, axis=1) if B.ndim == 2 else np.array([])
    col_norms = np.linalg.norm(B, axis=0) if B.ndim == 2 else np.array([])
    min_row = float(row_norms.min()) if row_norms.size else 0.0
    min_col = float(col_norms.min()) if col_norms.size else 0.0
    return fro, min_row, min_col


def condition_number(
    Z: np.ndarray,
    *,
    tol: float = 1e-12,
    normalize_columns: bool = True,
) -> float:
    Z = np.asarray(Z, dtype=float)
    if Z.size == 0:
        return float("inf")
    if normalize_columns:
        coln = np.linalg.norm(Z, axis=0)
        coln[coln == 0.0] = 1.0
        Z = Z / coln
    s = np.linalg.svd(Z, compute_uv=False)
    if s.size == 0:
        return float("inf")
    smin = float(s[-1])
    smax = float(s[0])
    if smin <= tol:
        return float("inf")
    return smax / smin


def pe_order_from_nmax(n_max: int) -> int:
    return 2 * int(n_max) + 1


def min_T_for_block_pe(pe_order: int, m: int) -> int:
    return int(pe_order) * (int(m) + 1) - 1


def generate_pe_input(
    *,
    T: int,
    m: int,
    dt: float,
    dwell: int,
    rng: np.random.Generator,
    pe_order: int,
    family: str = "prbs",
    pe_method: str = "block",
    pe_tol: float = 1e-8,
    max_tries: int = 128,
    scale: float = 1.0,
    prbs: Optional[PRBSSpec] = None,
    multisine: Optional[MultisineSpec] = None,
) -> tuple[np.ndarray, int]:
    if pe_method == "block":
        min_T = min_T_for_block_pe(pe_order, m)
        if T < min_T:
            raise ValueError(
                f"T={T} is too short for block-PE order {pe_order} with m={m}; "
                f"need at least T >= {min_T}."
            )

    if prbs is None:
        prbs = PRBSSpec(clock=float(dt) * int(dwell))
    if multisine is None:
        multisine = MultisineSpec()

    bundle = generate_pe_signal(
        family=family,
        T=T,
        m=m,
        dt=dt,
        pe_order=pe_order,
        rng=rng,
        prbs=prbs,
        multisine=multisine,
        ensure_pe=True,
        pe_method=pe_method,
        pe_tol=pe_tol,
        max_tries=max_tries,
    )

    U = np.asarray(bundle.u, dtype=float) * float(scale)
    pe_est = int(bundle.pe_order_est) if bundle.pe_order_est is not None else 0
    return U, pe_est

In [ ]:
import warnings
from typing import Tuple, Literal, Optional, Dict, Any, Callable

import numpy as np
import numpy.linalg as npl


# ---------------------------------------------------------------------
# Random-matrix factories for (A, B).
# All generators return numpy arrays with shapes:
#   A: (n, n),  B: (n, m)
# ---------------------------------------------------------------------


def ginibre(n: int, m: int, rng: np.random.Generator) -> Tuple[np.ndarray, np.ndarray]:
    A = rng.normal(loc=1.0, scale=1.0, size=(n, n)) / np.sqrt(n)
    B = rng.normal(loc=1.0, scale=1.0, size=(n, m)) / np.sqrt(n)
    return A, B


def binary(n: int, m: int, rng: np.random.Generator) -> Tuple[np.ndarray, np.ndarray]:
    """Binary ensemble.This may induce strong degeneracies (e.g., repeated singular values).
      Prefer rademacher() if you need symmetric +/- 1."""
    A = rng.choice([0.0, 1.0], size=(n, n))
    B = rng.choice([0.0, 1.0], size=(n, m))
    return A, B


def rademacher(n: int, m: int, rng: np.random.Generator) -> Tuple[np.ndarray, np.ndarray]:
    A = rng.choice([-1.0, 1.0], size=(n, n))
    B = rng.choice([-1.0, 1.0], size=(n, m))
    return A, B


def sparse_continuous_column(
    n: int,
    rng: np.random.Generator,
    p_density: float = 0.5,
) -> np.ndarray:

    v = rng.normal(loc=1.0, scale=1.0, size=(n, 1))
    m = rng.binomial(1, p_density, size=(n, 1)).astype(float)
    return v * m


def sparse_continuous(
    n: int,
    m: int,
    rng: np.random.Generator,
    which: Literal["A", "B", "both"] = "both",
    p_density: float = 0.3,
    p_density_A: Optional[float] = None,
    p_density_B: Optional[float] = None,
    check_zero_rows: bool = False,
    max_attempts: int = 200,
) -> Tuple[np.ndarray, np.ndarray]:
    p_density_A = p_density if p_density_A is None else p_density_A
    p_density_B = p_density if p_density_B is None else p_density_B

    A = rng.normal(loc=1.0, scale=1.0, size=(n, n))
    B = rng.normal(loc=1.0, scale=1.0, size=(n, m))

    A_mask = rng.binomial(1, p_density_A, size=(n, n)).astype(float)
    B_mask = rng.binomial(1, p_density_B, size=(n, m)).astype(float)

    if which in ("A", "both"):
        A *= A_mask
    if which in ("B", "both"):
        B *= B_mask

    if check_zero_rows and which in ("B", "both"):
        z = (np.linalg.norm(B, axis=1) == 0)
        tries = 0
        while np.any(z) and tries < max_attempts:
            B[z, :] = rng.normal(loc=1.0, scale=1.0, size=(z.sum(), m)) \
                      * rng.binomial(1, p_density_B, size=(z.sum(), m))
            z = (np.linalg.norm(B, axis=1) == 0)
            tries += 1

    return A, B



def stable(n: int, m: int, rng) -> tuple[np.ndarray, np.ndarray]:
    """
    Draw (A,B) with A Hurwitz (CT-stable).
    Strategy: draw M ~ N(0,1), then shift by (alpha + max Re lambda(M))I
    """
    alpha = 0.10  # margin
    M = rng.normal(loc=1.0, scale=1.0, size=(n, n))
    lam = np.linalg.eigvals(M)
    shift = float(max(0.0, np.max(np.real(lam))) + alpha)
    A = M - shift * np.eye(n)
    B = rng.normal(loc=1.0, scale=1.0, size=(n, m))
    return A, B


def stable_continuous(n, m, rng, lam_min=0.2, lam_max=1.5):
    """
    Strictly Hurwitz A with a spectral margin (all eigenvalues <= -lam_min), and random B.
    """
    # Orthonormal basis
    Q, _ = np.linalg.qr(rng.standard_normal((n, n)))
    # Draw negative eigenvalues with margin
    lam = lam_min + rng.uniform(0.0, lam_max - lam_min, size=n)
    A = - (Q @ np.diag(lam) @ Q.T)   # symmetric negative-definite ⇒ Hurwitz with margin >= lam_min
    # B: standard normal
    B = rng.normal(loc=1.0, scale=1.0, size=(n, m))
    return A, B

# ---------------------------------------------------------------------
# Convenience adapter used by run_single / CLI
# ---------------------------------------------------------------------
def sample_system_instance(cfg, rng: np.random.Generator) -> Tuple[np.ndarray, np.ndarray]:
    """Dispatch by cfg.ensemble with cfg’s density fields.

    Supported:
    - 'ginibre', 'binary', 'stable', 'sparse'
    """
    if cfg.ensemble == "ginibre":
        return ginibre(cfg.n, cfg.m, rng)
    if cfg.ensemble == "binary":
        return binary(cfg.n, cfg.m, rng)
    if cfg.ensemble == "sparse":
        return sparse_continuous(
            n=cfg.n, m=cfg.m, rng=rng,
            which=getattr(cfg, "sparse_which", "both"),
            p_density=getattr(cfg, "p_density", 0.5),
            p_density_A=getattr(cfg, "_density_A", None),
            p_density_B=getattr(cfg, "_density_B", None),
            check_zero_rows=getattr(cfg, "check_zero_rows", False),
        )
    if cfg.ensemble == "stable":
        return stable(cfg.n, cfg.m, rng)
    raise ValueError(f"unknown ensemble: {cfg.ensemble}")


def draw_initial_state(n: int, mode: str, rng: np.random.Generator) -> np.ndarray:
    """Initial condition sampler."""
    if mode == "gaussian":
        x0 = rng.standard_normal(n)
    elif mode == "rademacher":
        x0 = rng.choice([-1.0, 1.0], size=n)
    elif mode == "ones":
        x0 = np.ones(n)
    elif mode == "zero":
        x0 = np.zeros(n)
    else:
        raise ValueError(f"Unknown x0_mode={mode}")
    return x0

# ---------------------------------------------------------------------
# New: controllability tools + (A,B) construction with target rank
# ---------------------------------------------------------------------

def _ctrb_matrix(A: np.ndarray, B: np.ndarray, order: Optional[int] = None) -> np.ndarray:
    """
    Controllability matrix C = [B, AB, A^2 B, ..., A^{order-1} B].
    If order is None, uses n = A.shape[0].
    """
    n = int(A.shape[0])
    m = int(B.shape[1])
    ordr: int = n if order is None else int(order)

    C = np.empty((n, m * ordr), dtype=A.dtype)
    Ak = np.eye(n, dtype=A.dtype)
    for k in range(ordr):
        C[:, k*m:(k+1)*m] = Ak @ B
        Ak = A @ Ak
    return C


def _svd_rank(M, rtol=None, atol=None):
    s = np.linalg.svd(M, compute_uv=False)
    smax = float(s[0]) if s.size else 0.0
    thr = 0.0
    if atol is None:
        atol = max(M.shape)*np.finfo(float).eps*smax
    thr = atol
    if rtol is not None:
        thr = max(thr, float(rtol)*smax)
    return int(np.sum(s > thr))



def controllability_rank(A: np.ndarray, B: np.ndarray, order: Optional[int] = None, rtol: Optional[float] = None) -> Tuple[int, np.ndarray]:
    """
    Return (rank, C) where C is the controllability matrix and rank = rank(C).
    """
    C = _ctrb_matrix(A, B, order=order)
    return _svd_rank(C, rtol=rtol), C


def _call_with_filtered(fn: Callable, /, **kwargs):
    """Call fn with only kwargs it accepts (lets us forward base generator kwargs safely)."""
    import inspect
    sig = inspect.signature(fn)
    filtered = {k: v for k, v in kwargs.items() if k in sig.parameters}
    return fn(**filtered)


def _draw_base_pair(base: str, n: int, m: int, rng: np.random.Generator, **base_kwargs) -> Tuple[np.ndarray, np.ndarray]:
    """
    Draw (A,B) using one of the existing base generators in this module.
    """
    base = base.lower()
    if base == "ginibre":
        return ginibre(n, m, rng)
    if base == "stable":
        return stable(n, m, rng)
    if base == "binary":
        return binary(n, m, rng)
    if base in {"sparse", "sparse_continuous"}:
        kw = _filter_kwargs(sparse_continuous, base_kwargs)
        return sparse_continuous(n=n, m=m, rng=rng, **kw)
    raise ValueError(f"unknown base generator '{base}'")


def draw_with_ctrb_rank(
    n: int,
    m: int,
    r: int,
    rng: np.random.Generator,
    *,
    ensemble_type: str = "ginibre",
    max_tries: int = 50,
    embed_random_basis: bool = True,
    base_kwargs: Optional[Dict[str, Any]] = None,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, Any]]:
    """
    Construct (A,B) with *exact* controllability rank r in dimension n with m inputs.

    Theory (Kalman decomposition):
      We build a block pair (A_blk, B_blk) in a basis [R, U] where
      - dim(R) = r, (A_c, B_c) is controllable on R,
      - dim(U) = n-r, B has no support on U (=> unreachable),
      then embed with a random orthonormal Q to avoid axis artifacts.

    Parameters
    ----------
    n, m : state and input dimensions
    r    : desired controllability rank (0 <= r <= n)
    rng  : NumPy Generator for reproducibility
    base_c : base generator for the controllable block (ginibre/stable/binary/sparse)
    max_tries : how many attempts to draw a controllable (A_c,B_c) of size rxm
    embed_random_basis : if True, embed blocks with a random orthonormal matrix Q
    base_kwargs_c / base_kwargs_u : forwarded to the respective base generators

    Returns
    -------
    A, B, meta
      meta includes:
        - "Q": the embedding basis (np.eye(n) if embed_random_basis=False)
        - "Ar": A_c, "Au": A_u, "Br": B_c
        - "R_basis": the embedded reachable basis Q[:, :r]
        - "rank": verified controllability rank (== r)
    """
    if not (0 <= r <= n):
        raise ValueError(f"r must be in [0, n]. Got r={r}, n={n}")
    if m < 1:
        raise ValueError("m must be ≥ 1")

    base_kwargs = base_kwargs or {}

    # (1) build controllable block (A_c,B_c) of size r (or empty if r=0)
    if r == 0:
        A_c = np.zeros((0, 0))
        B_c = np.zeros((0, m))
    else:
        tries = 0
        while True:
            tries += 1
            A_c, B_c = _draw_base_pair(ensemble_type, r, m, rng, **base_kwargs)
            rk, _ = controllability_rank(A_c, B_c, order=r)
            if rk == r:
                break
            if tries >= max_tries:
                raise RuntimeError(f"Could not draw controllable block of size r={r} from base '{ensemble_type}' after {max_tries} tries.")

    # (2) build unreachable block A_u of size n-r (or empty if r=n)
    d = n - r
    if d == 0:
        A_u = np.zeros((0, 0))
    else:
        A_u, _Bu_dummy = _draw_base_pair(ensemble_type, d, m, rng, **base_kwargs)

    # (3) assemble block pair in canonical coordinates
    if r == 0 and d == 0:
        raise ValueError("n=0 is not supported.")
    A_blk = np.block([
        [A_c,                   np.zeros((r, d))],
        [np.zeros((d, r)),      A_u           ],
    ])
    B_blk = np.vstack([B_c, np.zeros((d, m))])  # no actuation on U

    # (4) embed with random orthonormal basis (optional)
    Q = np.eye(n)
    if embed_random_basis:
        Q, _ = np.linalg.qr(rng.standard_normal((n, n)))  # Haar orthonormal

    A = Q @ A_blk @ Q.T
    B = Q @ B_blk

    # (5) verify controllability rank exactly r (use full order=n and robust rtol)
    rtols = (1e-10, 1e-8, 1e-6, 1e-12, 1e-14)
    rk_final = None
    for rtol in rtols:
        rk_final, R_basis_num = controllability_rank(A, B, order=n, rtol=rtol)
        if rk_final == r:
            break
    if rk_final != r:
        raise RuntimeError(
            f"Target controllability rank r={r} not achieved after embedding "
            f"(got {rk_final}). Try a looser rtol or re-draw."
        )

    meta = {
        "Q": Q, "Ar": A_c, "Au": A_u, "Br": B_c,
        "R_basis": R_basis_num,  # numeric reachable basis (orthonormal)
        "rank": rk_final,
    }

    return A, B, meta


# ---------------------------------------------------------------------
# New: "good vs. bad" initial states for uncontrollable pairs
# ---------------------------------------------------------------------
def initial_state_classifier(
    A: np.ndarray,
    B: np.ndarray,
    *,
    rtol: Optional[float] = None,
) -> Dict[str, Any]:
    """
    Characterize initial states w.r.t. V(x0) (visible subspace via unified generator).
    For an uncontrollable pair (A,B) with rank r < n, write X = R ⊕ U where
    R = reachable subspace (dim r). Then V(x0) = R ⊕ K_U(x0_U),
    where x0_U is x0 projected onto U and K_U is the Krylov span under A|_U.

    'Good' initial states are those with rank K_U(x0_U) = dim(U) (i.e., x0 is
    a cyclic vector for A|_U), which makes V(x0) = ℝⁿ. Others are 'bad'.

    Returns a dictionary with:
      - "rank": controllability rank r
      - "R_basis": orthonormal basis of R (nxr)
      - "U_basis": orthonormal complement basis (nx(n-r))
      - "is_good": callable x0 -> bool
      - "is_bad":  callable x0 -> bool
      - "sample_good":  callable rng -> x0 (tries until good)
      - "sample_bad":   callable rng -> x0 (constructs via eigenvector in U if possible)
    """
    n = A.shape[0]
    r, C = controllability_rank(A, B, order=n, rtol=rtol)

    # SVD gives orthonormal bases: left singular vectors of C
    U_left, svals, _ = np.linalg.svd(C, full_matrices=True)
    R_basis = U_left[:, :r]                      # reachable subspace basis
    U_basis = U_left[:, r:] if r < n else np.zeros((n, 0))   # complement

    # Projected dynamics on U (note: R is A-invariant; complement used here is fine)
    def _restrict_to_U(M: np.ndarray) -> np.ndarray:
        if U_basis.shape[1] == 0:
            return np.zeros((0, 0), dtype=M.dtype)
        return U_basis.T @ M @ U_basis

    A_U = _restrict_to_U(A)
    dimU = A_U.shape[0]

    def _krylov_rank_on_U(x0: np.ndarray) -> int:
        if dimU == 0:
            return 0
        xU = U_basis.T @ x0
        # Build Krylov [xU, A_U xU, ..., A_U^{dimU-1} xU]
        K = np.empty((dimU, dimU), dtype=A.dtype)
        vk = xU
        for k in range(dimU):
            K[:, k] = vk
            vk = A_U @ vk
        return _svd_rank(K, rtol=rtol)

    def is_good(x0: np.ndarray) -> bool:
        """True iff V(x0) = ℝⁿ (i.e., Krylov on U has full rank dimU)."""
        return _krylov_rank_on_U(x0) == dimU

    def is_bad(x0: np.ndarray) -> bool:
        return not is_good(x0)

    def sample_good(rng: np.random.Generator) -> np.ndarray:
        """Draw a random x0 until it is 'good' (almost sure in continuous draws)."""
        if dimU == 0:
            # Everything is trivially good if the pair is controllable already.
            return rng.standard_normal(n)
        for _ in range(100):
            # Mix reachable + unreachable components to avoid degeneracies
            x = R_basis @ rng.standard_normal(r) + U_basis @ rng.standard_normal(dimU)
            if is_good(x):
                return x
        # Extremely unlikely fallback
        return R_basis @ rng.standard_normal(r) + U_basis @ rng.standard_normal(dimU)

    def sample_bad(rng: np.random.Generator) -> np.ndarray:
        """
        Construct a 'bad' x0 by taking an eigenvector of A|_U (if available),
        which yields Krylov rank 1 on U; else degenerately choose a basis vector.
        """
        if dimU == 0:
            return np.zeros(n)  # no 'bad' states if already controllable
        try:
            w, V = np.linalg.eig(A_U)
            # pick an eigenvector with largest magnitude to improve conditioning
            idx = int(np.argmax(np.abs(w)))
            v = np.real_if_close(V[:, idx])
            v = v / (np.linalg.norm(v) + 1e-12)
            xU = v
        except Exception:
            # fallback: a coordinate axis in U
            e = np.zeros(dimU); e[0] = 1.0
            xU = e
        xR = np.zeros(r)
        return R_basis @ xR + U_basis @ xU

    return {
        "rank": r,
        "R_basis": R_basis,
        "U_basis": U_basis,
        "is_good": is_good,
        "is_bad": is_bad,
        "sample_good": sample_good,
        "sample_bad": sample_bad,
    }

def _filter_kwargs(fn, kwargs):
    import inspect
    sig = inspect.signature(fn)
    return {k: v for k, v in kwargs.items() if k in sig.parameters}

In [ ]:
from __future__ import annotations
import numpy as np
import numpy.linalg as npl
from typing import Tuple, Optional
from scipy.linalg import expm, eigvals, solve_continuous_lyapunov, solve_discrete_lyapunov, null_space, qr

# ---------------------------------------------------------------------
# Metrics & core objects 
# - Krylov generators (full / pointwise / PE-truncated)
# - Visible subspace basis
# - Gramian-based tests
# - PBH margins (structured/unstructured)    [kept here; can be moved to pbh.py]
# - Mode overlaps / projection errors
#
# Disclaimer:
# - Any “infinite-horizon” CT Gramian is only returned when A is Hurwitz.
# - PBH margins are evaluated exactly at eigenvalues; near-defective cases
#   may need small complex-radius sweeps (optional hook provided).
# ---------------------------------------------------------------------

def build_visible_basis_dt(Ad, Bd, x0, tol=1e-10, max_pow=None):
    """
    Returns P (n×k) with orthonormal columns spanning
    span{ [x0 Bd], Ad[x0 Bd], ..., Ad^{n-1}[x0 Bd] }.

    Prefers column-pivoted QR from SciPy; falls back to SVD if SciPy unavailable.
    """
    import numpy as np
    n = Ad.shape[0]
    if max_pow is None:
        max_pow = n - 1

    K0 = np.column_stack([x0.reshape(-1, 1), Bd])  # n×(1+m)
    blocks = [K0]
    Ak = Ad.copy()
    for _ in range(max_pow):
        blocks.append(Ak @ K0)
        Ak = Ak @ Ad
    M = np.concatenate(blocks, axis=1)  # n×((n)*(1+m))

    thr = tol * np.linalg.norm(M, 'fro')

    # Try SciPy pivoted QR first
    try:
        result = qr(M, mode='economic', pivoting=True)
        if len(result) == 3:
            Q, R, piv = result
        elif len(result) == 2:
            Q, R = result
        else:
            Q = result[0]
            R = np.zeros((Q.shape[1], Q.shape[1]), dtype=Q.dtype)  # fallback: dummy R
        diag = np.abs(np.diag(R))
        r = int((diag > thr).sum())
        Q = np.asarray(Q)  # ensure Q is a NumPy array for slicing
        return Q[:, :r]
    except Exception:
        # Fallback: SVD-based column space
        U, s, Vt = np.linalg.svd(M, full_matrices=False)
        r = int((s > thr).sum())
        return U[:, :r]




def same_equiv_class(A1: np.ndarray, B1: np.ndarray,
                     A2: np.ndarray, B2: np.ndarray,
                     x0: np.ndarray, tol: float = 1e-10) -> tuple[bool, dict]:
    """
    Check (A2,B2) ∈ [A1,B1]_{x0} under full input richness.
    Criterion (per thesis):
      (i) B2 == B1
      (ii) A2|_V == A1|_V on V = K(A1,[x0 B1]).

    Returns (ok, info) where ok is True/False and info has diagnostics.
    """
    # 1) exact-equality of B (within tol)
    dB = np.linalg.norm(B2 - B1, ord='fro')

    # 2) build visible subspace for (A1,B1,x0)
    P, k = visible_subspace(A1, B1, x0, tol=tol)  # P is n×k with orthonormal cols
    n = A1.shape[0]
    I = np.eye(n)

    # 3) compare restrictions on V and check leakage
    A1_V = P.T @ A1 @ P
    A2_V = P.T @ A2 @ P
    dA_V = np.linalg.norm(A2_V - A1_V, ord='fro')

    # Optional: ensure A2 maps V into V numerically (no leakage)
    leak_A2 = np.linalg.norm((I - P @ P.T) @ A2 @ P, ord='fro')

    ok = (dB <= tol) and (dA_V <= tol)  

    info = {
        "dim_V": int(k),
        "||B2-B1||_F": float(dB),
        "||A2|V - A1|V||_F": float(dA_V),
        "leak_A2": float(leak_A2),
        "tol": float(tol),
    }
    return bool(ok), info


def visible_basis_dt(Ad,Bd,x0,tol_rank=1e-12):
    K = unified_generator(Ad, Bd, x0, mode="unrestricted")
    U, s, _ = np.linalg.svd(K, full_matrices=False)
    r = int((s > tol_rank * s[0]).sum())  # scale by s[0]
    return U[:, :r]


# ====================== Discretization ===============================

def cont2discrete_zoh(A: np.ndarray, B: np.ndarray, dt: float) -> Tuple[np.ndarray, np.ndarray]:
    """Zero-order hold (ZOH) discretization of (A,B) with step dt."""
    n, m = B.shape
    M = np.zeros((n + m, n + m))
    M[:n, :n] = A * dt
    M[:n, n:] = B * dt
    Md = expm(M)
    Ad = Md[:n, :n]
    Bd = Md[:n, n:]
    return Ad, Bd



# ====================== Krylov / Visible subspace ====================

def _krylov(A: np.ndarray, X: np.ndarray, depth: int) -> np.ndarray:
    """Return [X, AX, ..., A^{depth-1} X]. If depth<=0, return n×0."""
    n = A.shape[0]
    if depth <= 0:
        return np.zeros((n, 0), dtype=A.dtype)
    blocks = [X]
    P = X
    for _ in range(1, depth):
        P = A @ P
        blocks.append(P)
    return np.concatenate(blocks, axis=1)



def krylov_generator(A: np.ndarray, X: np.ndarray, depth: Optional[int] = None) -> np.ndarray:
    """Public wrapper. If depth is None, use n (Cayley–Hamilton upper bound)."""
    n = A.shape[0]
    d = n if depth is None else int(depth)
    return _krylov(A, X, d)

def krylov_smin_norm(A, B, x0):
    n = len(x0)
    G = np.concatenate([x0.reshape(-1,1), B], axis=1)
    blocks = [G]
    for _ in range(n-1):
        G = A @ G
        blocks.append(G)
    K = np.concatenate(blocks, axis=1)
    K /= np.maximum(np.linalg.norm(K, axis=0, keepdims=True), 1e-12)  # column normalize
    s = np.linalg.svd(K, compute_uv=False)
    return float(s.min()) if s.size else 0.0



def unified_generator(
    A: np.ndarray,
    B: np.ndarray,
    x0: np.ndarray,
    mode: str = "unrestricted",
    W: Optional[np.ndarray] = None,
    r: Optional[int] = None,
) -> np.ndarray:
    """
    - mode="unrestricted": K = [x0, B, A[x0 B], ..., A^{n-1}[x0 B]]
    - mode="pointwise":    K = [x0, B@W, A[x0 B@W], ..., A^{n-1}[x0 B@W]]
    - mode="moment-pe":    K = [Krylov(A,[x0 B], r-1), A^r x0, ..., A^{n-1} x0]
    """
    n = A.shape[0]
    if mode == "unrestricted":
        X = np.concatenate([x0.reshape(-1, 1), B], axis=1)
        return _krylov(A, X, n)
    elif mode == "pointwise":
        if W is None:
            raise ValueError("pointwise mode requires W (mxq).")
        X = np.concatenate([x0.reshape(-1, 1), B @ W], axis=1)
        return _krylov(A, X, n)
    elif mode == "moment-pe":
        if r is None or r < 1:
            raise ValueError("moment-pe mode requires r>=1.")
        X = np.concatenate([x0.reshape(-1, 1), B], axis=1)
        Khead = _krylov(A, X, max(1, min(r, n)))           # depth r
        # add [A^r x0, ..., A^{n-1} x0]
        v = x0.copy()
        for _ in range(r):
            v = A @ v
        tail = [v.reshape(-1, 1)]
        for _ in range(r + 1, n):
            v = A @ v
            tail.append(v.reshape(-1, 1))
        Ktail = np.concatenate(tail, axis=1) if tail else np.zeros((n, 0))
        return np.concatenate([Khead, Ktail], axis=1)
    else:
        raise ValueError(f"unknown mode: {mode}")

def visible_subspace_basis(
    A: np.ndarray,
    B: np.ndarray,
    x0: np.ndarray,
    *,
    mode: str = "unrestricted",
    W: Optional[np.ndarray] = None,
    r: Optional[int] = None,
    tol: float = 1e-10,
):
    """Return (P_V, k) where columns of P_V span the visible subspace."""
    K = unified_generator(A, B, x0, mode=mode, W=W, r=r)
    if K.size == 0:
        return np.zeros((A.shape[0], 0)), 0
    U, S, _ = npl.svd(K, full_matrices=False)
    k = int((S > tol).sum())
    return U[:, :k], k



def visible_subspace(
    A: np.ndarray,
    B: np.ndarray,
    x0: np.ndarray,
    mode: str = "unrestricted",
    W: Optional[np.ndarray] = None,
    r: Optional[int] = None,
    tol: float = 1e-10,
) -> Tuple[np.ndarray, int]:
    """Return an orthonormal basis P_V for the visible subspace V_• and its dimension.

    We compute P_V via rank-revealing SVD of the unified generator K(𝒰; x0).
    """
    K = unified_generator(A, B, x0, mode=mode, W=W, r=r)
    if K.size == 0:
        return np.zeros((A.shape[0], 0)), 0
    U, S, _ = npl.svd(K, full_matrices=False)
    k = int((S > tol).sum())
    return U[:, :k], k


def projector_from_basis(Vbasis: np.ndarray) -> np.ndarray:
    """Orthogonal projector onto span(Vbasis)."""
    if Vbasis.size == 0:
        return np.zeros((0, 0), dtype=float)
    Q, _ = npl.qr(Vbasis, mode="reduced")
    return Q @ Q.T


# ====================== Gramians =====================================

def is_hurwitz(A: np.ndarray, tol: float = 0.0) -> bool:
    lam = npl.eigvals(A)
    return bool(np.all(np.real(lam) < -tol))

def gramian_ct_infinite(A: np.ndarray, K: np.ndarray):
    """CT infinite-horizon Gramian for ẋ = A x + K w, if A is Hurwitz."""
    if not is_hurwitz(A, 0.0):
        return None
    try:
        return solve_continuous_lyapunov(A, -K @ K.T)
    except Exception:
        return None

def gramian_dt_infinite(Ad: np.ndarray, K: np.ndarray):
    """DT infinite-horizon Gramian for x_{k+1} = Ad x_k + K w_k, if ρ(Ad)<1."""
    try:
        rho = float(np.max(np.abs(npl.eigvals(Ad))))
    except Exception:
        rho = np.inf
    if rho < 1.0 - 1e-8:
        try:
            return solve_discrete_lyapunov(Ad, K @ K.T)
        except Exception:
            return None
    return None

def gramian_dt_finite(Ad: np.ndarray, K: np.ndarray, T: int) -> np.ndarray:
    """DT finite-horizon Gramian for x_{k+1}=Ad x_k + K w_k over T steps."""
    n = Ad.shape[0]
    W = np.zeros((n, n), dtype=float)
    P = np.eye(n)
    for _ in range(T):
        W = W + P @ (K @ K.T) @ P.T
        P = Ad @ P
    return W



# ====================== PBH margins (to move to pbh.py if desired) ===

def pbh_margin_structured(
    A: np.ndarray,
    B: np.ndarray,
    x0: np.ndarray,
    eigvals: Optional[np.ndarray] = None,
) -> float:
    """Structured Frobenius-distance proxy (TODO: link to the manuscript).

    Q has columns forming an orthonormal basis of x0^⊥, computed via null_space(x0^T).
    """
    if eigvals is None:
        eigvals = npl.eigvals(A)
    x = x0.reshape(-1, 1)
    Q = null_space(x.T)
    n = A.shape[0]
    aug = np.concatenate([x, B], axis=1)
    margin = np.inf
    for lam in eigvals:
        M = np.concatenate([lam * np.eye(n) - A, aug], axis=1).astype(np.complex128)
        smin = npl.svd(Q.T @ M, compute_uv=False).min().real if Q.size else npl.svd(M, compute_uv=False).min().real
        margin = min(margin, float(smin))
    return float(margin)


def pbh_margin_unstructured(
    A: np.ndarray,
    K: np.ndarray,
    eigvals: Optional[np.ndarray] = None,
) -> float:
    """Unstructured Frobenius-distance proxy: min_λ σ_min([λI−A, K])."""
    lam = npl.eigvals(A) if eigvals is None else eigvals
    n = A.shape[0]
    best = np.inf
    for l in lam:
        M = np.concatenate([l * np.eye(n) - A, K], axis=1)
        s = npl.svd(M, compute_uv=False)
        best = min(best, float(s[-1]))
    return best

def pbh_margin_with_ring(A: np.ndarray, K: np.ndarray,
                         ring_eps: float = 1e-6, ring_pts: int = 8) -> float:
    lam = np.linalg.eigvals(A)
    n = A.shape[0]
    margin = np.inf
    for l in lam:
        for t in range(ring_pts):
            theta = 2*np.pi*(t/ring_pts)
            lam_s = l + ring_eps*(np.cos(theta) + 1j*np.sin(theta))
            M = np.concatenate([lam_s*np.eye(n) - A, K], axis=1).astype(np.complex128)
            smin = np.linalg.svd(M, compute_uv=False).min().real
            margin = min(margin, float(smin))
    return float(margin)



# ====================== Overlaps / errors ============================

def left_eigvec_overlap(A: np.ndarray, X: np.ndarray) -> np.ndarray:
    """Return normalized overlaps with left eigenvectors of A.

    s_i = ||w_i^T X||_2 / (||w_i||_2 * ||X||_2), so s_i ∈ [0, 1].
    """
    lam, W = npl.eig(A.T)  # columns are eigenvectors of A^T
    num = npl.norm(W.T @ X, axis=1)
    den = npl.norm(W, axis=0)
    den = np.where(den == 0, 1.0, den)
    xnorm = float(npl.norm(X, 2))
    if not np.isfinite(xnorm) or xnorm == 0.0:
        xnorm = 1.0
    return (num / (den * xnorm)).real
def x0R0_principle_angle(x0: np.ndarray, R0: np.ndarray) -> float:
    """Cosine of principal angle between x0 and span(R0): ||P_R x0|| / ||x0||."""
    x0 = x0.reshape(-1, 1)
    if x0.shape[0] != R0.shape[0]:
        raise ValueError("x0 and R0 row dimensions must match.")
    if npl.norm(x0) == 0:
        return 0.0
    # projector onto span(R0)
    Rplus = npl.pinv(R0)
    PR = R0 @ Rplus
    return float(npl.norm(PR @ x0) / npl.norm(x0))

def pair_distance(
        Ahat: np.ndarray, 
        Bhat: np.ndarray, 
        A: np.ndarray, 
        B: np.ndarray) -> float:
    errA = float(npl.norm(Ahat - A, "fro"))
    errB = float(npl.norm(Bhat - B, "fro"))
    return float(np.mean([errA, errB]))


def projected_errors(
    Ahat: np.ndarray,
    Bhat: np.ndarray,
    A: np.ndarray,
    B: np.ndarray,
    Vbasis: np.ndarray,
) -> Tuple[float, float]:
    """||P_V (Ahat - A) P_V||_F, ||P_V (Bhat - B)||_F with P_V projector from Vbasis."""
    PV = projector_from_basis(Vbasis)
    dA = PV @ (Ahat - A) @ PV
    dB = PV @ (Bhat - B)
    return float(npl.norm(dA, "fro")), float(npl.norm(dB, "fro"))

def controllability_subspace_basis(A: np.ndarray, B: np.ndarray, rtol: float = 1e-10) -> np.ndarray:
    """Basis of span([B, AB, ..., A^{n-1}B]) via SVD (thin)."""
    n = A.shape[0]
    blocks = []
    Ak = np.eye(n)
    for k in range(n):
        blocks.append(Ak @ B)
        Ak = Ak @ A
    C = np.concatenate(blocks, axis=1)
    U, S, _ = np.linalg.svd(C, full_matrices=False)
    r = int(np.sum(S > rtol))
    return U[:, :r]

def eta0(A: np.ndarray, B: np.ndarray, x0: np.ndarray, rtol: float = 1e-10) -> float:
    """eta = ||Proj_{span(ctrl)} x0|| / ||x0||, ctrl = span([B, AB, ...])."""
    if np.linalg.norm(x0) == 0.0:
        return 0.0
    V = controllability_subspace_basis(A, B, rtol=rtol)
    px = V @ (V.T @ x0)
    return float(np.linalg.norm(px) / (np.linalg.norm(x0) + 1e-12))

def left_eig_overlaps(A: np.ndarray, x0: np.ndarray, B: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    alpha_i = |v_i^T x0| / ||x0|| ; beta_i = ||v_i^T B||_2  where v_i are (unit) left eigenvectors of A.
    Returns (eigs, alpha, beta). Handles complex values; returns real magnitudes.
    """
    w, Vt = np.linalg.eig(A.T)           # columns of Vt are left eigenvectors of A
    V = np.asarray(Vt)
    norms = np.linalg.norm(V, axis=0)
    norms[norms == 0.0] = 1.0
    V = V / norms
    alpha = np.abs(V.T.conj() @ x0) / (np.linalg.norm(x0) + 1e-12)
    beta = np.linalg.norm(V.T.conj() @ B, axis=1)
    return w, alpha.real, beta.real

def ctrl_growth_metrics(A: np.ndarray, B: np.ndarray, rtol: float = 1e-10) -> tuple[int, int]:
    """
    Simple Brunovsky-like growth: track ranks of C_k = [B, AB, ..., A^{k-1}B].
    nu_max = smallest k with rank(C_k) = n; nu_gap = last_increase - first_increase (in steps).
    If never full rank, nu_max is current k*, nu_gap computed on observed increases.
    """
    n = A.shape[0]
    blocks = []
    growth_steps = []
    rank_prev = 0
    for k in range(1, n + 1):
        blocks.append(np.linalg.matrix_power(A, k - 1) @ B)
        Ck = np.concatenate(blocks, axis=1)
        r = int(np.linalg.matrix_rank(Ck, tol=rtol))
        if r > rank_prev:
            growth_steps.append(k)
            rank_prev = r
        if r >= n:
            break
    if not growth_steps:
        return 0, 0
    nu_max = growth_steps[-1]
    nu_gap = growth_steps[-1] - growth_steps[0]
    return int(nu_max), int(nu_gap)


# --- Legacy shims replacing pyident/sys_utils.py ---

def c2d(A: np.ndarray, B: np.ndarray, dt: float):
    """
    Backward-compatible alias for ZOH discretization.
    Old name from sys_utils.c2d -> now calls cont2discrete_zoh.
    Returns (Ad, Bd).
    """
    return cont2discrete_zoh(A, B, dt)


def simulate(T: int, x0: np.ndarray, Ad: np.ndarray, Bd: np.ndarray, u: np.ndarray) -> np.ndarray:
    """
    Deterministic DT simulation (legacy signature from sys_utils.simulate):
        X[:,0] = x0
        X[:,t+1] = Ad @ X[:,t] + Bd @ u_t
    Accepts u as either shape (m, T) or (T, m). Returns X of shape (n, T+1).
    """
    if u.ndim != 2:
        raise ValueError(f"u must be 2D, got shape {u.shape}")
    m = Bd.shape[1]
    if u.shape == (m, T):      # (m, T)
        U = u.T
    elif u.shape == (T, m):    # (T, m)
        U = u
    else:
        raise ValueError(f"Incompatible u shape {u.shape} for B shape {Bd.shape} and T={T}")

    n = Ad.shape[0]
    X = np.zeros((n, T + 1), dtype=Ad.dtype)
    X[:, 0] = x0
    for t in range(T):
        X[:, t + 1] = Ad @ X[:, t] + Bd @ U[t, :]
    return X


# --- NEW: regressor diagnostics -------------------------------------
def regressor_stats(X0: np.ndarray, U: np.ndarray, rtol_rank: float = 1e-12) -> dict:
    """Rank/cond of Z=[X0;U]."""
    Z = np.vstack([X0, U])
    s = npl.svd(Z, compute_uv=False)
    if s.size == 0:
        return {"rank": 0, "cond": np.inf, "smin": 0.0}
    rank = int(np.sum(s > rtol_rank * s[0]))
    cond = float(s[0] / (s[-1] + 1e-18))
    return {"rank": rank, "cond": cond, "smin": float(s[-1])}


def regressor_stats_on_V(PV: np.ndarray, X0: np.ndarray, U: np.ndarray, rtol: float = 1e-12) -> dict:
    """
    Rank/condition of the regressor restricted to V: Z_V = [PV^T X0; U].
    """
    ZV = np.vstack([PV.T @ X0, U])
    s = npl.svd(ZV, compute_uv=False)
    if s.size == 0:
        return {"rank": 0, "cond": np.inf, "smin": 0.0}
    rank = int(np.sum(s > rtol * s[0]))
    cond = float(s[0] / (s[-1] + 1e-18))
    return {"rank": rank, "cond": cond, "smin": float(s[-1])}

# --- NEW: DT theoretical-class checker (relative thresholds) ----------
def same_equiv_class_dt_rel(Ad, Bd, Ahat, Bhat, x0,
                            rtol_eq: float = 1e-2,
                            rtol_rank: float = 1e-12,
                            use_leak: bool = True):
    # robust V basis
    K = unified_generator(Ad, Bd, x0, mode="unrestricted")
    if K.size == 0:
        return False, {"dim_V": 0}
    U, s, _ = npl.svd(K, full_matrices=False)
    k = int(np.sum(s > rtol_rank * s[0]))
    P = U[:, :k]; I = np.eye(Ad.shape[0])

    dA_V = float(npl.norm(P.T @ (Ahat - Ad) @ P, "fro"))
    leak = float(npl.norm((I - P @ P.T) @ Ahat @ P, "fro"))
    dB   = float(npl.norm(Bhat - Bd, "fro"))

    thrA = rtol_eq * max(1.0, npl.norm(P.T @ Ad @ P, "fro"))
    thrB = rtol_eq * max(1.0, npl.norm(Bd, "fro"))
    thrL = rtol_eq * max(1.0, npl.norm(Ad, "fro"))

    ok = (dA_V <= thrA) and (dB <= thrB) 
    info = {"dim_V": int(k), "dA_V": dA_V, "leak": leak, "dB": dB,
            "thrA": thrA, "thrB": thrB, "thrLeak": thrL}
    return ok, info

# --- NEW: data-equivalence residual ----------------------------------
def data_equivalence_residual(X0, X1, U, Ahat, Bhat, rtol: float = 1e-10):
    if U.ndim != 2:
        raise ValueError(f"U must be 2-D, got shape {U.shape}")
    T = X0.shape[1]
    if U.shape[1] == T:
        U_cm = U
    elif U.shape[0] == T:
        U_cm = U.T
    else:
        raise ValueError(f"U shape {U.shape} incompatible with time length {T}")

    R = X1 - Ahat @ X0 - Bhat @ U_cm
    rel = float(npl.norm(R, "fro") / (npl.norm(X1, "fro") + 1e-18))
    return (rel <= rtol), {"resid_rel": rel}

# --- NEW: DT Markov-parameter match ----------------------------------
def markov_params_dt(Ad, Bd, kmax: int):
    M = []
    Ak = np.eye(Ad.shape[0])
    for _ in range(kmax):
        M.append(Ak @ Bd)
        Ak = Ad @ Ak
    return M

def markov_match_dt(Ad, Bd, Ahat, Bhat, kmax: int, rtol: float = 1e-2):
    M  = markov_params_dt(Ad, Bd, kmax)
    Mh = markov_params_dt(Ahat, Bhat, kmax)
    errs = [float(npl.norm(Mh[k]-M[k],"fro") / (npl.norm(M[k],"fro")+1e-12)) for k in range(kmax)]
    ok = (max(errs) <= rtol)
    return ok, {"markov_errs": errs, "markov_err_max": float(max(errs))}

def mu_left_eigs(A, x0, B):
    wvals, Wt = np.linalg.eig(A.T)      # columns: left eigenvectors
    Wi = np.asarray(Wt); norms = np.linalg.norm(Wi, axis=0); norms[norms==0]=1.0
    Wi = Wi / norms
    X = np.concatenate([x0.reshape(-1,1), B], axis=1)
    mu = np.linalg.norm(Wi.T.conj() @ X, axis=1)  # == ||w_i^T [x0 B]||_2
    return wvals, mu

def x0_score(x0, cfg, rng, Ad, Bd, U):
    """Evaluate PE/conditioning scores for a candidate initial state."""
    X_nf = simulate_dt(x0, Ad, Bd, U, noise_std=0.0, rng=rng)
    X0_nf, X1_nf = X_nf[:, :-1], X_nf[:, 1:]
    P, _ = visible_subspace(Ad, Bd, x0)          # basis of V(x0)
    ZV = np.vstack([P.T @ X0_nf, U.T])                     # (k+m, T)
    svals = np.linalg.svd(ZV, compute_uv=False)
    kappa = (svals[0] / (svals[-1] + 1e-18)) if svals.size else np.inf
    cond_factor = (svals[-1] / (svals[0] + 1e-18))       # == 1/kappa
    margin = pbh_margin_structured(Ad, Bd, x0)
    return margin * cond_factor

In [ ]:
from __future__ import annotations
from typing import Tuple, Optional
import numpy as np
from numpy.linalg import svd, norm
from scipy.linalg import qr

EPS = 1e-12


def subspace_from_states(X: np.ndarray, tol: float = EPS) -> np.ndarray:
    """
    Orthonormal basis for ``span{ x_0, …, x_{T-1} }`` from a state snapshot matrix.

    Parameters
    ----------
    X : array, shape (n, T)
        Snapshot matrix whose columns are consecutive states.
    tol : float, optional
        Numerical tolerance used for rank detection.
    """
    X = np.asarray(X, dtype=float)
    if X.size == 0:
        n = X.shape[0] if X.ndim >= 1 else 0
        return np.zeros((n, 0), dtype=float)

    n, T = X.shape
    U, s, _ = svd(X, full_matrices=False)
    thr = tol * (s[0] if s.size else 1.0)
    r = int(np.sum(s > thr))
    if r == 0:
        return np.zeros((n, 0), dtype=float)
    return U[:, :r]


def visible_from_traj(X: np.ndarray, tol: float = EPS) -> np.ndarray:
    """
    Empirical visible subspace estimated from a state trajectory.

    Returns an orthonormal basis for the span of the provided states, matching
    the data-only estimator used in the experimental pipeline.
    """
    return subspace_from_states(X, tol=tol)


def build_visible_basis_dt(Ad, Bd, x0, tol=1e-10, max_pow=None):
    """
    Returns P (n×k) with orthonormal columns spanning
    span{ [x0 Bd], Ad[x0 Bd], ..., Ad^{n-1}[x0 Bd] }.

    Prefers column-pivoted QR from SciPy; falls back to SVD if SciPy unavailable.
    """
    import numpy as np
    n = Ad.shape[0]
    if max_pow is None:
        max_pow = n - 1

    K0 = np.column_stack([x0.reshape(-1, 1), Bd])  # n×(1+m)
    blocks = [K0]
    Ak = Ad.copy()
    for _ in range(max_pow):
        blocks.append(Ak @ K0)
        Ak = Ak @ Ad
    M = np.concatenate(blocks, axis=1)  # n×((n)*(1+m))

    thr = tol * np.linalg.norm(M, 'fro')

    # Try SciPy pivoted QR first
    try:
        result = qr(M, mode='economic', pivoting=True)
        if len(result) == 3:
            Q, R, piv = result
        elif len(result) == 2:
            Q, R = result
        else:
            Q = result[0]
            R = np.zeros((Q.shape[1], Q.shape[1]), dtype=Q.dtype)  # fallback: dummy R
        diag = np.abs(np.diag(R))
        r = int((diag > thr).sum())
        Q = np.asarray(Q)  # ensure Q is a NumPy array for slicing
        return Q[:, :r]
    except Exception:
        # Fallback: SVD-based column space
        U, s, Vt = np.linalg.svd(M, full_matrices=False)
        r = int((s > thr).sum())
        return U[:, :r]

# ---------- basic LA helpers ----------
def _svd_nullspace(M: np.ndarray, tol: Optional[float] = None) -> np.ndarray:
    """
    Orthonormal basis for ker(M) using SVD rank cut.
    Works for tall/wide matrices. Returns n x k matrix (possibly k=0).
    """
    if M.size == 0:
        return np.zeros((M.shape[1], 0))
    U, S, Vt = np.linalg.svd(M, full_matrices=True)
    if tol is None:
        tol = max(M.shape) * np.finfo(float).eps * (S[0] if S.size else 1.0)
    r = int(np.sum(S > tol))                # numerical rank
    return Vt.T[:, r:]                      # the last n-r columns span ker(M)


def _orth(A: np.ndarray) -> np.ndarray:
    if A.size == 0:
        return np.zeros((A.shape[0], 0))
    U, S, Vt = svd(A, full_matrices=False)
    tol = max(A.shape) * np.finfo(float).eps * (S[0] if S.size else 1.0)
    r = int(np.sum(S > tol))
    return U[:, :r]

def projector_from_basis(Vbasis):
    n = Vbasis.shape[0] if Vbasis.size else 0
    if Vbasis.size == 0:
        return np.zeros((n, n))
    Q, _ = np.linalg.qr(Vbasis, mode="reduced")
    return Q @ Q.T


def projector_onto_complement(Vbasis: np.ndarray) -> np.ndarray:
    """Orthogonal projector onto ``span(Vbasis)`` :sup:`⊥`.

    For an empty basis this reduces to the identity on the ambient space.
    """

    n = Vbasis.shape[0] if Vbasis.ndim >= 1 else 0
    if Vbasis.size == 0 or Vbasis.shape[1] == 0:
        return np.eye(n)

    return np.eye(n) - projector_from_basis(Vbasis)


def principal_angles(P: np.ndarray, Q: np.ndarray, rtol: float = 1e-12) -> np.ndarray:
    """
    Principal angles (radians) between the spans of P and Q.

    Returns an empty array if either basis is empty.
    """
    if P.size == 0 or Q.size == 0:
        return np.zeros((0,), dtype=float)
    P_orth, _ = np.linalg.qr(P, mode="reduced")
    Q_orth, _ = np.linalg.qr(Q, mode="reduced")
    s = np.linalg.svd(P_orth.T @ Q_orth, compute_uv=False)
    s = np.clip(s, -1.0, 1.0)
    if s.size == 0:
        return np.zeros((0,), dtype=float)
    s_eff = s[s > rtol * s[0]]
    if s_eff.size == 0:
        return np.zeros((0,), dtype=float)
    return np.arccos(s_eff)


def projector_gap(P: np.ndarray, Q: np.ndarray) -> float:
    """Spectral-norm gap ||Π_P - Π_Q||_2 between spans of P and Q."""
    if P.size == 0 and Q.size == 0:
        return 0.0
    PP = projector_from_basis(P)
    QQ = projector_from_basis(Q)
    return float(norm(PP - QQ, 2))


def normalize(x: np.ndarray, tol: float = 1e-12) -> np.ndarray:
    nrm = float(norm(x))
    if nrm <= tol:
        raise ValueError("Vector too small to normalize.")
    return x / nrm

# ---------- core: left-uncontrollable space ----------
def left_uncontrollable_subspace(A: np.ndarray, B: np.ndarray,
                                 tol: Optional[float] = None,
                                 max_iter: int = 50) -> np.ndarray:
    """
    Largest A^T-invariant subspace contained in ker(B^T).
    """
    n = A.shape[0]
    Wk = _svd_nullspace(B.T, tol=tol)      # start in ker(B^T)
    if Wk.shape[1] == 0:
        return Wk
    Wk = _orth(Wk)

    for _ in range(max_iter):
        Pk = projector_onto_complement(Wk)      # Pk = I - Wk Wk^T  (projects onto (span Wk)^\perp)
        N  = np.vstack([Pk @ A.T, B.T])    # enforce: Pk A^T w = 0 and B^T w = 0
        Wn = _svd_nullspace(N, tol=tol)
        Wn = _orth(Wn)

        # convergence: projector distance and/or dim stability
        if (Wn.shape[1] == Wk.shape[1] and
            norm(projector_onto_complement(Wn) - projector_onto_complement(Wk), 2) <= (tol or 1e-10)):
            return Wn
        Wk = Wn
    return Wk


# ---------- selecting a subset to keep dark ----------
def choose_dark_subset(W_all: np.ndarray, k: int = 1, rng: Optional[np.random.Generator] = None) -> np.ndarray:
    """Pick k columns from an orthonormal basis W_all to suppress (k ≥ 1)."""
    k = int(max(1, min(k, W_all.shape[1])))
    if W_all.shape[1] == 0:
        return W_all
    rng = np.random.default_rng() if rng is None else rng
    idx = rng.choice(W_all.shape[1], size=k, replace=False)
    return W_all[:, idx]

# ---------- main builders ----------
def make_dark_projector(A: np.ndarray, B: np.ndarray, k_off: int = 1,
                        rng: Optional[np.random.Generator] = None,
                        tol: Optional[float] = None) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Returns (W_all, W_off, P_dark) where W_all spans all uncontrollable left modes,
    W_off (⊆ W_all) has k_off columns chosen to stay dark, and P_dark projects onto ker(W_off^T).
    """
    W_all = left_uncontrollable_subspace(A, B, tol=tol)
    if W_all.shape[1] == 0:
        n = A.shape[0]
        return W_all, np.zeros((n, 0)), np.eye(n)
    W_off = choose_dark_subset(W_all, k=k_off, rng=rng)
    # columns of W_all are orthonormal; subset preserves orthonormality
    P_dark = projector_onto_complement(W_off)
    return W_all, W_off, P_dark

def build_projected_x0(A: np.ndarray, B: np.ndarray, x0_seed: np.ndarray,
                       k_off: int = 1, rng: Optional[np.random.Generator] = None,
                       tol: Optional[float] = None) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Project x0_seed onto ker(W_off^T) to make at least k_off uncontrollable directions dark.
    Returns (x0_proj, W_off, P_dark).
    """
    W_all, W_off, P_dark = make_dark_projector(A, B, k_off=k_off, rng=rng, tol=tol)
    x = P_dark @ x0_seed
    if norm(x) <= (tol or 1e-10):
        # try a few random seeds
        rng = np.random.default_rng() if rng is None else rng
        for _ in range(32):
            x_try = P_dark @ rng.standard_normal(A.shape[0])
            if norm(x_try) > (tol or 1e-10):
                x = x_try
                break
    if norm(x) <= (tol or 1e-10):
        # if still tiny (e.g., k_off == n), fall back to any unit vector in ker(W_off^T)
        # find an orth basis for ker(W_off^T) by orthonormal complement of W_off
        n = A.shape[0]
        if W_off.shape[1] >= n:
            raise RuntimeError("ker(W_off^T) is trivial; cannot project.")
        Qc = _orth(np.eye(n) - W_off @ W_off.T)
        x = Qc[:, 0]
    return normalize(x), W_off, P_dark

In [ ]:
# """Shared matplotlib boxplot styling helpers for sim_unctrb_* scripts."""
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


def set_default_mpl_style() -> None:
    """Lightweight, publication-friendly defaults."""
    plt.rcParams.update(
        {
            "figure.dpi": 120,
            "savefig.dpi": 300,
            "font.size": 11,
            "axes.titlesize": 12,
            "axes.labelsize": 11,
            "xtick.labelsize": 10,
            "ytick.labelsize": 10,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.linewidth": 1.0,
            "grid.alpha": 0.25,
            "grid.linewidth": 0.8,
        }
    )


def _finite_or_empty(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    return x[np.isfinite(x)]


def nice_boxplot(
    ax,
    data,
    labels,
    *,
    colors=None,
    colorize: bool = True,
    title=None,
    ylabel=None,
    whis=(5, 95),
    showfliers=False,
    notch=False,
    widths=0.6,
    jitter_points=True,
    jitter_alpha=0.25,
    point_size=10,
    annotate_n=True,
    median_linewidth: float = 2.0,
    yscale=None,
):
    """A clean default boxplot with optional jitter overlay and n-annotation."""
    clean = [_finite_or_empty(d) for d in data]
    clean = [c if c.size > 0 else np.array([np.nan]) for c in clean]

    bp = ax.boxplot(
        clean,
        labels=labels,
        patch_artist=True,
        widths=widths,
        whis=whis,
        showfliers=showfliers,
        notch=notch,
        medianprops=dict(linewidth=median_linewidth),
        whiskerprops=dict(linewidth=1.2),
        capprops=dict(linewidth=1.2),
        boxprops=dict(linewidth=1.2),
        flierprops=dict(markersize=3, alpha=0.35),
    )

    if colorize:
        if colors is None:
            colors = [f"C{i}" for i in range(len(labels))]
        for box, c in zip(bp["boxes"], colors):
            box.set_facecolor(c)
            box.set_alpha(0.25)
            box.set_edgecolor(c)
    else:
        for box in bp["boxes"]:
            box.set_facecolor("none")
            box.set_alpha(1.0)
            box.set_edgecolor("0.25")

    for k in ("whiskers", "caps", "medians"):
        for line in bp[k]:
            line.set_color("0.25")

    ax.grid(True, axis="y")
    ax.set_axisbelow(True)

    if title:
        ax.set_title(title)
    if ylabel:
        ax.set_ylabel(ylabel)

    if yscale is not None:
        ax.set_yscale(yscale)

    if jitter_points:
        rng = np.random.default_rng(0)
        for i, vals in enumerate(clean, start=1):
            vals = vals[np.isfinite(vals)]
            if vals.size == 0:
                continue
            x = i + rng.uniform(-0.12, 0.12, size=vals.size)
            ax.scatter(x, vals, s=point_size, alpha=jitter_alpha, linewidths=0)

    if annotate_n:
        ymin, ymax = ax.get_ylim()
        ytext = ymin + 0.02 * (ymax - ymin)
        for i, vals in enumerate(clean, start=1):
            n = int(np.isfinite(vals).sum())
            ax.text(i, ytext, f"n={n}", ha="center", va="bottom", fontsize=9, alpha=0.8)

    return bp


def nice_grouped_boxplot(
    ax,
    data_a,
    data_b,
    labels,
    *,
    label_a="standard",
    label_b="P-basis",
    color_a="#4C78A8",
    color_b="#F58518",
    whis=(5, 95),
    showfliers=False,
    widths=0.55,
    gap=2.4,
    inner=0.8,
    title=None,
    ylabel=None,
    jitter_points=True,
    yscale=None,
):
    """Side-by-side boxplots per algorithm, consistent legend + styling."""
    A = [_finite_or_empty(d) for d in data_a]
    B = [_finite_or_empty(d) for d in data_b]
    A = [c if c.size > 0 else np.array([np.nan]) for c in A]
    B = [c if c.size > 0 else np.array([np.nan]) for c in B]

    positions, all_data, tickpos = [], [], []
    for i in range(len(labels)):
        base = i * gap
        positions += [base, base + inner]
        all_data += [A[i], B[i]]
        tickpos.append(base + inner / 2)

    bp = ax.boxplot(
        all_data,
        positions=positions,
        widths=widths,
        patch_artist=True,
        whis=whis,
        showfliers=showfliers,
        medianprops=dict(linewidth=2.0),
        whiskerprops=dict(linewidth=1.2),
        capprops=dict(linewidth=1.2),
        boxprops=dict(linewidth=1.2),
        flierprops=dict(markersize=3, alpha=0.35),
    )

    colors = [color_a, color_b]
    for i, box in enumerate(bp["boxes"]):
        c = colors[i % 2]
        box.set_facecolor(c)
        box.set_alpha(0.25)
        box.set_edgecolor(c)

    for k in ("whiskers", "caps", "medians"):
        for line in bp[k]:
            line.set_color("0.25")

    ax.set_xticks(tickpos)
    ax.set_xticklabels(labels)
    ax.grid(True, axis="y")
    ax.set_axisbelow(True)

    if title:
        ax.set_title(title)
    if ylabel:
        ax.set_ylabel(ylabel)
    if yscale is not None:
        ax.set_yscale(yscale)

    if jitter_points:
        rng = np.random.default_rng(0)
        for j, vals in enumerate(all_data):
            vals = vals[np.isfinite(vals)]
            if vals.size == 0:
                continue
            x0 = positions[j]
            x = x0 + rng.uniform(-0.10, 0.10, size=vals.size)
            ax.scatter(x, vals, s=10, alpha=0.22, linewidths=0)

    ax.legend(
        handles=[
            Patch(facecolor=color_a, edgecolor=color_a, alpha=0.25, label=label_a),
            Patch(facecolor=color_b, edgecolor=color_b, alpha=0.25, label=label_b),
        ],
        frameon=False,
        loc="upper right",
    )
    return bp


def nice_violinplot(
    ax,
    data,
    labels,
    *,
    colors=None,
    colorize: bool = True,
    title=None,
    ylabel=None,
    widths=0.8,
    showmeans=False,
    showmedians=True,
    showextrema=False,
    jitter_points=True,
    jitter_alpha=0.25,
    point_size=10,
    annotate_n=True,
    median_linewidth: float = 2.0,
    yscale=None,
):
    """A clean default violin plot with optional jitter overlay and n-annotation."""
    clean = [_finite_or_empty(d) for d in data]
    clean = [c if c.size > 0 else np.array([np.nan]) for c in clean]

    vp = ax.violinplot(
        clean,
        showmeans=showmeans,
        showmedians=showmedians,
        showextrema=showextrema,
        widths=widths,
    )

    if colorize:
        if colors is None:
            colors = [f"C{i}" for i in range(len(labels))]
        for body, c in zip(vp["bodies"], colors):
            body.set_facecolor(c)
            body.set_alpha(0.25)
            body.set_edgecolor(c)
            body.set_linewidth(1.1)
    else:
        for body in vp["bodies"]:
            body.set_facecolor("none")
            body.set_alpha(1.0)
            body.set_edgecolor("0.25")
            body.set_linewidth(1.1)

    for key in ("cmeans", "cmedians", "cmaxes", "cmins", "cbars"):
        if key in vp:
            vp[key].set_color("0.25")
            if key == "cmedians":
                vp[key].set_linewidth(median_linewidth)
            else:
                vp[key].set_linewidth(1.2)

    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels)
    ax.grid(True, axis="y")
    ax.set_axisbelow(True)

    if title:
        ax.set_title(title)
    if ylabel:
        ax.set_ylabel(ylabel)

    if yscale is not None:
        ax.set_yscale(yscale)

    if jitter_points:
        rng = np.random.default_rng(0)
        for i, vals in enumerate(clean, start=1):
            vals = vals[np.isfinite(vals)]
            if vals.size == 0:
                continue
            x = i + rng.uniform(-0.12, 0.12, size=vals.size)
            ax.scatter(x, vals, s=point_size, alpha=jitter_alpha, linewidths=0)

    if annotate_n:
        ymin, ymax = ax.get_ylim()
        ytext = ymin + 0.02 * (ymax - ymin)
        for i, vals in enumerate(clean, start=1):
            n = int(np.isfinite(vals).sum())
            ax.text(i, ytext, f"n={n}", ha="center", va="bottom", fontsize=9, alpha=0.8)

    return vp

In [ ]:
# pyident/estimators.py
from __future__ import annotations
from typing import Optional, Tuple, Union, Dict, Any
import numpy as np
import scipy.linalg
import pysindy
import torch


class NODE(torch.nn.Module):
    def __init__(self, n, m, bias: bool = False):
        super().__init__()
        self.fc = torch.nn.Linear(n + m, n, bias=bias)

    def forward(self, t, x_and_u):
        return self.fc(x_and_u)

def _ensure_channel_major(U: np.ndarray, expected_T: int) -> np.ndarray:
    """Return ``U`` with shape ``(m, T)``; accept time-major ``(T, m)`` inputs."""

    if U.ndim != 2:
        raise ValueError(f"U must be 2-D, got shape {U.shape}.")

    if U.shape[1] == expected_T:
        return U
    if U.shape[0] == expected_T:
        return U.T

    raise ValueError(
        f"Unable to align U of shape {U.shape} with expected T={expected_T}."
    )


def node_fit_old(Xtrain: np.ndarray,
             Xp: np.ndarray,
             Utrain: np.ndarray,
             dt: float,
             epochs: int = 100):
    n, T = Xtrain.shape
    #m = Utrain.shape[0]
    #device = 'cpu'
    U_cm = _ensure_channel_major(Utrain, T)
    m = U_cm.shape[0]
    device = "cpu"

    model = NODE(n, m).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

    Xtrain_torch = torch.tensor(Xtrain.T, dtype=torch.float32, device=device)
    Utrain_torch = torch.tensor(U_cm.T, dtype=torch.float32, device=device)
    Xp_torch = torch.tensor(Xp.T, dtype=torch.float32, device=device)

    for _ in range(epochs):
        optimizer.zero_grad()
        xu = torch.cat([Xtrain_torch, Utrain_torch], dim=1)
        pred = model.fc(xu)
        loss = torch.nn.functional.mse_loss(pred, Xp_torch)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        W = model.fc.weight.detach().cpu().numpy()

    A_node = W[:, :n]
    B_node = W[:, n:]
    return A_node, B_node


def sindy_fit_old(X, Xp, U, dt):
    n, T = X.shape
    U_cm = _ensure_channel_major(U, T)
    X_all = np.hstack([X[:, :1], Xp])
    Xdot = np.empty_like(X)
    if T >= 2:
        Xdot[:, :-1] = (X_all[:, 2:] - X_all[:, :-2]) / (2.0 * dt)
        Xdot[:, 0] = (X_all[:, 1] - X_all[:, 0]) / dt
        Xdot[:, -1] = (X_all[:, -1] - X_all[:, -2]) / dt
    else:
        Xdot[:, 0] = (X_all[:, 1] - X_all[:, -2]) / dt 

    Z = np.vstack([X, U_cm])
    Theta = Xdot @ np.linalg.pinv(Z, rcond=1e-12)
    return Theta[:, :n], Theta[:, n:]

def sindy_fit_dt(X, U, dt, degree=1):
    from pysindy import SINDy
    from pysindy.feature_library import PolynomialLibrary
    lib = PolynomialLibrary(degree=degree, include_bias=False)
    model = SINDy(discrete_time=True, feature_library=lib)
    U_cm = _ensure_channel_major(U, X.shape[1])
    model.fit(X.T, u=U_cm.T, t=dt)
    coef = model.coefficients()  # (n, n+m)
    n = X.shape[0]
    return coef[:, :n], coef[:, n:]

class LinearContinuousNODE(torch.nn.Module):
    """Continuous-time linear NODE with control treated as constant input."""

    def __init__(self, n: int, m: int):
        super().__init__()
        self.A = torch.nn.Parameter(torch.zeros(n, n))
        self.B = torch.nn.Parameter(torch.zeros(n, m))

    def forward(self, t: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
        n = self.A.shape[0]
        x = z[..., :n]
        u = z[..., n:]
        dxdt = x @ self.A.T + u @ self.B.T
        dudt = torch.zeros_like(u)
        return torch.cat([dxdt, dudt], dim=-1)


def node_fit(
    Xtrain: np.ndarray,
    Xp: np.ndarray,
    Utrain: np.ndarray,
    dt: float,
    epochs: int = 500,
    lr: float = 1e-2,
    verbose: bool = True,
    log_every: int = 10,
    return_history: bool = False,
    return_diagnostics: bool = False,
    # Modern ML monitoring parameters
    patience: int = 25,
    min_delta: float = 1e-6,  # Relaxed from 1e-8 per feedback
    convergence_tol: float = 1e-6,  # Relaxed from 1e-8 per feedback
    max_grad_norm: float = 10.0,
    early_stopping: bool = True,
    # Log file parameters
    log_file: str | None = None,
    log_append: bool = True,
    # Enhanced numerical parameters per feedback
    device: str | None = None,
    dtype: str | None = None,
    seed: int | None = None,
    weight_decay: float = 1e-6,
    use_scheduler: bool = True,
    warmstart_lstsq: bool = True,
    continuous_residual_weight: float = 1e-2,
    return_ct_params: bool = True,
):
    """Train a linear continuous-time NODE and return discretized dynamics.

    The model parameterizes continuous-time matrices :math:`(A_c, B_c)` and uses
    the matrix exponential of the block matrix to obtain discrete-time dynamics
    during training. This keeps the NODE interpretation intact while avoiding the
    heavy per-epoch ODE solves that made the original implementation slow.

    Parameters
    ----------
    Xtrain : np.ndarray
        Input state data, shape (n, T)
    Xp : np.ndarray
        Next state data, shape (n, T)
    Utrain : np.ndarray
        Input control data
    dt : float
        Time step
    epochs : int, default=300
        Maximum number of training epochs (increased from 200)
    lr : float, default=1e-2
        Learning rate for Adam optimizer
    verbose : bool, default=True
        Whether to print training progress
    log_every : int, default=10
        Frequency of progress logging
    return_history : bool, default=False
        Whether to return loss history (legacy parameter)
    return_diagnostics : bool, default=False
        Whether to return comprehensive training diagnostics
    patience : int, default=25
        Early stopping patience (epochs without improvement)
    min_delta : float, default=1e-6
        Minimum improvement threshold for early stopping
    convergence_tol : float, default=1e-5
        Loss threshold for convergence detection
    max_grad_norm : float, default=10.0
        Maximum gradient norm (for gradient clipping)
    early_stopping : bool, default=True
        Whether to use early stopping
    log_file : str, optional
        Path to log file for storing training progress. If None, no file logging.
    log_append : bool, default=True
        Whether to append to existing log file or overwrite
    device : str, optional
        Device to run on ('cpu', 'cuda'). If None, defaults to 'cpu'.
    dtype : str, optional
        Tensor data type ('float32', 'float64'). If None, defaults to 'float64'.
    seed : int, optional
        Random seed for reproducibility
    weight_decay : float, default=1e-6
        L2 regularization weight for Adam optimizer
    use_scheduler : bool, default=True
        Whether to use ReduceLROnPlateau scheduler
    warmstart_lstsq : bool, default=True
        Whether to initialize with discrete least-squares solution
    continuous_residual_weight : float, default=1e-2
        Weight for continuous residual loss term to improve small-dt conditioning
    return_ct_params : bool, default=True
        Whether to return continuous-time parameters in diagnostics

    Returns
    -------
    Ad : np.ndarray
        Discrete-time A matrix
    Bd : np.ndarray
        Discrete-time B matrix
    diagnostics : dict, optional
        Training diagnostics if return_diagnostics=True or return_history=True
    """
    import time

    # Set random seed for reproducibility
    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)

    n, T = Xtrain.shape
    U_cm = _ensure_channel_major(Utrain, T)
    m = U_cm.shape[0]

    # Device and dtype configuration (High-ROI improvement A)
    if device is None:
        device_obj = torch.device("cpu")
    else:
        device_obj = torch.device(device)

    if dtype is None:
        dtype_obj = torch.float64  # Default to float64 for better conditioning
    else:
        dtype_obj = getattr(torch, dtype)

    # Adaptive convergence tolerance based on data scale
    if convergence_tol == 1e-6:
        data_scale = float(np.mean(np.square(Xp)))
    else:
        data_scale = 1.0
    adaptive_convergence_tol = convergence_tol * max(data_scale, 1e-6)

    # Create tensors with consistent dtype and device (High-ROI improvement A)
    Xk = torch.tensor(Xtrain.T, dtype=dtype_obj, device=device_obj)  # (T, n)
    Uk = torch.tensor(U_cm.T, dtype=dtype_obj, device=device_obj)    # (T, m)
    Xnext = torch.tensor(Xp.T, dtype=dtype_obj, device=device_obj)   # (T, n)

    eye_n = torch.eye(n, dtype=dtype_obj, device=device_obj)

    # Initialize parameters with warmstart if requested (High-ROI improvement E)
    if warmstart_lstsq:
        # Discrete least-squares warm start: min||Z*Theta - X+||
        Z = torch.cat([Xk, Uk], dim=1)  # (T, n+m)
        try:
            Theta = torch.linalg.lstsq(Z, Xnext).solution  # (n+m, n)
            Ad0 = Theta[:n, :].T  # (n, n)
            Bd0 = Theta[n:, :].T  # (n, m)

            # Attempt continuous-time initialization via matrix log (High-ROI improvement F)
            try:
                I = torch.eye(n, dtype=dtype_obj, device=device_obj)
                # Safe matrix log with fallback for singular cases
                if torch.linalg.det(Ad0) > 1e-12:
                    A_ct_init = torch.matrix_log(Ad0) / dt
                    # Solve for B_ct from ZOH relation: (Ad - I) = A_ct * dt * Bd_dt/dt
                    # Bd = A_ct^-1 * (Ad - I) * B_ct, so B_ct = A_ct \ (Ad-I) \ Bd
                    if torch.linalg.det(A_ct_init) > 1e-12:
                        B_ct_init = torch.linalg.solve(A_ct_init, torch.linalg.solve(Ad0 - I, Bd0))
                    else:
                        # Fallback to scaled discrete init if A_ct is singular
                        A_ct_init = (Ad0 - I) / dt
                        B_ct_init = Bd0 / dt
                else:
                    # Fallback for singular Ad0
                    A_ct_init = torch.randn_like(Ad0) * 0.1
                    B_ct_init = torch.randn(n, m, dtype=dtype_obj, device=device_obj) * 0.1
            except Exception:
                # Fallback to simple discrete-time scaling 
                A_ct_init = (Ad0 - torch.eye(n, dtype=dtype_obj, device=device_obj)) / dt
                B_ct_init = Bd0 / dt
        except Exception:
            # Fallback to random initialization if lstsq fails
            A_ct_init = torch.randn((n, n), dtype=dtype_obj, device=device_obj) * 0.1
            B_ct_init = torch.randn((n, m), dtype=dtype_obj, device=device_obj) * 0.1
    else:
        # Random initialization
        A_ct_init = torch.randn((n, n), dtype=dtype_obj, device=device_obj) * 0.1
        B_ct_init = torch.randn((n, m), dtype=dtype_obj, device=device_obj) * 0.1

    A_ct = torch.nn.Parameter(A_ct_init)
    B_ct = torch.nn.Parameter(B_ct_init)
    params = [A_ct, B_ct]
    # Add weight decay for regularization (High-ROI improvement I)
    optimizer = torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)

    # Add learning rate scheduler (High-ROI improvement H)
    scheduler = None
    if use_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=10, 
            threshold=1e-6, min_lr=lr*1e-3
        )

    # Enhanced training monitoring
    start_time = time.time()
    history: list[float] = []
    grad_norms: list[float] = []
    best_loss = float('inf')
    patience_counter = 0
    converged = False
    early_stopped = False

    # Initialize file logging
    log_file_handle = None
    if log_file is not None:
        import pathlib
        log_path = pathlib.Path(log_file)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        mode = 'a' if log_append else 'w'
        log_file_handle = open(log_path, mode, encoding='utf-8')

        # Write header for new log file
        if not log_append or log_path.stat().st_size == 0:
            log_file_handle.write("# NODE Training Log\n")
            log_file_handle.write(f"# Started: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
            log_file_handle.write(f"# Parameters: n={n}, m={m}, T={T}, dt={dt:.3f}\n")
            log_file_handle.write(f"# Hyperparameters: epochs={epochs}, lr={lr}, patience={patience}, tol={convergence_tol:.1e}\n")
            log_file_handle.write("epoch,mse_loss,grad_norm,best_loss,patience_counter,status\n")
        else:
            log_file_handle.write(f"\n# New session: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")

    def _compute_discrete_matrices() -> tuple[torch.Tensor, torch.Tensor]:
        """Efficient computation without block matrix exponential (High-ROI improvement B)"""
        # Ad = exp(A_ct * dt)
        Ad = torch.matrix_exp(A_ct * dt)

        # Bd = A_ct^-1 * (Ad - I) * B_ct (ZOH formula)
        bd_rhs = (Ad - eye_n) @ B_ct
        try:
            Bd = torch.linalg.solve(A_ct, bd_rhs)
        except RuntimeError:
            # Fallback with Tikhonov regularization if A_ct is singular
            Bd = torch.linalg.solve(A_ct + 1e-10 * eye_n, bd_rhs)

        return Ad, Bd

    for epoch in range(int(epochs)):
        optimizer.zero_grad()

        # Efficient discrete matrix computation (High-ROI improvement B)
        Ad, Bd = _compute_discrete_matrices()

        # Primary discrete-time prediction loss
        preds = Xk @ Ad.T + Uk @ Bd.T
        discrete_loss = torch.nn.functional.mse_loss(preds, Xnext)

        # Continuous residual term for better small-dt conditioning (High-ROI improvement G)
        continuous_residual = (Xnext - Xk) / dt - (Xk @ A_ct.T + Uk @ B_ct.T)
        continuous_loss = continuous_residual.pow(2).mean()

        # Combined loss
        loss = discrete_loss + continuous_residual_weight * continuous_loss

        # NaN/Inf guard (High-ROI improvement D)
        if not torch.isfinite(loss):
            raise RuntimeError(f"Loss became non-finite at epoch {epoch}; check dtype/init/lr.")

        loss.backward()

        # Gradient monitoring and clipping
        grad_norm = torch.nn.utils.clip_grad_norm_(params, max_norm=max_grad_norm)
        grad_norms.append(float(grad_norm))

        optimizer.step()

        loss_value = float(loss.item())
        history.append(loss_value)

        # Update learning rate scheduler (High-ROI improvement H)
        if scheduler is not None:
            scheduler.step(loss_value)

        # Convergence detection with adaptive tolerance
        if loss_value < adaptive_convergence_tol:
            converged = True
            message = f"[NODE] ✓ Converged at epoch {epoch}: loss={loss_value:.6e} < tol={convergence_tol:.1e}"
            if verbose:
                print(message)
            if log_file_handle:
                log_file_handle.write(f"{epoch},{loss_value:.6e},{grad_norm:.6e},{best_loss:.6e},{patience_counter},CONVERGED\n")
                log_file_handle.write(f"# {message}\n")
                log_file_handle.flush()
            break

        # Early stopping logic
        if early_stopping:
            if loss_value < best_loss - min_delta:
                best_loss = loss_value
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= patience:
                early_stopped = True
                message = f"[NODE] ⏹ Early stopping at epoch {epoch}: no improvement for {patience} epochs"
                if verbose:
                    print(message)
                if log_file_handle:
                    log_file_handle.write(f"{epoch},{loss_value:.6e},{grad_norm:.6e},{best_loss:.6e},{patience_counter},EARLY_STOPPED\n")
                    log_file_handle.write(f"# {message}\n")
                    log_file_handle.flush()
                break

        # Gradient explosion detection
        if grad_norm > max_grad_norm:
            if verbose:
                print(f"[NODE] ⚠️ High gradient norm at epoch {epoch}: {grad_norm:.3e}")

        # Progress logging
        if epoch % log_every == 0 or epoch == epochs - 1:
            improvement_info = ""
            status = "TRAINING"
            if epoch > 0:
                if early_stopping:
                    improvement_info = f" | best={best_loss:.6e} | patience={patience_counter}/{patience}"

            if verbose:
                print(f"[NODE] epoch {epoch:4d}  mse={loss_value:.6e}  grad_norm={grad_norm:.3e}{improvement_info}")

            # Log to file every log_every epochs
            if log_file_handle:
                log_file_handle.write(f"{epoch},{loss_value:.6e},{grad_norm:.6e},{best_loss:.6e},{patience_counter},{status}\n")
                if epoch % (log_every * 5) == 0:  # Flush periodically
                    log_file_handle.flush()

    training_time = time.time() - start_time
    epochs_actual = len(history)

    # Final logging and cleanup
    if log_file_handle:
        log_file_handle.write(f"# Training completed: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        log_file_handle.write(f"# Final results: epochs={epochs_actual}, final_loss={history[-1] if history else 'N/A':.6e}, ")
        log_file_handle.write(f"best_loss={best_loss:.6e}, time={training_time:.2f}s\n")
        log_file_handle.write(f"# Status: {'CONVERGED' if converged else 'EARLY_STOPPED' if early_stopped else 'COMPLETED'}\n")
        log_file_handle.write("# " + "="*60 + "\n")
        log_file_handle.close()

    # Final parameter extraction (High-ROI improvement O)
    with torch.no_grad():
        Ad, Bd = _compute_discrete_matrices()
        Ad_np = Ad.cpu().numpy()
        Bd_np = Bd.cpu().numpy()

        # Extract continuous-time parameters
        A_ct_np = A_ct.detach().cpu().numpy()
        B_ct_np = B_ct.detach().cpu().numpy()

        # ZOH consistency check (High-ROI improvement O)
        I_np = np.eye(n)
        Ad_expected = scipy.linalg.expm(A_ct_np * dt)
        Bd_expected = np.linalg.solve(A_ct_np, (Ad_expected - I_np)) @ B_ct_np if np.linalg.det(A_ct_np) > 1e-12 else Bd_np

        zoh_consistency_Ad = np.linalg.norm(Ad_np - Ad_expected, 'fro')
        zoh_consistency_Bd = np.linalg.norm(Bd_np - Bd_expected, 'fro')
        zoh_consistency = zoh_consistency_Ad + zoh_consistency_Bd

    # Enhanced diagnostics with CT parameters and consistency checks
    if return_diagnostics or return_history:
        # Safe final loss extraction (High-ROI improvement P) 
        final_loss = float(history[-1]) if history else float("nan")

        diagnostics = {
            'loss_history': np.array(history),
            'grad_norm_history': np.array(grad_norms),
            'final_loss': final_loss,
            'best_loss': best_loss,
            'epochs_actual': epochs_actual,
            'epochs_requested': epochs,
            'converged': converged,
            'early_stopped': early_stopped,
            'training_time_s': training_time,
            'convergence_rate': -np.log(final_loss / history[0]) / epochs_actual if len(history) > 1 and history[0] > 0 and np.isfinite(final_loss) else 0.0,
            # ZOH consistency metrics (High-ROI improvement O)
            'zoh_consistency_fro': zoh_consistency,
            'zoh_consistency_Ad': zoh_consistency_Ad,  
            'zoh_consistency_Bd': zoh_consistency_Bd,
            'hyperparameters': {
                'lr': lr,
                'patience': patience,
                'min_delta': min_delta,
                'convergence_tol': convergence_tol,
                'adaptive_convergence_tol': adaptive_convergence_tol,
                'max_grad_norm': max_grad_norm,
                'early_stopping': early_stopping,
                'weight_decay': weight_decay,
                'continuous_residual_weight': continuous_residual_weight,
                'warmstart_lstsq': warmstart_lstsq,
                'use_scheduler': use_scheduler,
                'device': str(device_obj),
                'dtype': str(dtype_obj),
            }
        }

        # Add continuous-time parameters if requested (High-ROI improvement O)
        if return_ct_params:
            diagnostics['A_ct'] = A_ct_np
            diagnostics['B_ct'] = B_ct_np

        return Ad_np, Bd_np, diagnostics

    return Ad_np, Bd_np


def sindy_fit(
    X: np.ndarray,
    Xp: np.ndarray,
    U: np.ndarray,
    dt: float,
    degree: int = 1,
    optimizer: Optional[object] = None,
    feature_library: Optional[object] = None,
    verbose: bool = True,
    return_diagnostics: bool = False,
):
    """Fit a discrete-time SINDy model using PySINDy and log reconstruction error."""

    n, T = X.shape
    U_cm = _ensure_channel_major(U, T)
    # Ensure the samples provided to PySINDy line up with the control inputs.
    # Previously we stacked ``[x_0, Xp]`` and passed an ``(n, T+1)`` array to
    # ``model.fit`` while providing only ``T`` control samples.  Newer releases
    # of PySINDy keep trying to reconcile this mismatch internally, which makes
    # the optimizer spin forever without producing output.  Feeding the ``T``
    # state samples directly and supplying the next-step targets via ``x_dot``
    # keeps the data aligned and avoids the hang.
    X_samples = X.T      # shape (T, n)
    X_targets = Xp.T     # shape (T, n)

    if feature_library is None:
        feature_library = pysindy.feature_library.PolynomialLibrary(
            degree=degree,
            include_bias=False,
        )
    if optimizer is None:
        optimizer = pysindy.optimizers.STLSQ(
            threshold=1e-4,
            max_iter=10,
            normalize_columns=True,
        )

    model = pysindy.SINDy(
        discrete_time=True,
        feature_library=feature_library,
        optimizer=optimizer,
    )
    model.fit(X_samples, u=U_cm.T, x_dot=X_targets, t=dt)

    coef = model.coefficients()
    Ahat = coef[:, :n]
    Bhat = coef[:, n:]

    X_pred = model.predict(X_samples, u=U_cm.T)
    recon_mse = float(np.mean((X_pred - Xp.T) ** 2))
    sparsity = float(np.mean(np.abs(coef) > 0))

    if verbose:
        print(
            f"[SINDy] recon_mse={recon_mse:.6e}  coeff_density={sparsity:.3f}"
        )

    if return_diagnostics:
        diagnostics = {
            "reconstruction_mse": recon_mse,
            "coefficient_density": sparsity,
        }
        return Ahat, Bhat, diagnostics

    return Ahat, Bhat


# -----------------------------
# 1) DMDc (minimum-norm pinv)
# -----------------------------
def dmdc_pinv(
    X: np.ndarray,   # (n, T)
    Xp: np.ndarray,  # (n, T)
    U: np.ndarray,   # (m, T)
    rcond: float = 1e-10,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Minimum-norm DMDc via pseudoinverse.
    Solves Xp ≈ [A B] [X; U] with Θ = Xp Z^+ , Z=[X;U].
    Exact in noiseless data; returns a solution even if Z is rank-deficient.
    """
    n, T = X.shape
    U_cm = _ensure_channel_major(U, T)
    Z = np.vstack([X, U_cm])  
    Z_pinv = np.linalg.pinv(Z, rcond=rcond)
    AB = Xp @ Z_pinv                   # (n, n+m)
    Ahat = AB[:, :n]
    Bhat = AB[:, n:]
    return Ahat, Bhat

# Backward-compat shim used across the repo/tests
def dmdc_fit(X: np.ndarray, Xp: np.ndarray, U: np.ndarray, rcond: float = 1e-10) -> Tuple[np.ndarray, np.ndarray]:
    return dmdc_pinv(X, Xp, U, rcond=rcond)


# -----------------------------
# 2) DMDc (ridge / Tikhonov)
# -----------------------------
def dmdc_ridge(
    X: np.ndarray, Xp: np.ndarray, U: np.ndarray, lam: float = 1e-6
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Ridge-regularized DMDc:
      Θ = (Xp Z^T) (Z Z^T + λ I)^{-1},  Z=[X;U].
    More stable when Z is ill-conditioned or short.
    """
    n, T = X.shape
    U_cm = _ensure_channel_major(U, T)
    Z = np.vstack([X, U_cm])  
    ZZt = Z @ Z.T
    G = ZZt + lam * np.eye(ZZt.shape[0], dtype=ZZt.dtype)
    AB = (Xp @ Z.T) @ np.linalg.solve(G, np.eye(G.shape[0], dtype=G.dtype))
    Ahat = AB[:, :n]
    Bhat = AB[:, n:]
    return Ahat, Bhat


# -----------------------------
# 3) DMDc (truncated SVD)
# -----------------------------
def dmdc_tsvd(
    X: np.ndarray, Xp: np.ndarray, U: np.ndarray,
    rank: Optional[int] = None, svd_tol: Optional[float] = None
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Truncated-SVD DMDc:
      Z = U Σ V^T ; use top-r or tol-based truncation to form Z^+_r.
    Useful to restrict to the excited/identifiable subspace explicitly.
    """
    n, T = X.shape
    U_cm = _ensure_channel_major(U, T)
    Z = np.vstack([X, U_cm]) 
    Uz, Sz, Vtz = np.linalg.svd(Z, full_matrices=False)
    if rank is None:
        if svd_tol is not None:
            r = int(np.sum(Sz > svd_tol))
        else:
            r = int(np.sum(Sz > max(1e-12, Sz.max() * 1e-12)))
    else:
        r = int(min(rank, Sz.size))
    if r <= 0:
        # fall back to pinv if everything is tiny
        return dmdc_pinv(X, Xp, U, rcond=1e-10)
    Z_pinv_r = (Vtz[:r, :].T) @ np.diag(1.0 / Sz[:r]) @ (Uz[:, :r].T)
    AB = Xp @ Z_pinv_r
    Ahat = AB[:, :n]
    Bhat = AB[:, n:]
    return Ahat, Bhat


# ----------------------------------------------------
# 4) MOESP (full-state) + one-step B refit (simplified)
# ----------------------------------------------------
def moesp_fullstate(
    u_ts: np.ndarray,   # (T, m)
    x_ts: np.ndarray,   # (T, n) full state (C=I), same T as u
    n: int,
    i: Optional[int] = None,   # kept for signature parity; not used in simplified flow
    f: Optional[int] = None,   # kept for signature parity; not used in simplified flow
    rcond: float = 1e-10,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Full-state identification in the noiseless case can be done via one-step LS,
    which equals the MOESP A-estimate under ideal conditions. We compute A,B from:
        Xp = A X + B U
    where X = x_ts[:-1].T, U = u_ts[:-1].T, Xp = x_ts[1:].T.
    """
    X  = x_ts[:-1].T   # (n, T-1)
    Xp = x_ts[1:].T    # (n, T-1)
    U  = u_ts[:-1].T   # (m, T-1)
    # use minimum-norm solution; for conditioning, user can switch to ridge/tsvd
    return dmdc_pinv(X, Xp, U, rcond=rcond)


def _demean_channels(arr: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    if arr.ndim == 1:
        arr = arr[:, None]
    mean = arr.mean(axis=0, keepdims=True)
    return arr - mean, mean.ravel()


def _block_hankel(time_series: np.ndarray, s: int) -> np.ndarray:
    """Return the block Hankel matrix with ``s`` block rows.

    Parameters
    ----------
    time_series : array_like, shape (T, d)
        Time-major sequence.
    s : int
        Number of block rows.
    """
    if s <= 0:
        raise ValueError("block size s must be positive")
    T, d = time_series.shape
    L = T - s + 1
    if L <= 0:
        raise ValueError("time series too short for requested block size")
    blocks = [time_series[j:j+L, :].T for j in range(s)]
    return np.vstack(blocks)


def _solve_ridge_normal(
    A: np.ndarray,
    B: np.ndarray,
    ridge: float = 0.0,
) -> np.ndarray:
    """Solve min_X ||AX - B||_F^2 + ridge ||X||_F^2."""
    if ridge > 0.0:
        AtA = A.T @ A + ridge * np.eye(A.shape[1], dtype=A.dtype)
        AtB = A.T @ B
        return np.linalg.solve(AtA, AtB)
    sol, *_ = np.linalg.lstsq(A, B, rcond=None)
    return sol


def moesp_fit(
    u_ts: np.ndarray,
    y_ts: np.ndarray,
    s: int,
    n: Optional[int] = None,
    svd_tol: Optional[float] = None,
    projector_rcond: float = 1e-12,
    ridge: float = 0.0,
    ls_rcond: Optional[float] = None,
    return_states: bool = False,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, Dict[str, Any]]:
    """Faithful MOESP implementation following Van Overschee & De Moor.

    Parameters
    ----------
    u_ts : array_like, shape (T, m)
        Input sequence (time-major).
    y_ts : array_like, shape (T, ell)
        Output sequence (time-major).
    s : int
        Past horizon / number of block rows (must be >= model order).
    n : int, optional
        System order. If ``None``, it is inferred from the singular values
        using ``svd_tol``.
    svd_tol : float, optional
        Threshold for selecting the model order when ``n`` is ``None``. When
        omitted, a relative tolerance of ``1e-12`` is used.
    projector_rcond : float, default=1e-12
        Regularization for the pseudo-inverse inside the oblique projection.
    ridge : float, default=0.0
        Ridge parameter used when solving the shift-invariance equation for ``A``.
    ls_rcond : float, optional
        Cutoff for the least-squares solves when regressing ``B`` and ``D``.
    return_states : bool, default=False
        If ``True``, the recovered conditional state sequence is included in the
        ``info`` dictionary under the ``"state_sequence"`` key.
    """

    u_ts = np.asarray(u_ts, dtype=float)
    y_ts = np.asarray(y_ts, dtype=float)
    if u_ts.ndim == 1:
        u_ts = u_ts[:, None]
    if y_ts.ndim == 1:
        y_ts = y_ts[:, None]

    if u_ts.shape[0] != y_ts.shape[0]:
        raise ValueError("u_ts and y_ts must share the same time dimension")

    T = u_ts.shape[0]
    m = u_ts.shape[1]
    ell = y_ts.shape[1]
    if s < 1:
        raise ValueError("s must be a positive integer")
    if T < 2 * s:
        raise ValueError("time series too short relative to chosen horizon")

    u_dm, u_mean = _demean_channels(u_ts)
    y_dm, y_mean = _demean_channels(y_ts)

    L = T - 2 * s + 1
    if L <= 0:
        raise ValueError("insufficient samples for the requested horizon")

    Up = _block_hankel(u_dm[: T - s, :], s)
    Uf = _block_hankel(u_dm[s:, :], s)
    Yf = _block_hankel(y_dm[s:, :], s)

    # Left annihilator of U_f via null space of U_f
    null_basis = scipy.linalg.null_space(Uf, rcond=projector_rcond)
    if null_basis.size == 0:
        raise ValueError("input Hankel has full column rank; cannot form annihilator")
    Q2 = null_basis
    Lf = Q2.T
    rank_Uf = Uf.shape[1] - Q2.shape[1]

    Yf_t = Yf @ Q2
    Up_t = Up @ Q2
    # Orthogonal projection of Y_f onto row(U_p)
    G = Up_t @ Up_t.T
    G_pinv = np.linalg.pinv(G, rcond=projector_rcond)
    Ob = Yf_t @ Up_t.T @ G_pinv @ Up_t

    U_svd, S_svd, Vh_svd = np.linalg.svd(Ob, full_matrices=False)
    if n is None:
        if S_svd.size == 0:
            raise ValueError("no singular values available to infer the order")
        if svd_tol is None:
            svd_tol = max(1e-12, S_svd[0] * 1e-12)
        n = int(np.sum(S_svd > svd_tol))
    if n <= 0:
        raise ValueError("model order must be positive")
    if n > S_svd.size:
        raise ValueError("requested order exceeds the rank of the projection")

    Sigma_sqrt = np.sqrt(S_svd[:n])
    Gamma = U_svd[:, :n] * Sigma_sqrt[np.newaxis, :]
    Xfp = (Sigma_sqrt[:, np.newaxis] * Vh_svd[:n, :])

    # Extract C and A using shift-invariance
    C = Gamma[:ell, :]
    Gamma_up = Gamma[: (s - 1) * ell, :]
    Gamma_dn = Gamma[ell:, :]
    if Gamma_up.size == 0 or Gamma_dn.size == 0:
        raise ValueError("horizon s too small to extract system matrices")
    A = _solve_ridge_normal(Gamma_up, Gamma_dn, ridge=ridge)

    # Recover conditional states (aligned with columns of Xfp)
    X_cond = Xfp

    # Regress D first using the aligned subset of samples
    idx = np.arange(s, s + X_cond.shape[1])
    if idx[-1] >= T:
        idx = idx[idx < T]
        X_cond = X_cond[:, : idx.size]
    U_reg = u_dm[idx, :]
    Y_reg = y_dm[idx, :]

    # y_k ≈ C x_k + D u_k
    residual_y = (Y_reg - (X_cond.T @ C.T))
    D_t, *_ = np.linalg.lstsq(U_reg, residual_y, rcond=ls_rcond)
    D = D_t.T

    # x_{k+1} ≈ A x_k + B u_k
    if X_cond.shape[1] < 2:
        raise ValueError("not enough aligned states to regress B")
    Xk = X_cond[:, :-1]
    Xk1 = X_cond[:, 1:]
    U_state = u_dm[idx[:-1], :]
    rhs = (Xk1 - A @ Xk).T
    B_t, *_ = np.linalg.lstsq(U_state, rhs, rcond=ls_rcond)
    B = B_t.T

    info: Dict[str, Any] = {
        "s": s,
        "n": n,
        "singular_values": S_svd,
        "rank_Uf": rank_Uf,
        "u_mean": u_mean,
        "y_mean": y_mean,
        "Lf": Lf,
    }
    if return_states:
        info["state_sequence"] = X_cond

    return A, B, C, D, info

# Backward-compat wrapper: same call sites as old code/tests
def moesp_fit_old(
    X: np.ndarray,      # (n, T)
    Xp: np.ndarray,     # (n, T)
    U: np.ndarray,      # (m, T)
    s: Optional[int] = None,   # unused in simplified flow
    n: Optional[int] = None,
    rcond: float = 1e-10,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Wrapper to accept (X,Xp,U) blocks (as used in run_single/tests) and call
    moesp_fullstate with time-major sequences.
    """
    T = X.shape[1]
    U_cm = _ensure_channel_major(U, T)
    assert Xp.shape[1] == T, "time lengths must match"
    u_ts = U_cm.T 
    x_ts = X.T                       # (T, n)
    n_use = int(n if n is not None else X.shape[0])
    return moesp_fullstate(u_ts, x_ts, n=n_use, i=s, f=None, rcond=rcond)

# ----------------------------------------------------
# 4) Noise-sensitive DMDc variants (TLS, IV)
# ----------------------------------------------------

def dmdc_tls(X: np.ndarray, Xp: np.ndarray, U: np.ndarray,
             rcond: float = 1e-12) -> tuple[np.ndarray, np.ndarray]:
    """
    Row-wise scaled TLS for Xp ≈ Θ [X;U].
    Column-scale the augmented matrix to improve conditioning; fallback to OLS if needed.
    """
    n, T = X.shape
    U_cm = _ensure_channel_major(U, T)
    Z = np.vstack([X, U_cm]).T 
    Y = Xp.T                                   

    Theta = np.zeros((n, Z.shape[1]), dtype=X.dtype)
    for j in range(n):
        M = np.hstack([Z, Y[:, [j]]])          # (T, n+m+1)
        # column scaling
        scales = np.linalg.norm(M, axis=0)
        scales[scales == 0.0] = 1.0
        Ms = M / scales

        _, _, Vt = np.linalg.svd(Ms, full_matrices=False)
        v = Vt[-1, :] / scales                 # unscale

        denom = v[-1]
        if abs(denom) < 1e-12:
            # fallback: ridge-OLS on this row
            lam = 1e-6
            G = (Z.T @ Z) + lam*np.eye(Z.shape[1])
            theta_j = np.linalg.solve(G, Z.T @ Y[:, j])
        else:
            theta_j = -v[:-1] / denom
        Theta[j, :] = theta_j

    Ahat = Theta[:, :n]
    Bhat = Theta[:, n:]
    return Ahat, Bhat



def _lag_stack(U: np.ndarray, L: int) -> np.ndarray:
    # U: (m, T) → stack [u_{t-1}; …; u_{t-L}] for t=L..T-1 → (mL, T-L)
    m, T = U.shape
    return np.vstack([U[:, L-ell:T-ell] for ell in range(1, L+1)])

def dmdc_iv(X: np.ndarray, Xp: np.ndarray, U: np.ndarray,
            L: int = 1, instruments: Optional[np.ndarray] = None,
            rcond: float = 1e-10) -> tuple[np.ndarray, np.ndarray]:
    """
    2SLS IV-DMDc:
      1) Project regressors Z onto instrument space Π_Φ
      2) OLS of Y on Z_hat
    """
    n, T = X.shape
    U_cm = _ensure_channel_major(U, T)
    assert Xp.shape[1] == T and U_cm.shape[1] == T
    if L < 1 or L >= T:
        raise ValueError("L must be in [1, T-1].")

    Y = Xp[:, L:]                               # (n, T-L)
    Z = np.vstack([X[:, L-1:T-1], U_cm[:, L-1:T-1]])  # ((n+m), T-L)

    if instruments is None:
        Phi = _lag_stack(U_cm, L)                  # (mL, T-L)
    else:
        assert instruments.shape[1] == T, "instruments must have same time length as X"
        Phi = instruments[:, L:]                # (p, T-L)

    # Π_Φ in time domain
    G = Phi @ Phi.T                             # (p, p)
    Gp = np.linalg.pinv(G, rcond=rcond)         # robust if Phi low-rank
    Pi = Phi.T @ Gp @ Phi                       # (T-L, T-L)

    Zhat = Z @ Pi                               # project regressors onto instrument span

    ZZ = Zhat @ Zhat.T                          # ((n+m), (n+m))
    Theta = (Y @ Zhat.T) @ np.linalg.pinv(ZZ, rcond=rcond)

    return Theta[:, :n], Theta[:, n:]


# ----------------------------------------------------
# 5) Identifiable-component projector (utility)
# ----------------------------------------------------
def project_identifiable(theta: np.ndarray, Z: np.ndarray, rcond: float = 1e-10) -> np.ndarray:
    """
    Project parameter matrix Θ onto the identifiable component given regressors Z=[X;U].
    This right-multiplies by the projector onto col(Z):
        P = Z Z^+   (shape (n+m)x(n+m)),
    so Θ_ident = Θ P only alters directions that were not excited by the data.
    """
    P = Z @ np.linalg.pinv(Z, rcond=rcond)   # projector onto col(Z)
    return theta @ P

## Experiment Modules (Flattened)
These are the experiment scripts used by `iclrii.py`, now inlined.

In [ ]:
# """Empirical score distributions conditioned on system properties.
#
# ```
# # Single-axis sparsity sweep (legacy mode)
# python -m pyident.experiments.sim_regcomb --n 6 --m 2 --samples 200 --x0-samples 1000 \
#     --property density --cond-grid 0:0.05:1 --outdir results/sim3_density
#
# # Full sweep (One axis)
# # S1: Sparsity
# python -m pyident.experiments.sim_regcomb --axes "sparsity" \
#     --sparsity-grid 0.0:0.1:1.0 --samples 100 \
#     --x0-samples 100 --outdir results/sim3_sparse
#
# # S2: State dimension
# python -m pyident.experiments.sim_regcomb --axes "ndim" \
#     --ndim-grid 2:2:20 --samples 100 \
#     --x0-samples 100 --outdir results/sim3_state
#
# # S3: Underactuation
# python -m pyident.experiments.sim_regcomb --axes "underactuation" \
#      --samples 100 --x0-samples 100 --outdir results/sim3_underactuation
#
# # Full sweeps (Two axes)
# # E1: Sparsity vs. state dimension
# python -m pyident.experiments.sim_regcomb --axes "sparsity, ndim" \
#     --sparsity-grid 0.0:0.1:1.0 --ndim-grid 2:2:20 --samples 100 \
#     --x0-samples 100 --outdir results/sim3_sparse_state
#
# # E2: State dimension vs. underactuation
# python -m pyident.experiments.sim_regcomb --axes "ndim, underactuation" \
#     --ndim-grid 2:2:20 --samples 100 \
#     --x0-samples 100 --outdir results/sim3_state_underactuation
#
# # E3: Underaction vs. sparsity
# python -m pyident.experiments.sim_regcomb --axes "underactuation, sparsity" \
#         --sparsity-grid 0.0:0.1:1.0 --samples 100 \
#         --x0-samples 100 --outdir results/sim3_underactuation_sparsity
#
# # Smaller runs for faster testing
# # E1: Sparsity vs. state dimension
# python -m pyident.experiments.sim_regcomb --axes "sparsity, ndim" \
#     --sparsity-grid 0.0:0.2:1.0 --ndim-grid 2:4:20 --samples 10 \
#     --x0-samples 10 --outdir results/sim3_sparse_state
#
# # E2: State dimension vs. underactuation
# python -m pyident.experiments.sim_regcomb --axes "ndim, underactuation" \
#     --ndim-grid 2:4:20 --samples 10 \
#     --x0-samples 10 --outdir results/sim3_state_underactuation
#
# # E3: Underaction vs. sparsity
# python -m pyident.experiments.sim_regcomb --axes "underactuation, sparsity" \
#         --sparsity-grid 0.0:0.2:1.0 --samples 10 \
#         --x0-samples 10 --outdir results/sim3_underactuation_sparsity
# ```
# """
from __future__ import annotations

import argparse
import math
import pathlib
from collections import defaultdict
from dataclasses import dataclass
from typing import Iterable, Any, Mapping, Sequence, Tuple

try:  # pragma: no cover - import guard for optional dependency
    import matplotlib.pyplot as plt
except ModuleNotFoundError as exc:  # pragma: no cover - exercised only without matplotlib
    plt = None  # type: ignore[assignment]
    _MATPLOTLIB_IMPORT_ERROR = exc
else:
    _MATPLOTLIB_IMPORT_ERROR = None
import numpy as np
import pandas as pd



EPS = 1e-12
DEFAULT_SCORES: tuple[str, ...] = ("pbh", "krylov", "mu")

SCORE_DISPLAY_NAMES = {
    "pbh": "PBH score",
    "krylov": "Krylov score",
    "mu": "Left eigenvalue score",
}

AXIS_TITLE_NAMES = {
    "sparsity": "Sparsity",
    "ndim": "State Dimension",
    "underactuation": "Underactuation",
}

AXIS_ALIASES = {
    "sparsity": "sparsity",
    "sparse": "sparsity",
    "density": "sparsity",
    "ndim": "ndim",
    "xdim": "ndim",
    "state_dimension": "ndim",
    "underactuation": "underactuation",
    "undera": "underactuation",
}

AXIS_COLUMN = {
    "sparsity": "axis_sparsity",
    "ndim": "axis_ndim",
    "underactuation": "axis_underactuation",
}

AXIS_LABEL = {
    "sparsity": "Density level",
    "ndim": "State dimension n",
    "underactuation": "Input dimension m",
}

DEFAULT_STATE_INPUT_VALUES = tuple(range(2, 21, 2))

def format_axis_tick(axis_name: str, value: Any) -> str:
    """Format axis tick labels for heatmaps."""

    try:
        numeric = float(value)
    except (TypeError, ValueError):
        return str(value)

    if axis_name == "sparsity":
        return f"{numeric:.1f}"
    if axis_name in {"ndim", "underactuation"} and math.isclose(
        numeric, round(numeric), rel_tol=0.0, abs_tol=1e-9
    ):
        return f"{int(round(numeric))}"
    return f"{numeric:g}"


def make_heatmap_title(
    x_axis: str, y_axis: str, score_name: str, data: pd.DataFrame
) -> str:
    """Construct a descriptive heatmap title for score summaries."""

    score_label = SCORE_DISPLAY_NAMES.get(score_name, score_name)
    x_label = AXIS_TITLE_NAMES.get(x_axis, x_axis.title())
    y_label = AXIS_TITLE_NAMES.get(y_axis, y_axis.title())
    base_title = f"{x_label} vs. {y_label}"

    if (x_axis, y_axis) == ("underactuation", "sparsity") and "n" in data:
        n_values = sorted({int(val) for val in data["n"].dropna().unique()})
        if n_values:
            if len(n_values) == 1:
                suffix = f", n = {n_values[0]}"
            else:
                suffix = ", n ∈ {" + ", ".join(str(val) for val in n_values) + "}"
            return f"{base_title} ({score_label}{suffix})"

    return f"{base_title} ({score_label})"

def sample_unit_sphere(n: int, rng: np.random.Generator) -> np.ndarray:
    """Uniform sample on the unit sphere S^{n-1}."""

    v = rng.standard_normal(n)
    nrm = float(np.linalg.norm(v))
    return v / (nrm if nrm > 0.0 else 1.0)


def matrix_density(M: np.ndarray, tol: float = 0.0) -> float:
    """Fraction of entries whose magnitude exceeds ``tol``."""

    return float(np.mean(np.abs(M) > tol))

def compute_scores(
    A: np.ndarray,
    B: np.ndarray,
    x0: np.ndarray,
    score_names: Sequence[str],
) -> Mapping[str, float]:
    """Evaluate selected x0-based empirical scores."""

    if not score_names:
        return {}

    score_set = set(score_names)
    scores: dict[str, float] = {}

    if "pbh" in score_set:
        scores["pbh"] = float(pbh_margin_structured(A, B, x0))

    if "krylov" in score_set:
        K = unified_generator(A, B, x0, mode="unrestricted")
        svals = np.linalg.svd(K, compute_uv=False)
        smin = float(svals.min()) if svals.size else 0.0
        scores["krylov"] = max(smin, EPS)

    if "mu" in score_set:
        Xaug = np.concatenate([x0.reshape(-1, 1), B], axis=1)
        mu = left_eigvec_overlap(A, Xaug)
        mu_min = float(np.min(mu)) if mu.size else 0.0
        scores["mu"] = max(mu_min, EPS)

    return scores


class Moment:
    count: int = 0
    total: float = 0.0
    total_sq: float = 0.0

    def update(self, value: float) -> None:
        self.count += 1
        self.total += value
        self.total_sq += value * value

    @property
    def mean(self) -> float:
        return self.total / self.count if self.count else float("nan")

    @property
    def std(self) -> float:
        if self.count == 0:
            return float("nan")
        mu = self.mean
        var = max(self.total_sq / self.count - mu * mu, 0.0)
        return math.sqrt(var)


def parse_grid(spec: str) -> np.ndarray:
    spec = spec.strip()
    if not spec:
        raise ValueError("conditioning grid specification cannot be empty")
    if ":" in spec:
        parts = spec.split(":")
        if len(parts) != 3:
            raise ValueError("range grid must have the form start:step:stop")
        start, step, stop = map(float, parts)
        if step <= 0:
            raise ValueError("grid step must be positive")
        if stop < start:
            raise ValueError("grid stop must not be smaller than the start value")

        # ``np.arange`` can accumulate floating point error and produce values that
        # overshoot the requested ``stop`` bound (for example 0.1:0.2:1.0 yields
        # 1.1).  This is problematic when the grid is used to parameterise
        # probabilities such as sparsity densities.  Build the sequence manually
        # and clamp it to the stop value instead.
        values: list[float] = []
        current = start
        # Allow a small tolerance when comparing against ``stop`` so that values
        # very close to the end-point (e.g. 0.6 + 0.2 ≈ 0.7999999999999999) are
        # still included.
        tol = 1e-12 * max(1.0, abs(stop))
        while current <= stop + tol:
            values.append(float(current))
            current += step
        return np.asarray(values, dtype=float)

    tokens = spec.replace(",", " ").split()
    values = [float(tok) for tok in tokens if tok]
    if not values:
        raise ValueError("no values parsed from grid specification")
    return np.asarray(values, dtype=float)


def ensure_output_dir(path: pathlib.Path) -> pathlib.Path:
    path.mkdir(parents=True, exist_ok=True)
    (path / "plots").mkdir(exist_ok=True)
    return path

def parse_axes_spec(spec: str | None) -> tuple[str, ...]:
    if not spec:
        return ()
    tokens = [tok.strip().lower() for tok in spec.replace(",", " ").split() if tok.strip()]
    if not tokens:
        raise ValueError("--axes was provided but no axis names were found")
    axes: list[str] = []
    for tok in tokens:
        if tok not in AXIS_ALIASES:
            raise ValueError(f"unknown axis '{tok}'")
        canon = AXIS_ALIASES[tok]
        if canon in axes:
            raise ValueError(f"duplicate axis '{tok}' in specification")
        axes.append(canon)
    if len(axes) > 2:
        raise ValueError("at most two axes can be combined")
    return tuple(axes)


def underactuation_grid(n_dim: int) -> list[int]:
    if n_dim <= 0:
        raise ValueError("state dimension must be positive when using the underactuation axis")
    step = max(1, n_dim // 8)
    values = list(range(1, n_dim + 1, step))
    if values[-1] != n_dim:
        values.append(n_dim)
    return sorted(set(values))


def freeze_items(mapping: Mapping[str, Any]) -> Tuple[Tuple[str, Any], ...]:
    """Create a hashable representation of a dictionary."""

    return tuple(sorted(mapping.items()))


def generate_system(
    property_name: str,
    property_value: float,
    n: int,
    m: int,
    rng: np.random.Generator,
    *,
    sparse_which: str,
    sparse_tol: float,
    base_density: float,
    deficiency_base: str,
    deficiency_embed_random: bool,
) -> tuple[np.ndarray, np.ndarray, dict]:
    """Draw a system according to the chosen conditioning property."""

    if property_name == "density":
        A, B = sparse_continuous(
            n=n,
            m=m,
            rng=rng,
            which=sparse_which,
            p_density=float(property_value),
        )
        meta = {
            "target_density": float(property_value),
            "density_A": matrix_density(A, tol=sparse_tol),
            "density_B": matrix_density(B, tol=sparse_tol),
            "density_AB": matrix_density(np.hstack([A, B]), tol=sparse_tol),
        }
        return A, B, meta

    if property_name == "state_dimension":
        A, B = sparse_continuous(
            n=n,
            m=m,
            rng=rng,
            which=sparse_which,
            p_density=float(base_density),
        )
        meta = {
            "density_A": matrix_density(A, tol=sparse_tol),
            "density_B": matrix_density(B, tol=sparse_tol),
            "density_AB": matrix_density(np.hstack([A, B]), tol=sparse_tol),
        }
        return A, B, meta

    if property_name == "underactuation":
        A, B = sparse_continuous(
            n=n,
            m=m,
            rng=rng,
            which=sparse_which,
            p_density=float(base_density),
        )
        meta = {
            "density_A": matrix_density(A, tol=sparse_tol),
            "density_B": matrix_density(B, tol=sparse_tol),
            "density_AB": matrix_density(np.hstack([A, B]), tol=sparse_tol),
            "underactuation": float(n - m),
            "input_fraction": float(m) / float(n) if n else float("nan"),
        }
        return A, B, meta

    if property_name == "deficiency":
        deficiency = int(round(property_value))
        rank_target = max(n - deficiency, 0)
        A, B, meta = draw_with_ctrb_rank(
            n=n,
            m=m,
            r=rank_target,
            rng=rng,
            ensemble_type=deficiency_base,
            embed_random_basis=deficiency_embed_random,
        )
        rk, _ = controllability_rank(A, B)
        meta_info = {
            "target_deficiency": deficiency,
            "achieved_rank": int(rk),
            "achieved_deficiency": int(n - rk),
        }
        meta_info.update(meta)
        return A, B, meta_info

    raise ValueError(f"unsupported property '{property_name}'")


def build_axis_scenarios(
    args: argparse.Namespace, axes: Iterable[str]
) -> list[tuple[str, float, int, int, dict[str, Any]]]:
    axes = tuple(axes)
    results: list[tuple[str, float, int, int, dict[str, Any]]] = []
    base_n = int(args.n)
    base_m = int(args.m)

    def append(
        property_name: str,
        property_value: float,
        n_cur: int,
        m_cur: int,
        axis_values: Mapping[str, Any],
    ) -> None:
        info: dict[str, Any] = {
            "property": property_name,
            "property_value": float(property_value),
            "axes": ",".join(axes),
            "n": n_cur,
            "m": m_cur,
        }
        for axis_name, axis_value in axis_values.items():
            info[AXIS_COLUMN[axis_name]] = axis_value
        if "underactuation" in axes:
            info.setdefault("underactuation_level", float(n_cur - m_cur))
            info.setdefault(
                "input_fraction", float(m_cur) / float(n_cur) if n_cur else float("nan")
            )
        if "ndim" in axes:
            info.setdefault("state_dimension", int(n_cur))
        if "sparsity" in axes:
            info.setdefault("target_density", float(axis_values.get("sparsity", property_value)))
        results.append((property_name, float(property_value), n_cur, m_cur, info))

    axis_set = set(axes)
    if len(axes) == 1:
        axis = axes[0]
        if axis == "sparsity":
            grid_spec = args.sparsity_grid or args.cond_grid
            if grid_spec is None:
                raise ValueError("a sparsity grid must be provided for the sparsity axis")
            densities = parse_grid(grid_spec)
            for dens in densities:
                append(
                    "density",
                    float(dens),
                    base_n,
                    base_m,
                    {"sparsity": float(dens)},
                )
        elif axis == "ndim":
            grid_spec = args.ndim_grid or args.n_grid or args.cond_grid
            if grid_spec is None:
                raise ValueError("an n-dimension grid must be provided for the ndim axis")
            n_values = parse_grid(grid_spec)
            for n_val in n_values:
                n_cur = int(round(n_val))
                if n_cur <= 0:
                    raise ValueError("state dimensions must be positive integers")
                m_cur = min(base_m, n_cur)
                append("state_dimension", float(n_cur), n_cur, m_cur, {"ndim": n_cur})
        elif axis == "underactuation":
            m_values = underactuation_grid(base_n)
            for m_cur in m_values:
                append(
                    "underactuation",
                    float(base_n - m_cur),
                    base_n,
                    m_cur,
                    {"underactuation": m_cur},
                )
        else:
            raise ValueError(f"unsupported axis '{axis}'")
        return results

    if axis_set == {"sparsity", "ndim"}:
        dens_spec = args.sparsity_grid or args.cond_grid
        nd_spec = args.ndim_grid or args.n_grid
        if dens_spec is None:
            raise ValueError("a sparsity grid must be provided when combining axes")
        if nd_spec is None:
            raise ValueError("an n-dimension grid must be provided when combining axes")
        densities = parse_grid(dens_spec)
        n_values = parse_grid(nd_spec)
        for dens in densities:
            for n_val in n_values:
                n_cur = int(round(n_val))
                if n_cur <= 0:
                    raise ValueError("state dimensions must be positive integers")
                m_cur = min(base_m, n_cur)
                axis_vals = {"sparsity": float(dens), "ndim": n_cur}
                append("density", float(dens), n_cur, m_cur, axis_vals)
        return results

    if axis_set == {"sparsity", "underactuation"}:
        dens_spec = args.sparsity_grid or args.cond_grid
        if dens_spec is None:
            raise ValueError("a sparsity grid must be provided when combining axes")
        densities = parse_grid(dens_spec)
        m_values = underactuation_grid(base_n)
        for dens in densities:
            for m_cur in m_values:
                axis_vals = {"sparsity": float(dens), "underactuation": m_cur}
                append("density", float(dens), base_n, m_cur, axis_vals)
        return results

    if axis_set == {"ndim", "underactuation"}:
        nd_spec = args.ndim_grid or args.n_grid or args.cond_grid
        if nd_spec is None:
            n_values = np.array(DEFAULT_STATE_INPUT_VALUES, dtype=float)
        else:
            n_values = parse_grid(nd_spec)
        default_m_values = sorted(
            {int(val) for val in DEFAULT_STATE_INPUT_VALUES} | {int(base_m)}
        )
        for n_val in n_values:
            n_cur = int(round(n_val))
            if n_cur <= 0:
                raise ValueError("state dimensions must be positive integers")
            m_values = default_m_values
            for m_cur in m_values:
                axis_vals = {"ndim": n_cur, "underactuation": m_cur}
                append("underactuation", float(n_cur - m_cur), n_cur, m_cur, axis_vals)
        return results

    raise ValueError(f"unsupported axis combination: {axes}")



def run(args: argparse.Namespace) -> None:
    rng = np.random.default_rng(args.seed)
    outdir = ensure_output_dir(pathlib.Path(args.outdir))

    if args.scores is None:
        score_names = DEFAULT_SCORES
    else:
        if not args.scores:
            raise ValueError("--scores was provided but no score names were given")
        score_names = tuple(dict.fromkeys(args.scores))


    x0_cache: dict[int, list[np.ndarray]] = {}

    # Moment accumulators: {(score, property_value) -> Moment}
    def get_x0_samples(n_dim: int) -> list[np.ndarray]:
        samples = x0_cache.get(n_dim)
        if samples is None:
            samples = [sample_unit_sphere(n_dim, rng) for _ in range(args.x0_samples)]
            x0_cache[n_dim] = samples
        return samples

    # Moment accumulators keyed by (score, frozen_property_info)
    accum: dict[tuple[str, Tuple[Tuple[str, Any], ...]], tuple[Moment, list[float]]] = defaultdict(
        lambda: (Moment(), [])
    )
    system_records: list[dict] = []

    valid_scores = set(DEFAULT_SCORES)
    invalid = set(score_names) - valid_scores
    if invalid:
        raise ValueError(f"unknown scores requested: {sorted(invalid)}")
    axes = parse_axes_spec(args.axes)
    if axes:
        if args.property is not None and args.property != "density":
            # ``--property`` is kept for legacy workflows; avoid confusion when
            # the new multi-axis mode is used.
            raise ValueError("--property cannot be combined with --axes")
        scenarios = build_axis_scenarios(args, axes)
    else:
        property_name = args.property
        scenarios: list[tuple[str, float, int, int, dict[str, Any]]] = []
        if property_name == "underactuation":
            n_grid_spec = args.n_grid or args.cond_grid
            if n_grid_spec is None:
                raise ValueError("--cond-grid or --n-grid must be provided for property=underactuation")
            if args.m_grid is None:
                raise ValueError("--m-grid must be provided for property=underactuation")
            n_values = parse_grid(n_grid_spec)
            m_values = parse_grid(args.m_grid)
            for n_val in n_values:
                n_cur = int(round(n_val))
                if n_cur <= 0:
                    raise ValueError("state dimensions must be positive integers")
                for m_val in m_values:
                    m_cur = int(round(m_val))
                    if m_cur <= 0:
                        raise ValueError("input dimensions must be positive integers")
                    prop_val = float(n_cur - m_cur)
                    info = {
                        "property": property_name,
                        "property_value": prop_val,
                        "n": n_cur,
                        "m": args.m,
                        "input_fraction": float(args.m) / float(n_cur),
                    }
                scenarios.append((property_name, prop_val, n_cur, args.m, info))
        else:
            conditioning_values = parse_grid(args.cond_grid)
            for prop_value in conditioning_values:
                info = {
                    "property": property_name,
                    "property_value": float(prop_value),
                    "n": args.n,
                    "m": args.m,
                }
                scenarios.append((property_name, float(prop_value), args.n, args.m, info))

    if not scenarios:
        raise ValueError("no scenarios were generated; please check the grid specifications")

    for property_name, prop_value, n_cur, m_cur, prop_info in scenarios:
        prop_key = freeze_items(prop_info)
        x0_samples = get_x0_samples(n_cur)

        for _ in range(args.samples):
            A, B, meta = generate_system(
                property_name,
                prop_value,
                n_cur,
                m_cur,
                rng,
                sparse_which=args.sparse_which,
                sparse_tol=args.sparse_tol,
                base_density=args.sparse_density,
                deficiency_base=args.deficiency_base,
                deficiency_embed_random=not args.deficiency_no_embed,
            )

            sys_record = {
                "system_index": len(system_records),
                "n": n_cur,
                "m": m_cur,
            }
            for key, value in prop_info.items():
                if np.isscalar(value):
                    sys_record[key] = value

            # Ensure meta captures realised densities and actuation metrics.
            meta.setdefault("density_A", matrix_density(A, tol=args.sparse_tol))
            meta.setdefault("density_B", matrix_density(B, tol=args.sparse_tol))
            meta.setdefault("density_AB", matrix_density(np.hstack([A, B]), tol=args.sparse_tol))
            meta.setdefault("underactuation", float(n_cur - m_cur))
            meta.setdefault(
                "input_fraction", float(m_cur) / float(n_cur) if n_cur else float("nan")
            )
            for key, value in meta.items():
                if np.isscalar(value):
                    sys_record[f"meta_{key}"] = value

            system_records.append(sys_record)

            for x0 in x0_samples:
                scores = compute_scores(A, B, x0, score_names)
                for score_name, value in scores.items():
                    key = (score_name, prop_key)
                    mom, buf = accum[key]
                    mom.update(float(value))
                    buf.append(float(value))
                    accum[key] = (mom, buf)

    # Summaries -------------------------------------------------------------
    summary_rows = []
    for (score_name, prop_key), (moment, buf) in accum.items():
        q05 = np.nan
        q50 = np.nan
        q95 = np.nan
        if buf:
            arr = np.array(buf, dtype=float)
            q05, q50, q95 = np.quantile(arr, [0.05, 0.5, 0.95])
        prop_info = dict(prop_key)
        summary_rows.append(
            {
                "score": score_name,
                **prop_info,
                "count": moment.count,
                "mean": moment.mean,
                "std": moment.std,
                "q05": q05,
                "q50": q50,
                "q95": q95,
            }
        )

    summary_df = pd.DataFrame(summary_rows)
    if summary_df.empty:
        raise RuntimeError("no summary statistics were computed; check the score configuration")

    sort_cols = ["score"]
    for col in ["property_value", "n", "m", *AXIS_COLUMN.values()]:
        if col in summary_df.columns:
            sort_cols.append(col)
    summary_df = summary_df.sort_values(sort_cols)
    summary_path = outdir / "scores_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    systems_df = pd.DataFrame(system_records)
    systems_df.to_csv(outdir / "systems.csv", index=False)

    # Plots ----------------------------------------------------------------
    if plt is None:
        raise RuntimeError(
            "matplotlib is required for plotting; please install it to run this experiment"
        ) from _MATPLOTLIB_IMPORT_ERROR

    if axes:
        axis_columns = [AXIS_COLUMN[a] for a in axes]
        if len(axes) == 2:
            x_axis, y_axis = axis_columns[0], axis_columns[1]
            x_label = AXIS_LABEL[axes[0]]
            y_label = AXIS_LABEL[axes[1]]
            heat_threshold = float(getattr(args, "heatthr", 1e-12))
            for score_name in summary_df["score"].unique():
                sub = summary_df[summary_df["score"] == score_name]
                if sub.empty or x_axis not in sub.columns or y_axis not in sub.columns:
                    continue
                pivot = sub.pivot(index=y_axis, columns=x_axis, values="mean")
                pivot = pivot.sort_index().sort_index(axis=1)
                if pivot.empty:
                    continue

                data = pivot.to_numpy()
                n_rows, n_cols = data.shape
                fig, ax = plt.subplots(figsize=(7.2, 5.2))
                im = ax.imshow(
                    data,
                    origin="lower",
                    aspect="auto",
                    extent=(-0.5, n_cols - 0.5, -0.5, n_rows - 0.5),
                )
                ax.set_xlim(-0.5, n_cols - 0.5)
                ax.set_ylim(-0.5, n_rows - 0.5)
                ax.set_xticks(np.arange(n_cols))
                ax.set_yticks(np.arange(n_rows))
                ax.set_xticklabels(
                    [format_axis_tick(axes[0], value) for value in pivot.columns]
                )
                ax.set_yticklabels(
                    [format_axis_tick(axes[1], value) for value in pivot.index]
                )

                ax.set_xlabel(x_label)
                ax.set_ylabel(y_label)

                ax.set_title(make_heatmap_title(axes[0], axes[1], score_name, sub))
                special_state_under = axes[0] == "ndim" and axes[1] == "underactuation"
                if special_state_under and pivot.size:
                    column_values = list(pivot.columns)
                    row_values = list(pivot.index)
                    diag_coords: list[tuple[float, float]] = []
                    for col_idx, col_val in enumerate(column_values):
                        for row_idx, row_val in enumerate(row_values):
                            if math.isclose(float(col_val), float(row_val), rel_tol=0.0, abs_tol=1e-9):
                                diag_coords.append((float(col_idx), float(row_idx)))
                                break
                    if diag_coords:
                        xs, ys = zip(*diag_coords)
                        ax.plot(xs, ys, color="red", linewidth=2.0, solid_capstyle="round")
                        x_start, x_end = xs[0], xs[-1]
                        y_start, y_end = ys[0], ys[-1]
                        x_mid = 0.5 * (x_start + x_end)
                        y_mid = 0.5 * (y_start + y_end)
                        if len(xs) > 1:
                            rotation = math.degrees(math.atan2(y_end - y_start, x_end - x_start))
                        else:
                            rotation = 45.0
                        ax.text(
                            x_mid,
                            y_mid - 0.35,
                            "n = m",
                            color="red",
                            fontsize=8,
                            rotation=rotation,
                            rotation_mode="anchor",
                            ha="center",
                            va="center",
                        )
                cbar = fig.colorbar(im, ax=ax, pad=0.02)          
                cbar.set_label(f"{score_name} score (mean)")
                fig.tight_layout()
                fig.canvas.draw()
                if special_state_under and pivot.size:
                    #arrowprops = dict(
                    #    arrowstyle="-|>", color="red", linewidth=2.4, mutation_scale=12
                    #)
                    ax_pos = ax.get_position()
                    cbar_pos = cbar.ax.get_position()
                    horizontal_y = ax_pos.y0 - 0.07
                    horizontal_start = ax_pos.x0 + 0.02 * ax_pos.width
                    horizontal_end = ax_pos.x1 - 0.02 * ax_pos.width
                    ax.annotate(
                        "",
                        xy=(horizontal_end, horizontal_y),
                        xytext=(horizontal_start, horizontal_y),
                        xycoords=fig.transFigure,
                        textcoords=fig.transFigure,
                        #arrowprops=arrowprops,
                        annotation_clip=False,
                    )
                    vertical_x = cbar_pos.x1 + 0.02
                    vertical_top = min(cbar_pos.y1, 0.96)
                    vertical_bottom = max(cbar_pos.y0, 0.08)
                    ax.annotate(
                        "",
                        xy=(vertical_x, vertical_bottom),
                        xytext=(vertical_x, vertical_top),
                        xycoords=fig.transFigure,
                        textcoords=fig.transFigure,
                        #arrowprops=arrowprops,
                        annotation_clip=False,
                    )               
                name = f"{score_name}_heatmap_{axes[0]}_{axes[1]}".replace(",", "_")
                plot_path = outdir / "plots" / f"{name}.png"
                fig.savefig(plot_path, dpi=200)
                base_cmap = im.get_cmap()
                base_norm = im.norm               
                plt.close(fig)


                thr_fig, thr_ax = plt.subplots(figsize=(7.2, 5.2))
                thr_im = thr_ax.imshow(
                    data,
                    origin="lower",
                    aspect="auto",
                    extent=(-0.5, n_cols - 0.5, -0.5, n_rows - 0.5),
                    cmap=base_cmap,
                    norm=base_norm,
                )
                thr_ax.set_xlim(-0.5, n_cols - 0.5)
                thr_ax.set_ylim(-0.5, n_rows - 0.5)
                thr_ax.set_xticks(np.arange(n_cols))
                thr_ax.set_yticks(np.arange(n_rows))
                thr_ax.set_xticklabels(
                    [format_axis_tick(axes[0], value) for value in pivot.columns]
                )
                thr_ax.set_yticklabels(
                    [format_axis_tick(axes[1], value) for value in pivot.index]
                )

                thr_ax.set_xlabel(x_label)
                thr_ax.set_ylabel(y_label)

                threshold_title = make_heatmap_title(axes[0], axes[1], score_name, sub)
                thr_ax.set_title(
                    f"{threshold_title} (red < {heat_threshold:.1e})"
                )

                if special_state_under and pivot.size:
                    column_values = list(pivot.columns)
                    row_values = list(pivot.index)
                    diag_coords: list[tuple[float, float]] = []
                    for col_idx, col_val in enumerate(column_values):
                        for row_idx, row_val in enumerate(row_values):
                            if math.isclose(float(col_val), float(row_val), rel_tol=0.0, abs_tol=1e-9):
                                diag_coords.append((float(col_idx), float(row_idx)))
                                break
                    if diag_coords:
                        xs, ys = zip(*diag_coords)
                        thr_ax.plot(xs, ys, color="red", linewidth=2.0, solid_capstyle="round")
                        x_start, x_end = xs[0], xs[-1]
                        y_start, y_end = ys[0], ys[-1]
                        x_mid = 0.5 * (x_start + x_end)
                        y_mid = 0.5 * (y_start + y_end)
                        if len(xs) > 1:
                            rotation = math.degrees(math.atan2(y_end - y_start, x_end - x_start))
                        else:
                            rotation = 45.0
                        thr_ax.text(
                            x_mid,
                            y_mid - 0.35,
                            "n = m",
                            color="red",
                            fontsize=8,
                            rotation=rotation,
                            rotation_mode="anchor",
                            ha="center",
                            va="center",
                        )

                if not math.isnan(heat_threshold):
                    mask = data < heat_threshold
                else:
                    mask = np.zeros_like(data, dtype=bool)
                if np.any(mask):
                    red_overlay = np.zeros((n_rows, n_cols, 4), dtype=float)
                    red_overlay[mask] = (1.0, 0.0, 0.0, 1.0)
                    thr_ax.imshow(
                        red_overlay,
                        origin="lower",
                        aspect="auto",
                        extent=(-0.5, n_cols - 0.5, -0.5, n_rows - 0.5),
                    )

                thr_cbar = thr_fig.colorbar(thr_im, ax=thr_ax, pad=0.02)
                thr_cbar.set_label(
                    f"{score_name} score (mean; red < {heat_threshold:.1e})"
                )
                thr_fig.tight_layout()
                thr_name = f"{score_name}_heatmap_{axes[0]}_{axes[1]}_thr".replace(",", "_")
                thr_path = outdir / "plots" / f"{thr_name}.png"
                thr_fig.savefig(thr_path, dpi=200)
                plt.close(thr_fig)

                # Create a companion plot that visualizes descent directions with arrows.
                grad_y, grad_x = np.gradient(data) if data.size else (data, data)
                descent_x = -grad_x
                descent_y = -grad_y
                magnitudes = np.hypot(descent_x, descent_y)
                with np.errstate(divide="ignore", invalid="ignore"):
                    normalized_x = np.divide(
                        descent_x,
                        magnitudes,
                        out=np.zeros_like(descent_x),
                        where=magnitudes > 0.0,
                    )
                    normalized_y = np.divide(
                        descent_y,
                        magnitudes,
                        out=np.zeros_like(descent_y),
                        where=magnitudes > 0.0,
                    )

                arrow_length = 0.35
                arrow_dx = normalized_x * arrow_length
                arrow_dy = normalized_y * arrow_length

                arrow_fig, arrow_ax = plt.subplots(figsize=(7.2, 5.2))
                arrow_ax.set_xlim(-0.5, n_cols - 0.5)
                arrow_ax.set_ylim(-0.5, n_rows - 0.5)
                arrow_ax.set_facecolor("white")

                x_coords, y_coords = np.meshgrid(
                    np.arange(n_cols), np.arange(n_rows)
                )
                arrow_ax.quiver(
                    x_coords,
                    y_coords,
                    arrow_dx,
                    arrow_dy,
                    color="red",
                    angles="xy",
                    scale_units="xy",
                    scale=1.0,
                    width=0.01,
                    headwidth=4,
                    headlength=6,
                )
                arrow_ax.set_xticks(np.arange(n_cols))
                arrow_ax.set_yticks(np.arange(n_rows))
                arrow_ax.set_xticklabels(
                    [format_axis_tick(axes[0], value) for value in pivot.columns]
                )
                arrow_ax.set_yticklabels(
                    [format_axis_tick(axes[1], value) for value in pivot.index]
                )

                arrow_ax.set_xlabel(x_label)
                arrow_ax.set_ylabel(y_label)
                arrow_title = make_heatmap_title(axes[0], axes[1], score_name, sub)
                arrow_ax.set_title(f"{arrow_title} (descent directions)")

                if special_state_under and pivot.size:
                    column_values = list(pivot.columns)
                    row_values = list(pivot.index)
                    diag_coords: list[tuple[float, float]] = []
                    for col_idx, col_val in enumerate(column_values):
                        for row_idx, row_val in enumerate(row_values):
                            if math.isclose(float(col_val), float(row_val), rel_tol=0.0, abs_tol=1e-9):
                                diag_coords.append((float(col_idx), float(row_idx)))
                                break
                    if diag_coords:
                        xs, ys = zip(*diag_coords)
                        arrow_ax.plot(
                            xs, ys, color="red", linewidth=2.0, solid_capstyle="round"
                        )
                        x_start, x_end = xs[0], xs[-1]
                        y_start, y_end = ys[0], ys[-1]
                        x_mid = 0.5 * (x_start + x_end)
                        y_mid = 0.5 * (y_start + y_end)
                        if len(xs) > 1:
                            rotation = math.degrees(
                                math.atan2(y_end - y_start, x_end - x_start)
                            )
                        else:
                            rotation = 45.0
                        arrow_ax.text(
                            x_mid,
                            y_mid - 0.35,
                            "n = m",
                            color="red",
                            fontsize=8,
                            rotation=rotation,
                            rotation_mode="anchor",
                            ha="center",
                            va="center",
                        )

                arrow_fig.tight_layout()
                arrow_name = f"{score_name}_descent_{axes[0]}_{axes[1]}".replace(",", "_")
                arrow_path = outdir / "plots" / f"{arrow_name}.png"
                arrow_fig.savefig(arrow_path, dpi=200)
                plt.close(arrow_fig)

            return

        axis = axes[0]
        axis_col = axis_columns[0]
        axis_label = AXIS_LABEL[axis]

        for score_name in summary_df["score"].unique():
            sub = summary_df[summary_df["score"] == score_name]
            if sub.empty or axis_col not in sub.columns:
                continue

            sub = sub.sort_values(axis_col)
            x = sub[axis_col].to_numpy()
            mean = sub["mean"].to_numpy()
            std = sub["std"].to_numpy()

            plt.figure(figsize=(6.4, 4.0))
            plt.plot(x, mean, marker="o", label=f"{score_name} mean")
            plt.fill_between(x, mean - std, mean + std, alpha=0.2, label="±1 std")
            plt.xlabel(axis_label)
            plt.ylabel(f"{score_name} score")
            plt.title(f"{score_name} vs {axis}")
            plt.grid(True, alpha=0.3)
            plt.legend()
            plot_path = outdir / "plots" / f"{score_name}_vs_{axis}.png"

            plt.tight_layout()
            plt.savefig(plot_path, dpi=200)
            plt.close()
        return

    xlabel = {
        "density": "Density level",
        "deficiency": "Controllability deficiency",
        "state_dimension": "State dimension n",
    }.get(args.property, args.property)

    for score_name in summary_df["score"].unique():
        sub = summary_df[summary_df["score"] == score_name]
        if sub.empty:
            continue
        sub = sub.sort_values("property_value")

        x = sub["property_value"].to_numpy()
        mean = sub["mean"].to_numpy()
        std = sub["std"].to_numpy()

        plt.figure(figsize=(6.4, 4.0))
        plt.plot(x, mean, marker="o", label=f"{score_name} mean")
        plt.fill_between(x, mean - std, mean + std, alpha=0.2, label="±1 std")
        plt.xlabel(xlabel)
        plt.ylabel(f"{score_name} score")
        plt.title(f"{score_name} vs {args.property}")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plot_path = outdir / "plots" / f"{score_name}_vs_{args.property}.png"
        plt.tight_layout()
        plt.savefig(plot_path, dpi=200)
        plt.close()


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--n", type=int, default=10, help="state dimension")
    parser.add_argument("--m", type=int, default=2, help="input dimension")
    parser.add_argument(
        "--samples", type=int, default=200, help="number of (A,B) draws per conditioning value"
    )
    parser.add_argument(
        "--x0-samples", type=int, default=1000, help="number of x0 draws reused for every system"
    )
    parser.add_argument(
        "--axes",
        default=None,
        help="comma-separated list of axes to sweep (subset of: sparsity, ndim, underactuation)",
    )
    parser.add_argument(
        "--scores",
        nargs="*",
        choices=DEFAULT_SCORES,
        default=None,
        metavar="SCORE",
        help="score names to compute (subset of: %s)"
        % ", ".join(DEFAULT_SCORES),
    )
    parser.add_argument(
        "--property",
        default="density",
        choices=["density", "deficiency", "state_dimension", "underactuation"],
        help="system property to condition on",
    )
    parser.add_argument(
        "--cond-grid",
        default="0:0.05:1",
        help="conditioning grid specification (e.g., '0:0.05:1' or '0,0.5,1')",
    )
    parser.add_argument(
        "--n-grid",
        default=None,
        help="grid specification for state dimension sweeps (overrides --cond-grid for n)",
    )
    parser.add_argument(
        "--m-grid",
        default=None,
        help="grid specification for input dimension sweeps when property=underactuation",
    )
    parser.add_argument(
        "--sparsity-grid",
        default=None,
        help="grid specification for sparsity sweeps when using --axes",
    )
    parser.add_argument(
        "--ndim-grid",
        default=None,
        help="grid specification for state dimension sweeps when using --axes",
    )
    parser.add_argument("--seed", type=int, default=12345, help="base RNG seed")
    parser.add_argument("--outdir", default="results/sim3", help="output directory")

    # Sparse ensemble controls (for density property)
    parser.add_argument(
        "--sparse-which",
        default="both",
        choices=["A", "B", "both"],
        help="which matrices to sparsify when property=density",
    )
    parser.add_argument(
        "--sparse-tol",
        type=float,
        default=1e-12,
        help="tolerance when measuring realised density",
    )
    parser.add_argument(
        "--sparse-density",
        type=float,
        default=0.3,
        help="baseline density used when property is not 'density'",
    )

    parser.add_argument(
        "--heatthr",
        type=float,
        default=1e-12,
        help=(
            "heatmap threshold: cells with mean scores below this value are coloured red"
        ),
    )

    # Deficiency property controls
    parser.add_argument(
        "--deficiency-base",
        default="ginibre",
        choices=["ginibre", "stable", "binary", "sparse"],
        help="base ensemble for controllable/uncontrollable blocks",
    )
    parser.add_argument(
        "--deficiency-no-embed",
        action="store_true",
        help="disable random basis embedding in draw_with_ctrb_rank",
    )

    return parser


def main(argv: Sequence[str] | None = None) -> None:
    parser = build_parser()
    args = parser.parse_args(argv)
    run(args)


if __name__ == "__main__":
    main()

In [ ]:
# """Controllability fraction conditioned on system properties.
#
# This experiment mirrors ``sim_regcomb`` but replaces x0-based score
# statistics with the fraction of controllable pairs (A, B), i.e. the
# fraction of draws where rank(C_n(A,B)) = n.
#
# Example
# -------
# ```
# python -m pyident.experiments.sim_regcomb_ctrb --axes "sparsity, ndim" \
#     --sparsity-grid 0.0:0.1:1.0 --ndim-grid 2:2:20 --samples 100 \
#     --outdir results/sim3_sparse_state
# ```
# """
from __future__ import annotations

import argparse
import math
import pathlib
from collections import defaultdict
from typing import Any, Mapping, Sequence, Tuple

try:  # pragma: no cover - import guard for optional dependency
    import matplotlib.pyplot as plt
except ModuleNotFoundError as exc:  # pragma: no cover - exercised only without matplotlib
    plt = None  # type: ignore[assignment]
    _MATPLOTLIB_IMPORT_ERROR = exc
else:
    _MATPLOTLIB_IMPORT_ERROR = None
import numpy as np
import pandas as pd



SCORE_NAME = "controllable_fraction"
SCORE_DISPLAY_NAMES = {
    SCORE_NAME: "Controllable fraction",
}


def make_heatmap_title(x_axis: str, y_axis: str, data: pd.DataFrame) -> str:
    """Construct a descriptive heatmap title for controllability summaries."""

    x_label = AXIS_TITLE_NAMES.get(x_axis, x_axis.title())
    y_label = AXIS_TITLE_NAMES.get(y_axis, y_axis.title())
    base_title = f"{x_label} vs. {y_label}"

    if (x_axis, y_axis) == ("underactuation", "sparsity") and "n" in data:
        n_values = sorted({int(val) for val in data["n"].dropna().unique()})
        if n_values:
            if len(n_values) == 1:
                suffix = f", n = {n_values[0]}"
            else:
                suffix = ", n ∈ {" + ", ".join(str(val) for val in n_values) + "}"
            return f"{base_title} (controllable fraction{suffix})"

    return f"{base_title} (controllable fraction)"


def _legacy_scenarios(args: argparse.Namespace) -> list[tuple[str, float, int, int, dict[str, Any]]]:
    property_name = args.property
    scenarios: list[tuple[str, float, int, int, dict[str, Any]]] = []

    if property_name == "underactuation":
        n_grid_spec = args.n_grid or args.cond_grid
        if n_grid_spec is None:
            raise ValueError("--cond-grid or --n-grid must be provided for property=underactuation")
        if args.m_grid is None:
            raise ValueError("--m-grid must be provided for property=underactuation")
        n_values = parse_grid(n_grid_spec)
        m_values = parse_grid(args.m_grid)
        for n_val in n_values:
            n_cur = int(round(n_val))
            if n_cur <= 0:
                raise ValueError("state dimensions must be positive integers")
            for m_val in m_values:
                m_cur = int(round(m_val))
                if m_cur <= 0:
                    raise ValueError("input dimensions must be positive integers")
                prop_val = float(n_cur - m_cur)
                info = {
                    "property": property_name,
                    "property_value": prop_val,
                    "n": n_cur,
                    "m": m_cur,
                    "input_fraction": float(m_cur) / float(n_cur),
                }
                scenarios.append((property_name, prop_val, n_cur, m_cur, info))
        return scenarios

    conditioning_values = parse_grid(args.cond_grid)
    for prop_value in conditioning_values:
        info = {
            "property": property_name,
            "property_value": float(prop_value),
            "n": args.n,
            "m": args.m,
        }
        scenarios.append((property_name, float(prop_value), args.n, args.m, info))
    return scenarios


def run(args: argparse.Namespace) -> None:
    rng = np.random.default_rng(args.seed)
    outdir = ensure_output_dir(pathlib.Path(args.outdir))

    axes = parse_axes_spec(args.axes)
    if axes:
        if args.property is not None and args.property != "density":
            raise ValueError("--property cannot be combined with --axes")
        scenarios = build_axis_scenarios(args, axes)
    else:
        scenarios = _legacy_scenarios(args)

    if not scenarios:
        raise ValueError("no scenarios were generated; please check the grid specifications")

    accum: dict[tuple[str, Tuple[Tuple[str, Any], ...]], tuple[Moment, list[float]]] = defaultdict(
        lambda: (Moment(), [])
    )
    system_records: list[dict] = []
    A_list: list[np.ndarray] = []
    B_list: list[np.ndarray] = []

    for property_name, prop_value, n_cur, m_cur, prop_info in scenarios:
        prop_key = freeze_items(prop_info)

        for _ in range(args.samples):
            A, B, meta = generate_system(
                property_name,
                prop_value,
                n_cur,
                m_cur,
                rng,
                sparse_which=args.sparse_which,
                sparse_tol=args.sparse_tol,
                base_density=args.sparse_density,
                deficiency_base=args.deficiency_base,
                deficiency_embed_random=not args.deficiency_no_embed,
            )

            rk, _ = controllability_rank(A, B, order=n_cur, rtol=args.ctrb_rtol)
            controllable = float(rk == n_cur)

            sys_record = {
                "system_index": len(system_records),
                "n": n_cur,
                "m": m_cur,
                "controllability_rank": int(rk),
                "controllable": int(controllable),
            }
            for key, value in prop_info.items():
                if np.isscalar(value):
                    sys_record[key] = value

            meta.setdefault("density_A", matrix_density(A, tol=args.sparse_tol))
            meta.setdefault("density_B", matrix_density(B, tol=args.sparse_tol))
            meta.setdefault(
                "density_AB", matrix_density(np.hstack([A, B]), tol=args.sparse_tol)
            )
            meta.setdefault("underactuation", float(n_cur - m_cur))
            meta.setdefault(
                "input_fraction", float(m_cur) / float(n_cur) if n_cur else float("nan")
            )
            for key, value in meta.items():
                if np.isscalar(value):
                    sys_record[f"meta_{key}"] = value

            system_records.append(sys_record)
            A_list.append(A)
            B_list.append(B)

            key = (SCORE_NAME, prop_key)
            mom, buf = accum[key]
            mom.update(controllable)
            buf.append(controllable)
            accum[key] = (mom, buf)

    summary_rows = []
    for (score_name, prop_key), (moment, buf) in accum.items():
        q05 = np.nan
        q50 = np.nan
        q95 = np.nan
        if buf:
            arr = np.array(buf, dtype=float)
            q05, q50, q95 = np.quantile(arr, [0.05, 0.5, 0.95])
        prop_info = dict(prop_key)
        summary_rows.append(
            {
                "score": score_name,
                **prop_info,
                "count": moment.count,
                "mean": moment.mean,
                "std": moment.std,
                "q05": q05,
                "q50": q50,
                "q95": q95,
            }
        )

    summary_df = pd.DataFrame(summary_rows)
    if summary_df.empty:
        raise RuntimeError("no summary statistics were computed; check the configuration")

    sort_cols = ["score"]
    for col in ["property_value", "n", "m", *AXIS_COLUMN.values()]:
        if col in summary_df.columns:
            sort_cols.append(col)
    summary_df = summary_df.sort_values(sort_cols)
    summary_path = outdir / "scores_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    systems_df = pd.DataFrame(system_records)
    systems_df.to_csv(outdir / "systems.csv", index=False)
    if args.save_matrices:
        matrices_path = outdir / "systems_matrices.npz"
        np.savez_compressed(
            matrices_path,
            A=np.array(A_list, dtype=object),
            B=np.array(B_list, dtype=object),
            system_index=systems_df["system_index"].to_numpy(),
            n=systems_df["n"].to_numpy(),
            m=systems_df["m"].to_numpy(),
        )

    if plt is None:
        raise RuntimeError(
            "matplotlib is required for plotting; please install it to run this experiment"
        ) from _MATPLOTLIB_IMPORT_ERROR

    score_label = SCORE_DISPLAY_NAMES.get(SCORE_NAME, SCORE_NAME)
    heat_threshold = float(getattr(args, "heatthr", 1e-12))

    if axes:
        axis_columns = [AXIS_COLUMN[a] for a in axes]
        if len(axes) == 2:
            x_axis, y_axis = axis_columns[0], axis_columns[1]
            x_label = AXIS_LABEL[axes[0]]
            y_label = AXIS_LABEL[axes[1]]

            sub = summary_df[summary_df["score"] == SCORE_NAME]
            if not sub.empty and x_axis in sub.columns and y_axis in sub.columns:
                pivot = sub.pivot(index=y_axis, columns=x_axis, values="mean")
                pivot = pivot.sort_index().sort_index(axis=1)
                if not pivot.empty:
                    data = pivot.to_numpy()
                    n_rows, n_cols = data.shape

                    fig, ax = plt.subplots(figsize=(7.2, 5.2))
                    im = ax.imshow(
                        data,
                        origin="lower",
                        aspect="auto",
                        extent=(-0.5, n_cols - 0.5, -0.5, n_rows - 0.5),
                    )
                    ax.set_xlim(-0.5, n_cols - 0.5)
                    ax.set_ylim(-0.5, n_rows - 0.5)
                    ax.set_xticks(np.arange(n_cols))
                    ax.set_yticks(np.arange(n_rows))
                    ax.set_xticklabels(
                        [format_axis_tick(axes[0], value) for value in pivot.columns]
                    )
                    ax.set_yticklabels(
                        [format_axis_tick(axes[1], value) for value in pivot.index]
                    )

                    ax.set_xlabel(x_label)
                    ax.set_ylabel(y_label)
                    ax.set_title(make_heatmap_title(axes[0], axes[1], sub))

                    special_state_under = axes[0] == "ndim" and axes[1] == "underactuation"
                    if special_state_under and pivot.size:
                        column_values = list(pivot.columns)
                        row_values = list(pivot.index)
                        diag_coords: list[tuple[float, float]] = []
                        for col_idx, col_val in enumerate(column_values):
                            for row_idx, row_val in enumerate(row_values):
                                if math.isclose(
                                    float(col_val), float(row_val), rel_tol=0.0, abs_tol=1e-9
                                ):
                                    diag_coords.append((float(col_idx), float(row_idx)))
                                    break
                        if diag_coords:
                            xs, ys = zip(*diag_coords)
                            ax.plot(xs, ys, color="red", linewidth=2.0, solid_capstyle="round")
                            x_start, x_end = xs[0], xs[-1]
                            y_start, y_end = ys[0], ys[-1]
                            x_mid = 0.5 * (x_start + x_end)
                            y_mid = 0.5 * (y_start + y_end)
                            if len(xs) > 1:
                                rotation = math.degrees(
                                    math.atan2(y_end - y_start, x_end - x_start)
                                )
                            else:
                                rotation = 45.0
                            ax.text(
                                x_mid,
                                y_mid - 0.35,
                                "n = m",
                                color="red",
                                fontsize=8,
                                rotation=rotation,
                                rotation_mode="anchor",
                                ha="center",
                                va="center",
                            )

                    cbar = fig.colorbar(im, ax=ax, pad=0.02)
                    cbar.set_label(f"{score_label} (mean)")
                    fig.tight_layout()
                    plot_path = (
                        outdir
                        / "plots"
                        / f"{SCORE_NAME}_heatmap_{axes[0]}_{axes[1]}.png"
                    )
                    fig.savefig(plot_path, dpi=200)
                    base_cmap = im.get_cmap()
                    base_norm = im.norm
                    plt.close(fig)

                    thr_fig, thr_ax = plt.subplots(figsize=(7.2, 5.2))
                    thr_im = thr_ax.imshow(
                        data,
                        origin="lower",
                        aspect="auto",
                        extent=(-0.5, n_cols - 0.5, -0.5, n_rows - 0.5),
                        cmap=base_cmap,
                        norm=base_norm,
                    )
                    thr_ax.set_xlim(-0.5, n_cols - 0.5)
                    thr_ax.set_ylim(-0.5, n_rows - 0.5)
                    thr_ax.set_xticks(np.arange(n_cols))
                    thr_ax.set_yticks(np.arange(n_rows))
                    thr_ax.set_xticklabels(
                        [format_axis_tick(axes[0], value) for value in pivot.columns]
                    )
                    thr_ax.set_yticklabels(
                        [format_axis_tick(axes[1], value) for value in pivot.index]
                    )

                    thr_ax.set_xlabel(x_label)
                    thr_ax.set_ylabel(y_label)
                    thr_ax.set_title(
                        f"{make_heatmap_title(axes[0], axes[1], sub)} (red < {heat_threshold:.1e})"
                    )

                    if special_state_under and pivot.size:
                        column_values = list(pivot.columns)
                        row_values = list(pivot.index)
                        diag_coords = []
                        for col_idx, col_val in enumerate(column_values):
                            for row_idx, row_val in enumerate(row_values):
                                if math.isclose(
                                    float(col_val), float(row_val), rel_tol=0.0, abs_tol=1e-9
                                ):
                                    diag_coords.append((float(col_idx), float(row_idx)))
                                    break
                        if diag_coords:
                            xs, ys = zip(*diag_coords)
                            thr_ax.plot(xs, ys, color="red", linewidth=2.0, solid_capstyle="round")
                            x_start, x_end = xs[0], xs[-1]
                            y_start, y_end = ys[0], ys[-1]
                            x_mid = 0.5 * (x_start + x_end)
                            y_mid = 0.5 * (y_start + y_end)
                            if len(xs) > 1:
                                rotation = math.degrees(
                                    math.atan2(y_end - y_start, x_end - x_start)
                                )
                            else:
                                rotation = 45.0
                            thr_ax.text(
                                x_mid,
                                y_mid - 0.35,
                                "n = m",
                                color="red",
                                fontsize=8,
                                rotation=rotation,
                                rotation_mode="anchor",
                                ha="center",
                                va="center",
                            )

                    if not math.isnan(heat_threshold):
                        mask = data < heat_threshold
                    else:
                        mask = np.zeros_like(data, dtype=bool)
                    if np.any(mask):
                        red_overlay = np.zeros((n_rows, n_cols, 4), dtype=float)
                        red_overlay[mask] = (1.0, 0.0, 0.0, 1.0)
                        thr_ax.imshow(
                            red_overlay,
                            origin="lower",
                            aspect="auto",
                            extent=(-0.5, n_cols - 0.5, -0.5, n_rows - 0.5),
                        )

                    thr_cbar = thr_fig.colorbar(thr_im, ax=thr_ax, pad=0.02)
                    thr_cbar.set_label(
                        f"{score_label} (mean; red < {heat_threshold:.1e})"
                    )
                    thr_fig.tight_layout()
                    thr_path = (
                        outdir
                        / "plots"
                        / f"{SCORE_NAME}_heatmap_{axes[0]}_{axes[1]}_thr.png"
                    )
                    thr_fig.savefig(thr_path, dpi=200)
                    plt.close(thr_fig)

            return

        axis = axes[0]
        axis_col = axis_columns[0]
        axis_label = AXIS_LABEL[axis]

        sub = summary_df[summary_df["score"] == SCORE_NAME]
        if sub.empty or axis_col not in sub.columns:
            return

        sub = sub.sort_values(axis_col)
        x = sub[axis_col].to_numpy()
        mean = sub["mean"].to_numpy()
        std = sub["std"].to_numpy()

        plt.figure(figsize=(6.4, 4.0))
        plt.plot(x, mean, marker="o", label=f"{score_label} mean")
        plt.fill_between(x, mean - std, mean + std, alpha=0.2, label="±1 std")
        plt.xlabel(axis_label)
        plt.ylabel(score_label)
        plt.title(f"{score_label} vs {axis}")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plot_path = outdir / "plots" / f"{SCORE_NAME}_vs_{axis}.png"
        plt.tight_layout()
        plt.savefig(plot_path, dpi=200)
        plt.close()
        return

    xlabel = {
        "density": "Density level",
        "deficiency": "Controllability deficiency",
        "state_dimension": "State dimension n",
    }.get(args.property, args.property)

    sub = summary_df[summary_df["score"] == SCORE_NAME]
    if sub.empty:
        return

    sub = sub.sort_values("property_value")
    x = sub["property_value"].to_numpy()
    mean = sub["mean"].to_numpy()
    std = sub["std"].to_numpy()

    plt.figure(figsize=(6.4, 4.0))
    plt.plot(x, mean, marker="o", label=f"{score_label} mean")
    plt.fill_between(x, mean - std, mean + std, alpha=0.2, label="±1 std")
    plt.xlabel(xlabel)
    plt.ylabel(score_label)
    plt.title(f"{score_label} vs {args.property}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plot_path = outdir / "plots" / f"{SCORE_NAME}_vs_{args.property}.png"
    plt.tight_layout()
    plt.savefig(plot_path, dpi=200)
    plt.close()


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--n", type=int, default=10, help="state dimension")
    parser.add_argument(
        "--m",
        type=int,
        default=None,
        help="input dimension (defaults to n when omitted)",
    )
    parser.add_argument(
        "--samples", type=int, default=200, help="number of (A,B) draws per conditioning value"
    )
    parser.add_argument(
        "--axes",
        default=None,
        help="comma-separated list of axes to sweep (subset of: sparsity, ndim, underactuation)",
    )
    parser.add_argument(
        "--property",
        default="density",
        choices=["density", "deficiency", "state_dimension", "underactuation"],
        help="system property to condition on",
    )
    parser.add_argument(
        "--cond-grid",
        default="0:0.05:1",
        help="conditioning grid specification (e.g., '0:0.05:1' or '0,0.5,1')",
    )
    parser.add_argument(
        "--n-grid",
        default=None,
        help="grid specification for state dimension sweeps (overrides --cond-grid for n)",
    )
    parser.add_argument(
        "--m-grid",
        default=None,
        help="grid specification for input dimension sweeps when property=underactuation",
    )
    parser.add_argument(
        "--sparsity-grid",
        default=None,
        help="grid specification for sparsity sweeps when using --axes",
    )
    parser.add_argument(
        "--ndim-grid",
        default=None,
        help="grid specification for state dimension sweeps when using --axes",
    )
    parser.add_argument("--seed", type=int, default=12345, help="base RNG seed")
    parser.add_argument("--outdir", default="results/sim3", help="output directory")

    parser.add_argument(
        "--ctrb-rtol",
        type=float,
        default=None,
        help="relative tolerance for controllability rank (default: numpy SVD tolerance)",
    )

    parser.add_argument(
        "--sparse-which",
        default="both",
        choices=["A", "B", "both"],
        help="which matrices to sparsify when property=density",
    )
    parser.add_argument(
        "--sparse-tol",
        type=float,
        default=1e-12,
        help="tolerance when measuring realised density",
    )
    parser.add_argument(
        "--sparse-density",
        type=float,
        default=0.3,
        help="baseline density used when property is not 'density'",
    )

    parser.add_argument(
        "--heatthr",
        type=float,
        default=1e-12,
        help=(
            "heatmap threshold: cells with mean scores below this value are coloured red"
        ),
    )

    parser.add_argument(
        "--deficiency-base",
        default="ginibre",
        choices=["ginibre", "stable", "binary", "sparse"],
        help="base ensemble for controllable/uncontrollable blocks",
    )
    parser.add_argument(
        "--deficiency-no-embed",
        action="store_true",
        help="disable random basis embedding in draw_with_ctrb_rank",
    )
    parser.add_argument(
        "--save-matrices",
        action="store_true",
        help="save A,B draws to systems_matrices.npz for downstream filtering",
    )

    return parser


def main(argv: Sequence[str] | None = None) -> None:
    parser = build_parser()
    args = parser.parse_args(argv)
    if args.m is None:
        args.m = int(args.n)
    run(args)


if __name__ == "__main__":
    main()

sim_regcomb_ctrb_run = run
sim_regcomb_ctrb_build_parser = build_parser

In [ ]:
# """Box plots of identifiability scores for uncontrollable systems.
#
# Workflow
# --------
# 1) Draw (A,B) from the same ensembles as ``sim_regcomb`` *or* load a filtered dataset.
# 2) Keep only uncontrollable systems with density in [density-min, density-max] (generation mode).
# 3) Sample x0 via different modes (unit sphere or masked sphere).
# 4) Compute identifiability scores (PBH margin, left-eigenvector score).
# 5) Create box/violin plots per score showing distributions per x0 mode.
#
# Example
# -------
# python -m pyident.experiments.sim_unctrb_x0_boxplot --axes "sparsity, ndim" \
#     --sparsity-grid 0.0:0.1:1.0 --ndim-grid 2:2:20 --samples 100 \
#     --x0-samples 100 --outdir results/unctrb_x0
# """
from __future__ import annotations

import argparse
import math
import pathlib
from collections import defaultdict
from typing import Any, Iterable, Mapping, Sequence

try:  # pragma: no cover - import guard for optional dependency
    import matplotlib.pyplot as plt
except ModuleNotFoundError as exc:  # pragma: no cover - exercised only without matplotlib
    plt = None  # type: ignore[assignment]
    _MATPLOTLIB_IMPORT_ERROR = exc
else:
    _MATPLOTLIB_IMPORT_ERROR = None

import numpy as np
import pandas as pd
from matplotlib import colormaps



SCORES = ("pbh", "mu")


def sample_unit_sphere(n: int, rng: np.random.Generator) -> np.ndarray:
    v = rng.standard_normal(n)
    nrm = float(np.linalg.norm(v))
    return v / (nrm if nrm > 0.0 else 1.0)


def sample_masked_sphere(
    n: int,
    rng: np.random.Generator,
    p_keep: float,
    *,
    renorm: bool,
    min_norm: float = 0.0,
    min_support: int = 0,
    max_attempts: int = 128,
) -> np.ndarray:
    last_x0: np.ndarray | None = None
    min_support = int(max(0, min_support))
    max_attempts = int(max(1, max_attempts))

    for _ in range(max_attempts):
        x0 = sample_unit_sphere(n, rng)
        mask = rng.random(n) < p_keep
        if min_support and int(mask.sum()) < min_support:
            continue
        x0 = x0 * mask
        if renorm:
            nrm = float(np.linalg.norm(x0))
            if nrm > 0.0:
                x0 = x0 / nrm
        if float(np.linalg.norm(x0)) <= min_norm:
            last_x0 = x0
            continue
        return x0

    raise RuntimeError(
        f"Failed to draw masked x0 with min_norm={min_norm} and min_support={min_support} "
        f"after {max_attempts} attempts (last norm={float(np.linalg.norm(last_x0)) if last_x0 is not None else 0.0})."
    )


def x0_sampler(
    mode: str,
    p_keep: float,
    renorm: bool,
    *,
    min_norm: float = 0.0,
    min_support: int = 0,
    max_attempts: int = 128,
):
    if mode == "sphere":
        return lambda n, rng: sample_unit_sphere(n, rng)
    if mode == "mask":
        return lambda n, rng: sample_masked_sphere(
            n,
            rng,
            p_keep,
            renorm=renorm,
            min_norm=min_norm,
            min_support=min_support,
            max_attempts=max_attempts,
        )
    raise ValueError(f"unknown x0 sampling mode '{mode}'")


def format_mode_label(mode: str, p_keep: float) -> str:
    if mode == "sphere":
        return "sphere"
    return f"mask_p={p_keep:g}"

def filter_outliers(values: np.ndarray, method: str, k: float, trim_frac: float) -> np.ndarray:
    if values.size == 0 or method == "none":
        return values
    if method == "trim":
        frac = max(0.0, min(0.49, trim_frac))
        if frac <= 0.0:
            return values
        lo, hi = np.quantile(values, [frac, 1.0 - frac])
        return values[(values >= lo) & (values <= hi)]
    if method == "iqr":
        q1, q3 = np.quantile(values, [0.25, 0.75])
        iqr = q3 - q1
        if iqr == 0.0:
            return values
        lo = q1 - k * iqr
        hi = q3 + k * iqr
        return values[(values >= lo) & (values <= hi)]
    if method == "zscore":
        mu = float(np.mean(values))
        sd = float(np.std(values))
        if sd == 0.0:
            return values
        z = (values - mu) / sd
        return values[np.abs(z) <= k]
    if method == "mad":
        med = float(np.median(values))
        mad = float(np.median(np.abs(values - med)))
        if mad == 0.0:
            return values
        z = 0.6745 * (values - med) / mad
        return values[np.abs(z) <= k]
    raise ValueError(f"unknown outlier filter '{method}'")


def _legacy_scenarios(args: argparse.Namespace) -> list[tuple[str, float, int, int, dict[str, Any]]]:
    property_name = args.property
    scenarios: list[tuple[str, float, int, int, dict[str, Any]]] = []

    if property_name == "underactuation":
        n_grid_spec = args.n_grid or args.cond_grid
        if n_grid_spec is None:
            raise ValueError("--cond-grid or --n-grid must be provided for property=underactuation")
        if args.m_grid is None:
            raise ValueError("--m-grid must be provided for property=underactuation")
        n_values = parse_grid(n_grid_spec)
        m_values = parse_grid(args.m_grid)
        for n_val in n_values:
            n_cur = int(round(n_val))
            if n_cur <= 0:
                raise ValueError("state dimensions must be positive integers")
            for m_val in m_values:
                m_cur = int(round(m_val))
                if m_cur <= 0:
                    raise ValueError("input dimensions must be positive integers")
                prop_val = float(n_cur - m_cur)
                info = {
                    "property": property_name,
                    "property_value": prop_val,
                    "n": n_cur,
                    "m": m_cur,
                    "input_fraction": float(m_cur) / float(n_cur),
                }
                scenarios.append((property_name, prop_val, n_cur, m_cur, info))
        return scenarios

    conditioning_values = parse_grid(args.cond_grid)
    for prop_value in conditioning_values:
        info = {
            "property": property_name,
            "property_value": float(prop_value),
            "n": args.n,
            "m": args.m,
        }
        scenarios.append((property_name, float(prop_value), args.n, args.m, info))
    return scenarios


def _density_value(meta: Mapping[str, Any], source: str) -> float:
    if source == "A":
        return float(meta.get("density_A", np.nan))
    if source == "B":
        return float(meta.get("density_B", np.nan))
    return float(meta.get("density_AB", np.nan))


def _score_values(A: np.ndarray, B: np.ndarray, x0: np.ndarray) -> dict[str, float]:
    scores: dict[str, float] = {}
    scores["pbh"] = float(pbh_margin_structured(A, B, x0))
    Xaug = np.concatenate([x0.reshape(-1, 1), B], axis=1)
    mu_vals = left_eigvec_overlap(A, Xaug)
    scores["mu"] = float(np.min(mu_vals)) if mu_vals.size else 0.0
    return scores


def _scenario_label(info: Mapping[str, Any], axes: Sequence[str]) -> str:
    if not axes:
        return "all"
    parts = []
    for axis in axes:
        col = AXIS_COLUMN[axis]
        val = info.get(col, info.get(axis, None))
        parts.append(f"{axis}={val}")
    return ",".join(parts) if parts else "all"


def run(args: argparse.Namespace) -> None:
    rng = np.random.default_rng(args.seed)
    outdir = ensure_output_dir(pathlib.Path(args.outdir))

    if args.dataset_csv and not args.dataset_npz:
        raise ValueError("--dataset-npz must be provided with --dataset-csv")

    mode_configs: list[tuple[str, float]] = []
    p_list = [float(p) for p in args.mask_ps]
    if not args.exclude_sphere and 1.0 not in p_list:
        p_list.append(1.0)
    p_list = sorted(set(p_list))
    for p in p_list:
        if math.isclose(p, 1.0, rel_tol=0.0, abs_tol=1e-12):
            mode_configs.append(("sphere", 1.0))
        else:
            mode_configs.append(("mask", p))

    records: list[dict[str, Any]] = []

    if args.dataset_csv:
        systems_df = pd.read_csv(args.dataset_csv)
        matrices = np.load(args.dataset_npz, allow_pickle=True)
        A_list = matrices["A"]
        B_list = matrices["B"]
        sys_idx = matrices["system_index"]
        index_map = {int(idx): pos for pos, idx in enumerate(sys_idx)}
        density_col = {
            "A": "meta_density_A",
            "B": "meta_density_B",
            "AB": "meta_density_AB",
        }.get(args.density_source, "meta_density_AB")

        for _, row in systems_df.iterrows():
            sys_id = int(row["system_index"])
            pos = index_map.get(sys_id)
            if pos is None:
                raise RuntimeError(f"system_index {sys_id} missing from matrices file")
            A = A_list[pos]
            B = B_list[pos]
            n_cur = int(row["n"])
            m_cur = int(row["m"])
            dval = float(row.get(density_col, np.nan))
            rk = int(row.get("controllability_rank", -1))

            for mode, p_keep in mode_configs:
                sampler = x0_sampler(
                    mode,
                    p_keep,
                    renorm=args.mask_renorm,
                    min_norm=args.x0_min_norm,
                    min_support=args.x0_min_support,
                    max_attempts=args.x0_max_attempts,
                )
                for _ in range(args.x0_samples):
                    x0 = sampler(n_cur, rng)
                    scores = _score_values(A, B, x0)
                    for score_name, value in scores.items():
                        rec = {
                            "score": score_name,
                            "value": float(value),
                            "mode": format_mode_label(mode, p_keep),
                            "p_keep": p_keep,
                            "n": n_cur,
                            "m": m_cur,
                            "density": dval,
                            "controllability_rank": rk,
                        }
                        for col, val in row.items():
                            if np.isscalar(val):
                                rec[col] = val
                        records.append(rec)
    else:
        axes = parse_axes_spec(args.axes)
        if axes:
            if args.property is not None and args.property != "density":
                raise ValueError("--property cannot be combined with --axes")
            scenarios = build_axis_scenarios(args, axes)
        else:
            scenarios = _legacy_scenarios(args)

        if not scenarios:
            raise ValueError("no scenarios were generated; please check the grid specifications")

        density_min = float(args.density_min)
        density_max = float(args.density_max)
        if density_min > density_max:
            raise ValueError("density-min must be <= density-max")

        for property_name, prop_value, n_cur, m_cur, prop_info in scenarios:
            accepted = 0
            draws = 0
            max_draws = int(args.max_draws)
            while accepted < args.samples and draws < max_draws:
                draws += 1
                A, B, meta = generate_system(
                    property_name,
                    prop_value,
                    n_cur,
                    m_cur,
                    rng,
                    sparse_which=args.sparse_which,
                    sparse_tol=args.sparse_tol,
                    base_density=args.sparse_density,
                    deficiency_base=args.deficiency_base,
                    deficiency_embed_random=not args.deficiency_no_embed,
                )

                rk, _ = controllability_rank(A, B, order=n_cur, rtol=args.ctrb_rtol)
                if rk >= n_cur:
                    continue

                meta.setdefault("density_A", matrix_density(A, tol=args.sparse_tol))
                meta.setdefault("density_B", matrix_density(B, tol=args.sparse_tol))
                meta.setdefault(
                    "density_AB", matrix_density(np.hstack([A, B]), tol=args.sparse_tol)
                )
                dval = _density_value(meta, args.density_source)
                if not (density_min <= dval <= density_max):
                    continue

                accepted += 1

                for mode, p_keep in mode_configs:
                    sampler = x0_sampler(
                        mode,
                        p_keep,
                        renorm=args.mask_renorm,
                        min_norm=args.x0_min_norm,
                        min_support=args.x0_min_support,
                        max_attempts=args.x0_max_attempts,
                    )
                    for _ in range(args.x0_samples):
                        x0 = sampler(n_cur, rng)
                        scores = _score_values(A, B, x0)
                        for score_name, value in scores.items():
                            rec = {
                                "score": score_name,
                                "value": float(value),
                                "mode": format_mode_label(mode, p_keep),
                                "p_keep": p_keep,
                                "n": n_cur,
                                "m": m_cur,
                                "density": dval,
                                "controllability_rank": int(rk),
                            }
                            for key, val in prop_info.items():
                                if np.isscalar(val):
                                    rec[key] = val
                            records.append(rec)

            if accepted < args.samples:
                raise RuntimeError(
                    f"only accepted {accepted} systems (target {args.samples}) for scenario {prop_info}"
                )

    scores_df = pd.DataFrame(records)
    scores_path = outdir / "identifiability_scores.csv"
    scores_df.to_csv(scores_path, index=False)

    if plt is None:
        raise RuntimeError(
            "matplotlib is required for plotting; please install it to run this experiment"
        ) from _MATPLOTLIB_IMPORT_ERROR

    set_default_mpl_style()
    yscale = None if args.yscale == "none" else args.yscale
    axes = parse_axes_spec(args.axes)
    if axes:
        scenario_groups: list[tuple[str, pd.DataFrame]] = []
        group_cols = [AXIS_COLUMN[a] for a in axes]
        missing = [col for col in group_cols if col not in scores_df.columns]
        if missing:
            raise ValueError(f"axis columns missing from dataset: {missing}")
        for group_vals, group_df in scores_df.groupby(group_cols, dropna=False):
            if not isinstance(group_vals, tuple):
                group_vals = (group_vals,)
            info = {col: val for col, val in zip(group_cols, group_vals)}
            label = _scenario_label(info, axes)
            scenario_groups.append((label, group_df))
    else:
        scenario_groups = [("all", scores_df)]

    for scen_label, scen_df in scenario_groups:
        for score_name in SCORES:
            sub = scen_df[scen_df["score"] == score_name]
            if sub.empty:
                continue
            p_vals = sorted({float(p) for p in sub["p_keep"]})
            order = []
            if 1.0 in p_vals:
                order.append(format_mode_label("sphere", 1.0))
            order += [format_mode_label("mask", p) for p in p_vals if p < 1.0]
            data = []
            for mode in order:
                vals = sub[sub["mode"] == mode]["value"].to_numpy()
                vals = filter_outliers(vals, args.outlier_filter, args.outlier_k, args.outlier_trim)
                data.append(vals)

            total_count = int(sum(np.isfinite(arr).sum() for arr in data))
            n_tag = f"_n{total_count}"

            colors = None
            colorize = False
            if args.viridis:
                medians = []
                for vals in data:
                    finite = vals[np.isfinite(vals)]
                    medians.append(float(np.median(finite)) if finite.size else float("nan"))
                med_arr = np.array(medians, dtype=float)
                if np.any(np.isfinite(med_arr)):
                    vmin = float(np.nanmin(med_arr))
                    vmax = float(np.nanmax(med_arr))
                else:
                    vmin = 0.0
                    vmax = 1.0
                if vmax <= vmin:
                    vmax = vmin + 1.0
                normed = (med_arr - vmin) / (vmax - vmin)
                cmap = colormaps.get_cmap("viridis")
                colors = [cmap(float(x)) if np.isfinite(x) else cmap(0.0) for x in normed]
                colorize = True

            fig, ax = plt.subplots(figsize=(7.2, 4.6))
            title = f"{score_name} score distribution"
            if scen_label != "all":
                title = f"{title} ({scen_label})"
            if args.violin:
                nice_violinplot(
                    ax,
                    data,
                    order,
                    title=title,
                    ylabel=score_name,
                    yscale=yscale,
                    colorize=colorize,
                    colors=colors,
                    jitter_points=False,
                    annotate_n=False,
                    median_linewidth=3.0,
                )
            else:
                nice_boxplot(
                    ax,
                    data,
                    order,
                    title=title,
                    ylabel=score_name,
                    yscale=yscale,
                    colorize=colorize,
                    colors=colors,
                    jitter_points=False,
                    annotate_n=False,
                    median_linewidth=3.0,
                )
            fig.tight_layout()
            safe_label = scen_label.replace(" ", "_").replace(",", "_")
            plot_kind = "violin" if args.violin else "boxplot"
            plot_name = f"{score_name}_{plot_kind}_{safe_label}{n_tag}.png"
            fig.savefig(outdir / "plots" / plot_name, bbox_inches="tight")
            plt.close(fig)


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--dataset-csv",
        default=None,
        help="filtered systems CSV (from filter_unctrb_dataset.py)",
    )
    parser.add_argument(
        "--dataset-npz",
        default=None,
        help="filtered systems matrices NPZ (from filter_unctrb_dataset.py)",
    )
    parser.add_argument("--n", type=int, default=10, help="state dimension")
    parser.add_argument("--m", type=int, default=2, help="input dimension")
    parser.add_argument(
        "--samples", type=int, default=50, help="accepted uncontrollable systems per scenario"
    )
    parser.add_argument(
        "--x0-samples", type=int, default=100, help="number of x0 draws per mode per system"
    )
    parser.add_argument(
        "--axes",
        default=None,
        help="comma-separated list of axes to sweep (subset of: sparsity, ndim, underactuation)",
    )
    parser.add_argument(
        "--property",
        default="density",
        choices=["density", "deficiency", "state_dimension", "underactuation"],
        help="system property to condition on",
    )
    parser.add_argument(
        "--cond-grid",
        default="0:0.05:1",
        help="conditioning grid specification (e.g., '0:0.05:1' or '0,0.5,1')",
    )
    parser.add_argument(
        "--n-grid",
        default=None,
        help="grid specification for state dimension sweeps (overrides --cond-grid for n)",
    )
    parser.add_argument(
        "--m-grid",
        default=None,
        help="grid specification for input dimension sweeps when property=underactuation",
    )
    parser.add_argument(
        "--sparsity-grid",
        default=None,
        help="grid specification for sparsity sweeps when using --axes",
    )
    parser.add_argument(
        "--ndim-grid",
        default=None,
        help="grid specification for state dimension sweeps when using --axes",
    )
    parser.add_argument("--seed", type=int, default=12345, help="base RNG seed")
    parser.add_argument("--outdir", default="results/unctrb_x0", help="output directory")

    parser.add_argument(
        "--ctrb-rtol",
        type=float,
        default=None,
        help="relative tolerance for controllability rank (default: numpy SVD tolerance)",
    )

    parser.add_argument(
        "--density-min",
        type=float,
        default=0.3,
        help="minimum density for filtering uncontrollable systems",
    )
    parser.add_argument(
        "--density-max",
        type=float,
        default=0.7,
        help="maximum density for filtering uncontrollable systems",
    )
    parser.add_argument(
        "--density-source",
        choices=["A", "B", "AB"],
        default="AB",
        help="which density to filter on: A, B, or AB",
    )
    parser.add_argument(
        "--max-draws",
        type=int,
        default=20000,
        help="maximum raw draws per scenario to reach the accepted sample count",
    )

    parser.add_argument(
        "--mask-ps",
        nargs="*",
        default=[0.25, 0.5, 0.75, 1.0],
        help="Bernoulli keep probabilities for masked x0 sampling",
    )
    parser.add_argument(
        "--exclude-sphere",
        action="store_true",
        help="do not automatically include sphere (p=1) when using --mask-ps",
    )
    parser.add_argument(
        "--mask-renorm",
        action="store_true",
        help="renormalize masked x0 to unit length when nonzero",
    )
    parser.add_argument(
        "--x0-min-norm",
        type=float,
        default=0.0,
        help="minimum ||x0|| to accept for masked sampling (default: 0.0)",
    )
    parser.add_argument(
        "--x0-min-support",
        type=int,
        default=0,
        help="minimum number of nonzero entries in masked x0 (default: 0)",
    )
    parser.add_argument(
        "--x0-max-attempts",
        type=int,
        default=128,
        help="max attempts to draw a valid masked x0 (default: 128)",
    )
    parser.add_argument(
        "--violin",
        action="store_true",
        help="use violin plots instead of boxplots",
    )
    parser.add_argument(
        "--viridis",
        action="store_true",
        help="color boxes/violins by median using the viridis colormap",
    )
    parser.add_argument(
        "--yscale",
        choices=["none", "log"],
        default="none",
        help="y-axis scale for plots (default: none)",
    )
    parser.add_argument(
        "--outlier-filter",
        choices=["none", "iqr", "zscore", "mad", "trim"],
        default="none",
        help="outlier filtering for plots (default: none)",
    )
    parser.add_argument(
        "--outlier-k",
        type=float,
        default=1.5,
        help="threshold for iqr/zscore/mad filtering (default: 1.5)",
    )
    parser.add_argument(
        "--outlier-trim",
        type=float,
        default=0.05,
        help="fraction to trim from each tail when outlier-filter=trim (default: 0.05)",
    )

    parser.add_argument(
        "--sparse-which",
        default="both",
        choices=["A", "B", "both"],
        help="which matrices to sparsify when property=density",
    )
    parser.add_argument(
        "--sparse-tol",
        type=float,
        default=1e-12,
        help="tolerance when measuring realised density",
    )
    parser.add_argument(
        "--sparse-density",
        type=float,
        default=0.3,
        help="baseline density used when property is not 'density'",
    )

    parser.add_argument(
        "--deficiency-base",
        default="ginibre",
        choices=["ginibre", "stable", "binary", "sparse"],
        help="base ensemble for controllable/uncontrollable blocks",
    )
    parser.add_argument(
        "--deficiency-no-embed",
        action="store_true",
        help="disable random basis embedding in draw_with_ctrb_rank",
    )

    return parser


def main(argv: Sequence[str] | None = None) -> None:
    parser = build_parser()
    args = parser.parse_args(argv)
    run(args)


if __name__ == "__main__":
    main()

sim_unctrb_x0_boxplot_run = run
sim_unctrb_x0_boxplot_build_parser = build_parser

In [ ]:
# """Estimate (A,B) errors on low-PBH (A,B,x0) triples.
#
# Pipeline
# --------
# 1) Load (A,B) dataset saved by filter_unctrb_dataset.py.
# 2) Re-generate x0 samples (same procedure as sim_unctrb_x0_boxplot).
# 3) Select (A,B,x0) with PBH score below a threshold.
# 4) Simulate trajectories under PRBS excitation.
# 5) Estimate (A,B) via chosen estimators and compute relative errors
#    in the standard basis and the V(x0)-basis.
# """
from __future__ import annotations

import argparse
import math
import pathlib
from typing import Any, Dict, Iterable, Sequence

import numpy as np
import pandas as pd



def x0_sampler(
    mode: str,
    p_keep: float,
    renorm: bool,
    *,
    min_norm: float = 0.0,
    min_support: int = 0,
    max_attempts: int = 128,
):
    if mode == "sphere":
        return lambda n, rng: sample_unit_sphere(n, rng)
    if mode == "mask":
        return lambda n, rng: sample_masked_sphere(
            n,
            rng,
            p_keep,
            renorm=renorm,
            min_norm=min_norm,
            min_support=min_support,
            max_attempts=max_attempts,
        )
    raise ValueError(f"unknown x0 sampling mode '{mode}'")


def compute_pbh(A: np.ndarray, B: np.ndarray, x0: np.ndarray) -> float:
    return float(pbh_margin_structured(A, B, x0))


def compute_mu_min(A: np.ndarray, B: np.ndarray, x0: np.ndarray) -> float:
    Xaug = np.concatenate([x0.reshape(-1, 1), B], axis=1)
    mu_vals = left_eigvec_overlap(A, Xaug)
    return float(np.min(mu_vals)) if mu_vals.size else 0.0


def relative_errors(Ahat: np.ndarray, Bhat: np.ndarray, Atrue: np.ndarray, Btrue: np.ndarray) -> Dict[str, float]:
    errA = float(np.linalg.norm(Ahat - Atrue, ord="fro"))
    errB = float(np.linalg.norm(Bhat - Btrue, ord="fro"))
    nrmA = float(np.linalg.norm(Atrue, ord="fro") + 1e-15)
    nrmB = float(np.linalg.norm(Btrue, ord="fro") + 1e-15)
    relA = errA / nrmA
    relB = errB / nrmB
    return {
        "errA_rel": relA,
        "errB_rel": relB,
        "err_mean_rel": 0.5 * (relA + relB),
    }


def _visible_basis(A: np.ndarray, B: np.ndarray, x0: np.ndarray, tol: float) -> np.ndarray:
    return build_visible_basis_dt(A, B, x0, tol=tol)


def _moesp_wrapper(X0: np.ndarray, X1: np.ndarray, U_cm: np.ndarray, dt: float) -> tuple[np.ndarray, np.ndarray]:
    n = X0.shape[0]
    return moesp_fit_old(X0, X1, U_cm, n=n)


def _resolve_estimators(names: Sequence[str], dmdc_fn):
    registry = {
        "SINDy": lambda X0, X1, U_cm, dt: sindy_fit(X0, X1, U_cm, dt),
        "DMDc": lambda X0, X1, U_cm, dt: dmdc_fn(X0, X1, U_cm, dt),
        "MOESP": _moesp_wrapper,
        "NODE": lambda X0, X1, U_cm, dt: node_fit(
            X0, X1, U_cm, dt, epochs=200, lr=1e-2, early_stopping=True, verbose=False
        ),
    }
    lookup = {k.lower(): k for k in registry.keys()}
    resolved = []
    for name in names:
        key = name.strip().lower()
        if not key:
            continue
        if key not in lookup:
            raise ValueError(f"Unknown estimator '{name}'. Choose from {list(registry)}.")
        resolved.append(lookup[key])
    if not resolved:
        raise ValueError("No estimators selected.")
    return {name: registry[name] for name in resolved}


def _dmdc_callable(args: argparse.Namespace):
    if args.ridge:
        return lambda X0, X1, U_cm, dt: dmdc_ridge(X0, X1, U_cm, lam=float(args.ridge_lam))
    return lambda X0, X1, U_cm, dt: dmdc_tls(X0, X1, U_cm)


def run(args: argparse.Namespace) -> None:
    rng = np.random.default_rng(args.seed)
    outdir = pathlib.Path(args.outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    systems_df = pd.read_csv(args.dataset_csv)
    matrices = np.load(args.dataset_npz, allow_pickle=True)
    A_list = matrices["A"]
    B_list = matrices["B"]
    sys_idx = matrices["system_index"]
    index_map = {int(idx): pos for pos, idx in enumerate(sys_idx)}

    if "n" in systems_df.columns and systems_df["n"].notna().any():
        n_max = int(systems_df["n"].max())
    else:
        n_max = int(max(A.shape[0] for A in A_list))
    pe_order = pe_order_from_nmax(n_max)

    mode_configs: list[tuple[str, float]] = []
    p_list = [float(p) for p in args.mask_ps]
    if not args.exclude_sphere and 1.0 not in p_list:
        p_list.append(1.0)
    p_list = sorted(set(p_list))
    for p in p_list:
        if math.isclose(p, 1.0, rel_tol=0.0, abs_tol=1e-12):
            mode_configs.append(("sphere", 1.0))
        else:
            mode_configs.append(("mask", p))

    dmdc_fn = _dmdc_callable(args)

    estimators = _resolve_estimators(
        args.algos.split(",") if args.algos else ["SINDy", "DMDc", "MOESP", "NODE"],
        dmdc_fn,
    )

    selected_rows: list[dict[str, Any]] = []
    selected_A: list[np.ndarray] = []
    selected_B: list[np.ndarray] = []
    selected_x0: list[np.ndarray] = []
    selected_sys_idx: list[int] = []
    selected_x0_idx: list[int] = []

    results_rows: list[dict[str, Any]] = []

    for _, row in systems_df.iterrows():
        sys_id = int(row["system_index"])
        pos = index_map.get(sys_id)
        if pos is None:
            raise RuntimeError(f"system_index {sys_id} missing from matrices file")
        A = A_list[pos]
        B = B_list[pos]
        n = int(row["n"])
        m = int(row["m"])

        rho = spectral_radius(A)
        b_fro, b_row, b_col = b_norms(B)
        if args.max_spectral_radius is not None and rho > float(args.max_spectral_radius):
            continue
        if args.min_b_norm is not None and b_fro < float(args.min_b_norm):
            continue
        if args.min_b_row_norm is not None and b_row < float(args.min_b_row_norm):
            continue
        if args.min_b_col_norm is not None and b_col < float(args.min_b_col_norm):
            continue

        for mode, p_keep in mode_configs:
            sampler = x0_sampler(
                mode,
                p_keep,
                renorm=args.mask_renorm,
                min_norm=args.x0_min_norm,
                min_support=args.x0_min_support,
                max_attempts=args.x0_max_attempts,
            )
            for x0_idx in range(args.x0_samples):
                x0 = sampler(n, rng)
                pbh = compute_pbh(A, B, x0)
                if pbh >= args.pbh_threshold:
                    continue

                Vbasis = _visible_basis(A, B, x0, tol=args.visible_tol)
                if Vbasis.shape[1] < int(args.min_visible_dim):
                    continue

                # Store selected triple
                sel = {
                    "system_index": sys_id,
                    "x0_index": x0_idx,
                    "mode": format_mode_label(mode, p_keep),
                    "p_keep": p_keep,
                    "pbh": float(pbh),
                    "mu_min": compute_mu_min(A, B, x0),
                    "pe_order": int(pe_order),
                    "pe_order_est": np.nan,
                    "meta_spectral_radius": float(rho),
                    "meta_b_norm_fro": float(b_fro),
                    "meta_b_min_row_norm": float(b_row),
                    "meta_b_min_col_norm": float(b_col),
                }
                for col, val in row.items():
                    if np.isscalar(val):
                        sel[col] = val

                # Simulate trajectory
                U, pe_est = generate_pe_input(
                    T=args.T,
                    m=m,
                    dt=args.dt,
                    dwell=args.dwell,
                    rng=rng,
                    pe_order=pe_order,
                    family=args.input_family,
                    pe_method=args.pe_method,
                    pe_tol=args.pe_tol,
                    max_tries=args.pe_max_tries,
                    scale=args.u_scale,
                )
                X = simulate_dt(x0, A, B, U)
                X0 = X[:, :-1]
                X1 = X[:, 1:]
                U_cm = U.T

                sel["pe_order_est"] = int(pe_est)

                selected_rows.append(sel)
                selected_A.append(A)
                selected_B.append(B)
                selected_x0.append(x0)
                selected_sys_idx.append(sys_id)
                selected_x0_idx.append(x0_idx)

                A_V = Vbasis.T @ A @ Vbasis if Vbasis.size else np.zeros((0, 0))
                B_V = Vbasis.T @ B if Vbasis.size else np.zeros((0, B.shape[1]))
                z_cond = condition_number(np.vstack([X0, U_cm]), normalize_columns=True)

                for algo_name, algo_fn in estimators.items():
                    try:
                        if algo_name == "DMDc" and z_cond > float(args.dmdc_z_cond_max):
                            raise RuntimeError(
                                f"skipped: Z cond {z_cond:.2e} > {float(args.dmdc_z_cond_max):.2e}"
                            )
                        Ahat, Bhat = algo_fn(X0, X1, U_cm, args.dt)
                        Ahat_V = Vbasis.T @ Ahat @ Vbasis if Vbasis.size else np.zeros((0, 0))
                        Bhat_V = Vbasis.T @ Bhat if Vbasis.size else np.zeros((0, Bhat.shape[1]))
                        err_std = relative_errors(Ahat, Bhat, A, B)
                        err_P = relative_errors(Ahat_V, Bhat_V, A_V, B_V)
                        err_msg = ""
                    except Exception as exc:
                        err_std = {"errA_rel": np.nan, "errB_rel": np.nan, "err_mean_rel": np.nan}
                        err_P = {"errA_rel": np.nan, "errB_rel": np.nan, "err_mean_rel": np.nan}
                        err_msg = str(exc)

                    results_rows.append(
                        {
                            **sel,
                            "algo": algo_name,
                            "dim_visible": int(Vbasis.shape[1]),
                            "z_cond": float(z_cond),
                            "errA_rel": err_std["errA_rel"],
                            "errB_rel": err_std["errB_rel"],
                            "err_mean_rel": err_std["err_mean_rel"],
                            "errA_rel_P": err_P["errA_rel"],
                            "errB_rel_P": err_P["errB_rel"],
                            "err_mean_rel_P": err_P["err_mean_rel"],
                            "estimator_error": err_msg,
                        }
                    )

                if args.max_selected is not None and len(selected_rows) >= args.max_selected:
                    break
            if args.max_selected is not None and len(selected_rows) >= args.max_selected:
                break
        if args.max_selected is not None and len(selected_rows) >= args.max_selected:
            break

    if not selected_rows:
        raise RuntimeError("No (A,B,x0) pairs satisfied the PBH threshold.")

    selected_df = pd.DataFrame(selected_rows)
    selected_suffix = args.suffix or f"pbh_lt_{args.pbh_threshold:g}"
    selected_csv = outdir / f"selected_{selected_suffix}.csv"
    selected_npz = outdir / f"selected_{selected_suffix}.npz"
    selected_df.to_csv(selected_csv, index=False)
    np.savez_compressed(
        selected_npz,
        A=np.array(selected_A, dtype=object),
        B=np.array(selected_B, dtype=object),
        x0=np.array(selected_x0, dtype=object),
        system_index=np.array(selected_sys_idx, dtype=int),
        x0_index=np.array(selected_x0_idx, dtype=int),
    )

    results_df = pd.DataFrame(results_rows)
    results_df.to_csv(outdir / "estimation_errors.csv", index=False)


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--dataset-csv", required=True, help="filtered systems CSV")
    parser.add_argument("--dataset-npz", required=True, help="filtered systems NPZ")
    parser.add_argument("--outdir", required=True, help="output directory")

    parser.add_argument("--seed", type=int, default=12345, help="RNG seed")
    parser.add_argument("--x0-samples", type=int, default=10, help="x0 samples per mode per system")
    parser.add_argument("--mask-ps", nargs="*", default=[0.25, 0.5, 0.75, 1.0], help="Bernoulli keep probs")
    parser.add_argument("--exclude-sphere", action="store_true", help="do not automatically include sphere (p=1) when using --mask-ps")
    parser.add_argument("--mask-renorm", action="store_true", help="renormalize masked x0")
    parser.add_argument("--x0-min-norm", type=float, default=0.0, help="minimum ||x0|| for masked sampling")
    parser.add_argument("--x0-min-support", type=int, default=0, help="minimum nonzeros in masked x0")
    parser.add_argument("--x0-max-attempts", type=int, default=128, help="max attempts for masked x0")

    parser.add_argument("--pbh-threshold", type=float, required=True, help="PBH cutoff for selection")
    parser.add_argument("--max-selected", type=int, default=None, help="cap on selected triples")
    parser.add_argument("--suffix", default=None, help="suffix for selected dataset files")

    parser.add_argument("--T", type=int, default=100, help="trajectory horizon")
    parser.add_argument("--dt", type=float, default=1.0, help="sampling interval for estimators")
    parser.add_argument("--u-scale", type=float, default=3.0, help="PRBS amplitude")
    parser.add_argument("--dwell", type=int, default=1, help="PRBS dwell time")
    parser.add_argument("--visible-tol", type=float, default=1e-12, help="tolerance for V(x0) basis")
    parser.add_argument(
        "--min-visible-dim",
        type=int,
        default=0,
        help="minimum allowed dim V(x0); smaller dims are skipped",
    )
    parser.add_argument(
        "--input-family",
        choices=["prbs", "multisine"],
        default="prbs",
        help="input family used for PE generation",
    )
    parser.add_argument(
        "--pe-method",
        choices=["block", "moment"],
        default="block",
        help="PE verification method",
    )
    parser.add_argument("--pe-tol", type=float, default=1e-8, help="PE rank tolerance")
    parser.add_argument("--pe-max-tries", type=int, default=128, help="max PE generation attempts")
    parser.add_argument(
        "--dmdc-z-cond-max",
        type=float,
        default=1e8,
        help="max allowed condition number of Z=[X;U] for DMDc (skip if exceeded)",
    )
    parser.add_argument(
        "--ridge",
        action="store_true",
        help="use ridge-regularized DMDc instead of TLS",
    )
    parser.add_argument(
        "--ridge-lam",
        type=float,
        default=1e-6,
        help="ridge parameter for DMDc when --ridge is enabled",
    )
    parser.add_argument(
        "--max-spectral-radius",
        type=float,
        default=None,
        help="maximum allowed spectral radius of A (skip if exceeded)",
    )
    parser.add_argument(
        "--min-b-norm",
        type=float,
        default=None,
        help="minimum allowed Frobenius norm of B",
    )
    parser.add_argument(
        "--min-b-row-norm",
        type=float,
        default=None,
        help="minimum allowed row 2-norm of B",
    )
    parser.add_argument(
        "--min-b-col-norm",
        type=float,
        default=None,
        help="minimum allowed column 2-norm of B",
    )

    parser.add_argument(
        "--algos",
        type=str,
        default=None,
        help="Comma-separated list of estimators (default: SINDy,DMDc,MOESP,NODE)",
    )
    return parser


def main() -> None:
    parser = build_parser()
    args = parser.parse_args()
    run(args)


if __name__ == "__main__":
    main()

sim_unctrb_pbh_estimators_run = run
sim_unctrb_pbh_estimators_build_parser = build_parser

In [ ]:
# """Box plots of estimation errors for selected (A,B,x0) triples.
#
# Loads the selected dataset produced by sim_unctrb_pbh_estimators and
# re-runs estimators to compute error distributions in:
#   - standard basis
#   - visible-subspace restriction (A|_V, B|_V with V=V(x0))
# """
from __future__ import annotations

import argparse
import pathlib
from typing import Any, Dict, Sequence

import numpy as np
import pandas as pd



ALG_ORDER = ["DMDc", "MOESP", "SINDy", "NODE"]


def relative_errors(Ahat: np.ndarray, Bhat: np.ndarray, Atrue: np.ndarray, Btrue: np.ndarray) -> Dict[str, float]:
    errA = float(np.linalg.norm(Ahat - Atrue, ord="fro"))
    errB = float(np.linalg.norm(Bhat - Btrue, ord="fro"))
    nrmA = float(np.linalg.norm(Atrue, ord="fro") + 1e-15)
    nrmB = float(np.linalg.norm(Btrue, ord="fro") + 1e-15)
    relA = errA / nrmA
    relB = errB / nrmB
    return {
        "errA_rel": relA,
        "errB_rel": relB,
        "err_mean_rel": 0.5 * (relA + relB),
    }


def _visible_basis(A: np.ndarray, B: np.ndarray, x0: np.ndarray, tol: float) -> np.ndarray:
    return build_visible_basis_dt(A, B, x0, tol=tol)


def _moesp_wrapper(X0: np.ndarray, X1: np.ndarray, U_cm: np.ndarray, dt: float) -> tuple[np.ndarray, np.ndarray]:
    n = X0.shape[0]
    return moesp_fit_old(X0, X1, U_cm, n=n)


def _resolve_estimators(names: Sequence[str], dmdc_fn):
    registry = {
        "SINDy": lambda X0, X1, U_cm, dt: sindy_fit(X0, X1, U_cm, dt),
        "DMDc": lambda X0, X1, U_cm, dt: dmdc_fn(X0, X1, U_cm, dt),
        "MOESP": _moesp_wrapper,
        "NODE": lambda X0, X1, U_cm, dt: node_fit(
            X0, X1, U_cm, dt, epochs=200, lr=1e-2, early_stopping=True, verbose=False
        ),
    }
    lookup = {k.lower(): k for k in registry.keys()}
    resolved = []
    for name in names:
        key = name.strip().lower()
        if not key:
            continue
        if key not in lookup:
            raise ValueError(f"Unknown estimator '{name}'. Choose from {list(registry)}.")
        resolved.append(lookup[key])
    if not resolved:
        raise ValueError("No estimators selected.")
    return {name: registry[name] for name in resolved}


def _dmdc_callable(args: argparse.Namespace):
    if args.ridge:
        return lambda X0, X1, U_cm, dt: dmdc_ridge(X0, X1, U_cm, lam=float(args.ridge_lam))
    return lambda X0, X1, U_cm, dt: dmdc_tls(X0, X1, U_cm)


def run(args: argparse.Namespace) -> None:
    rng = np.random.default_rng(args.seed)
    outdir = pathlib.Path(args.outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    if args.selected_csv:
        selected_df = pd.read_csv(args.selected_csv)
    else:
        selected_df = None

    sel = np.load(args.selected_npz, allow_pickle=True)
    A_list = sel["A"]
    B_list = sel["B"]
    x0_list = sel["x0"]

    n_max = int(max(A.shape[0] for A in A_list))
    pe_order = pe_order_from_nmax(n_max)

    dmdc_fn = _dmdc_callable(args)

    estimators = _resolve_estimators(
        args.algos.split(",") if args.algos else ALG_ORDER,
        dmdc_fn,
    )

    rows: list[dict[str, Any]] = []

    for idx in range(len(A_list)):
        A = A_list[idx]
        B = B_list[idx]
        x0 = x0_list[idx]
        n = A.shape[0]
        m = B.shape[1]

        U, pe_est = generate_pe_input(
            T=args.T,
            m=m,
            dt=args.dt,
            dwell=args.dwell,
            rng=rng,
            pe_order=pe_order,
            family=args.input_family,
            pe_method=args.pe_method,
            pe_tol=args.pe_tol,
            max_tries=args.pe_max_tries,
            scale=args.u_scale,
        )
        X = simulate_dt(x0, A, B, U)
        X0 = X[:, :-1]
        X1 = X[:, 1:]
        U_cm = U.T

        Vbasis = _visible_basis(A, B, x0, tol=args.visible_tol)
        if Vbasis.shape[1] < int(args.min_visible_dim):
            continue
        # Restrict to the visible subspace V(x0), matching the theory:
        # A|_V = V^T A V, B|_V = V^T B
        if Vbasis.size:
            A_V = Vbasis.T @ A @ Vbasis
            B_V = Vbasis.T @ B
        else:
            A_V = np.zeros((0, 0), dtype=float)
            B_V = np.zeros((0, B.shape[1]), dtype=float)

        base_info = {
            "sample_index": idx,
            "dim_visible": int(Vbasis.shape[1]),
            "pe_order": int(pe_order),
            "pe_order_est": int(pe_est),
        }
        if selected_df is not None:
            for col, val in selected_df.iloc[idx].items():
                if np.isscalar(val):
                    base_info[col] = val

        z_cond = condition_number(np.vstack([X0, U_cm]), normalize_columns=True)

        for algo_name, algo_fn in estimators.items():
            try:
                if algo_name == "DMDc" and z_cond > float(args.dmdc_z_cond_max):
                    raise RuntimeError(
                        f"skipped: Z cond {z_cond:.2e} > {float(args.dmdc_z_cond_max):.2e}"
                    )
                Ahat, Bhat = algo_fn(X0, X1, U_cm, args.dt)
                if Vbasis.size:
                    Ahat_V = Vbasis.T @ Ahat @ Vbasis
                    Bhat_V = Vbasis.T @ Bhat
                else:
                    Ahat_V = np.zeros((0, 0), dtype=float)
                    Bhat_V = np.zeros((0, Bhat.shape[1]), dtype=float)
                err_std = relative_errors(Ahat, Bhat, A, B)
                err_P = relative_errors(Ahat_V, Bhat_V, A_V, B_V)
                err_msg = ""
            except Exception as exc:
                err_std = {"errA_rel": np.nan, "errB_rel": np.nan, "err_mean_rel": np.nan}
                err_P = {"errA_rel": np.nan, "errB_rel": np.nan, "err_mean_rel": np.nan}
                err_msg = str(exc)

            rows.append(
                {
                    **base_info,
                    "algo": algo_name,
                    "z_cond": float(z_cond),
                    "err_mean_rel": err_std["err_mean_rel"],
                    "err_mean_rel_P": err_P["err_mean_rel"],
                    "estimator_error": err_msg,
                }
            )

    results_df = pd.DataFrame(rows)
    results_df.to_csv(outdir / "estimation_errors_boxplot.csv", index=False)

    try:
        import matplotlib.pyplot as plt
    except ModuleNotFoundError as exc:  # pragma: no cover
        raise RuntimeError("matplotlib is required for plotting") from exc

    set_default_mpl_style()
    order = [name for name in ALG_ORDER if name in results_df["algo"].unique()]
    std_data = [results_df[results_df["algo"] == name]["err_mean_rel"].to_numpy() for name in order]
    P_data = [results_df[results_df["algo"] == name]["err_mean_rel_P"].to_numpy() for name in order]
    yscale = None if args.yscale == "none" else args.yscale

    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    nice_boxplot(
        ax,
        std_data,
        order,
        title="Standard-basis estimation error",
        ylabel="mean relative error (standard)",
        yscale=yscale,
    )
    fig.tight_layout()
    fig.savefig(outdir / "boxplot_err_standard.png", bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    nice_boxplot(
        ax,
        P_data,
        order,
        title="Visible-subspace estimation error",
        ylabel="mean relative error (V-basis)",
        yscale=yscale,
    )
    fig.tight_layout()
    fig.savefig(outdir / "boxplot_err_Pbasis.png", bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8.4, 4.6))
    nice_grouped_boxplot(
        ax,
        std_data,
        P_data,
        order,
        title="Standard vs visible-subspace estimation error",
        ylabel="mean relative error",
        yscale=yscale,
    )
    fig.tight_layout()
    fig.savefig(outdir / "boxplot_err_standard_vs_Pbasis.png", bbox_inches="tight")
    plt.close(fig)


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--selected-npz", required=True, help="selected dataset NPZ from sim_unctrb_pbh_estimators")
    parser.add_argument("--selected-csv", default=None, help="selected dataset CSV (optional, for metadata)")
    parser.add_argument("--outdir", required=True, help="output directory")

    parser.add_argument("--seed", type=int, default=12345, help="RNG seed")
    parser.add_argument("--T", type=int, default=100, help="trajectory horizon")
    parser.add_argument("--dt", type=float, default=1.0, help="sampling interval for estimators")
    parser.add_argument("--u-scale", type=float, default=3.0, help="PRBS amplitude")
    parser.add_argument("--dwell", type=int, default=1, help="PRBS dwell time")
    parser.add_argument("--visible-tol", type=float, default=1e-12, help="tolerance for V(x0) basis")
    parser.add_argument(
        "--min-visible-dim",
        type=int,
        default=0,
        help="minimum allowed dim V(x0); smaller dims are skipped",
    )
    parser.add_argument(
        "--input-family",
        choices=["prbs", "multisine"],
        default="prbs",
        help="input family used for PE generation",
    )
    parser.add_argument(
        "--pe-method",
        choices=["block", "moment"],
        default="block",
        help="PE verification method",
    )
    parser.add_argument("--pe-tol", type=float, default=1e-8, help="PE rank tolerance")
    parser.add_argument("--pe-max-tries", type=int, default=128, help="max PE generation attempts")
    parser.add_argument(
        "--dmdc-z-cond-max",
        type=float,
        default=1e8,
        help="max allowed condition number of Z=[X;U] for DMDc (skip if exceeded)",
    )
    parser.add_argument(
        "--ridge",
        action="store_true",
        help="use ridge-regularized DMDc instead of TLS",
    )
    parser.add_argument(
        "--ridge-lam",
        type=float,
        default=1e-6,
        help="ridge parameter for DMDc when --ridge is enabled",
    )

    parser.add_argument(
        "--algos",
        type=str,
        default=None,
        help="Comma-separated list of estimators (default: DMDc,MOESP,SINDy,NODE)",
    )
    parser.add_argument(
        "--yscale",
        choices=["none", "log"],
        default="none",
        help="y-axis scale for boxplots (default: none)",
    )
    return parser


def main() -> None:
    parser = build_parser()
    args = parser.parse_args()
    run(args)


if __name__ == "__main__":
    main()

sim_unctrb_pbh_error_boxplots_run = run
sim_unctrb_pbh_error_boxplots_build_parser = build_parser

## Pipeline
Runs the three stages (A,B), x0 scores, and estimator errors. Outputs are saved to disk and previewed below.

In [ ]:
from types import SimpleNamespace


def _mask_ps_from_scheme(*, sphere: bool, sparse_p: Optional[float]) -> list[float]:
    if sparse_p is not None and sphere:
        raise ValueError("Use either sphere=True or sparse_p, not both.")
    if sparse_p is not None:
        p = float(sparse_p)
        if not (0.0 < p <= 1.0):
            raise ValueError("sparse_p must be in (0, 1].")
        return [p]
    return [1.0]  # default sphere


def run_iclrii_pipeline():
    base_out = pathlib.Path(base_outdir)
    ensemble_out = base_out / ensemble_dir
    x0_out = base_out / x0_dir
    pbh_out = base_out / pbh_dir
    boxplot_out = pbh_out / "boxplots"

    # Stage 1: (A,B) ensemble
    print("Stage 1: Generating (A,B) ensemble...")
    regcomb_args = sim_regcomb_ctrb_build_parser().parse_args(
        [
            "--axes",
            "sparsity,ndim",
            "--sparsity-grid",
            sparsity_grid,
            "--ndim-grid",
            ndim_grid,
            "--samples",
            str(samples),
            "--outdir",
            str(ensemble_out),
            "--save-matrices",
        ]
    )
    if regcomb_args.m is None:
        regcomb_args.m = int(regcomb_args.n)
    sim_regcomb_ctrb_run(regcomb_args)

    systems_csv = ensemble_out / "systems.csv"
    systems_npz = ensemble_out / "systems_matrices.npz"
    summary_csv = ensemble_out / "scores_summary.csv"

    systems_df = pd.read_csv(systems_csv)
    summary_df = pd.read_csv(summary_csv)
    print(f"Saved {len(systems_df)} systems -> {systems_csv}")
    display(systems_df.head())
    display(summary_df.head())

    # Stage 2: x0 identifiability scores
    mask_ps = _mask_ps_from_scheme(sphere=sphere, sparse_p=sparse_p)
    print("Stage 2: Computing x0 identifiability scores...")
    x0_args = sim_unctrb_x0_boxplot_build_parser().parse_args(
        [
            "--dataset-csv",
            str(systems_csv),
            "--dataset-npz",
            str(systems_npz),
            "--x0-samples",
            str(x0_samples),
            "--mask-ps",
            *[str(p) for p in mask_ps],
            "--mask-renorm",
            "--x0-min-norm",
            str(x0_min_norm),
            "--x0-min-support",
            str(x0_min_support),
            "--x0-max-attempts",
            str(x0_max_attempts),
            "--outdir",
            str(x0_out),
            "--outlier-trim",
            str(outlier_trim),
        ]
    )
    sim_unctrb_x0_boxplot_run(x0_args)

    scores_csv = x0_out / "identifiability_scores.csv"
    scores_df = pd.read_csv(scores_csv)
    print(f"Saved {len(scores_df)} score rows -> {scores_csv}")
    display(scores_df.head())

    # Stage 3: Estimator errors (NODE, DMDc, MOESP, SINDy)
    print("Stage 3: Running estimators...")
    pbh_threshold = float("inf")
    suffix = selected_suffix or "all_x0"

    pbh_args = sim_unctrb_pbh_estimators_build_parser().parse_args(
        [
            "--dataset-csv",
            str(systems_csv),
            "--dataset-npz",
            str(systems_npz),
            "--outdir",
            str(pbh_out),
            "--seed",
            str(seed),
            "--x0-samples",
            str(x0_samples),
            "--mask-ps",
            *[str(p) for p in mask_ps],
            "--mask-renorm",
            "--x0-min-norm",
            str(x0_min_norm),
            "--x0-min-support",
            str(x0_min_support),
            "--x0-max-attempts",
            str(x0_max_attempts),
            "--pbh-threshold",
            str(pbh_threshold),
            "--suffix",
            suffix,
            "--min-visible-dim",
            str(min_visible_dim),
            "--T",
            str(T),
            "--dt",
            str(dt),
            "--u-scale",
            str(u_scale),
            "--dwell",
            str(dwell),
            "--input-family",
            input_family,
            "--pe-method",
            pe_method,
            "--pe-tol",
            str(pe_tol),
            "--pe-max-tries",
            str(pe_max_tries),
            "--algos",
            algos,
            "--dmdc-z-cond-max",
            str(dmdc_z_cond_max),
        ]
        + (["--ridge", "--ridge-lam", str(ridge_lam)] if ridge else [])
    )
    sim_unctrb_pbh_estimators_run(pbh_args)

    selected_csv = pbh_out / f"selected_{suffix}.csv"
    selected_npz = pbh_out / f"selected_{suffix}.npz"
    errors_csv = pbh_out / "estimation_errors.csv"

    selected_df = pd.read_csv(selected_csv)
    errors_df = pd.read_csv(errors_csv)
    print(f"Saved {len(selected_df)} selected triples -> {selected_csv}")
    display(selected_df.head())
    display(errors_df.head())

    # Stage 4: Error boxplots
    print("Stage 4: Error boxplots...")
    box_args = sim_unctrb_pbh_error_boxplots_build_parser().parse_args(
        [
            "--selected-npz",
            str(selected_npz),
            "--selected-csv",
            str(selected_csv),
            "--outdir",
            str(boxplot_out),
            "--seed",
            str(seed),
            "--min-visible-dim",
            str(min_visible_dim),
            "--T",
            str(T),
            "--dt",
            str(dt),
            "--u-scale",
            str(u_scale),
            "--dwell",
            str(dwell),
            "--input-family",
            input_family,
            "--pe-method",
            pe_method,
            "--pe-tol",
            str(pe_tol),
            "--pe-max-tries",
            str(pe_max_tries),
            "--algos",
            algos,
            "--dmdc-z-cond-max",
            str(dmdc_z_cond_max),
        ]
        + (["--ridge", "--ridge-lam", str(ridge_lam)] if ridge else [])
    )
    sim_unctrb_pbh_error_boxplots_run(box_args)

    box_errors_csv = boxplot_out / "estimation_errors_boxplot.csv"
    box_errors_df = pd.read_csv(box_errors_csv)
    print(f"Saved {len(box_errors_df)} boxplot rows -> {box_errors_csv}")
    display(box_errors_df.head())

    return {
        "systems": systems_df,
        "scores": scores_df,
        "selected": selected_df,
        "errors": errors_df,
        "boxplot_errors": box_errors_df,
        "paths": {
            "ensemble_out": ensemble_out,
            "x0_out": x0_out,
            "pbh_out": pbh_out,
            "boxplot_out": boxplot_out,
        },
    }


# Run the pipeline top-to-bottom
results = run_iclrii_pipeline()